# QoS Buddy — Optimization Agent Benchmark and Defence

## Notebook Introduction

This notebook evaluates the QoS Buddy Optimization Agent as a decision policy over network remediation actions. The input is a diagnostic contract containing a root cause, confidence, KPI evidence, and an action mask. The output is one safe remediation action.

We keep the benchmark conservative. We separate real holdout episodes from synthetic balancing examples, report frozen and online/prequential performance separately, record LLM cache coverage instead of assuming it, and test robustness under diagnosis noise and posterior uncertainty.

### Table of Contents

1. [Reproducibility preamble](#sec-01)
2. [Data ingestion and incident interval matching](#sec-02)
3. [Data quality filtering](#sec-03)
4. [Data engineering protocol](#sec-04)
5. [Exploratory KPI and anomaly figures](#sec-05)
6. [Root-cause vocabulary and diagnostic contract](#sec-06)
7. [Action catalogue and root-cause action mask](#sec-07)
8. [Reward logic and reward-mode checks](#sec-08)
9. [Episode construction and chronological split](#sec-09)
10. [Shared feature encoder](#sec-10)
11. [Reference baselines](#sec-11)
12. [M6 LinUCB contextual bandit](#sec-12)
13. [EpsilonGreedy bandit](#sec-13)
14. [M7 LLM-guided LinUCB](#sec-14)
15. [M8 offline reward ranker](#sec-15)
16. [Benchmark comparison and visual diagnostics](#sec-16)
17. [LLM reasoning case studies](#sec-17)
18. [Counterfactual and diagnosis-posterior robustness](#sec-18)
19. [Reward mode comparison](#sec-19)
20. [Final verdict](#sec-20)
21. [Notebook integrity sweep](#sec-21)
22. [Report figures and deployment contract](#sec-22)
23. [Conclusion](#conclusion)


<a id="sec-01"></a>

## 01 — Reproducibility Preamble

We begin by fixing the execution environment before any data or model code runs. The project root, deterministic seed, figure registry, plotting defaults, and file-hash helpers create a stable audit trail for every later table and figure.

In [1]:
from __future__ import annotations
import hashlib, random, time, warnings
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
import requests

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="plotly")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR     = PROJECT_ROOT
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim"
FIG_DIR      = PROJECT_ROOT / "reports" / "figures"

for d in (INTERIM_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

pio.templates.default = "plotly_white"
PLOTLY_FONT   = dict(family="Georgia, serif", size=13, color="#222")
PLOTLY_LAYOUT = dict(font=PLOTLY_FONT, margin=dict(l=60, r=30, t=70, b=60),
                     title=dict(x=0.02, xanchor="left"))

ARTEFACT_REGISTRY: dict[str, str] = {}

def sha256_file(path: Path) -> str:
    """Function helper for the benchmark pipeline."""
    hasher = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

def register_hash(name: str, path: Path) -> str:
    """Function helper for the benchmark pipeline."""
    digest = sha256_file(path)
    if name in ARTEFACT_REGISTRY and ARTEFACT_REGISTRY[name] != digest:
        raise RuntimeError(f"Hash mismatch for '{name}'")
    ARTEFACT_REGISTRY[name] = digest
    return digest

def style(fig: go.Figure, title: str, subtitle: str | None = None) -> go.Figure:
    """Function helper for the benchmark pipeline."""
    full_title = title if subtitle is None else f"{title}<br><sup>{subtitle}</sup>"
    fig.update_layout(title_text=full_title, **PLOTLY_LAYOUT)
    return fig

def save_fig(fig: go.Figure, name: str) -> Path:
    """Function helper for the benchmark pipeline."""
    html_path = FIG_DIR / f"{name}.html"
    fig.write_html(html_path, include_plotlyjs="cdn")
    register_hash(f"figure:{name}.html", html_path)
    return html_path

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Random seed locked at {RANDOM_STATE}")

PROJECT_ROOT: C:\Users\amani\OneDrive\Desktop\pi - Copy - Copy - Copy (2)
Random seed locked at 42


**Interpretation:** The run starts from a fixed project root, fixed random seed, and a figure/artifact registry. The executed output points to the current workspace, so the notebook is no longer carrying stale results from a copied folder.

<a id="sec-02"></a>

## 02 — Data Ingestion and Schema Reconciliation

With the environment fixed, we load the QoS time-series files and incident files, reconcile their schemas, and compare two incident matching strategies. We keep interval matching as the implemented protocol because it respects active incident windows; nearest-start matching remains only as a rejected audit baseline.

In [2]:
qos_files = sorted(DATA_DIR.glob("qos_timeseries_*.csv"))
inc_files = sorted(DATA_DIR.glob("incidents_*.csv"))

if not qos_files:
    raise FileNotFoundError(f"No qos_timeseries_*.csv files found under {DATA_DIR}")
if not inc_files:
    raise FileNotFoundError(f"No incidents_*.csv files found under {DATA_DIR}")

def load_qos_frames(files: list[Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load all QoS time-series files and return the frame plus per-file audit rows."""
    pieces, per_file = [], []
    for f in files:
        piece = pd.read_csv(f, low_memory=False)
        piece["source_file"] = f.name
        per_file.append({"file": f.name, "rows": len(piece), "cols": piece.shape[1]})
        pieces.append(piece)
    df = pd.concat(pieces, ignore_index=True, sort=False)
    for col in ("teams_in_meeting",):
        if col in df.columns:
            df = df.drop(columns=col)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    bad_ts = int(df["timestamp"].isna().sum())
    if bad_ts:
        raise ValueError(f"QoS data contains {bad_ts} unparsable timestamps")
    df["timestamp"] = df["timestamp"].dt.tz_convert("Africa/Tunis")
    before = len(df)
    df = df.drop_duplicates(subset=["timestamp", "node_id", "cell_id", "zone_id"])
    per_file_df = pd.DataFrame(per_file)
    per_file_df["kind"] = "qos_timeseries"
    per_file_df["rows_dropped_global_dedup"] = before - len(df)
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df, per_file_df

def load_incident_frames(files: list[Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load incident files and keep header-only files visible in the audit."""
    pieces, per_file = [], []
    for f in files:
        piece = pd.read_csv(f)
        per_file.append({"file": f.name, "rows": len(piece), "cols": piece.shape[1], "header_only": piece.empty})
        if piece.empty:
            warnings.warn(f"{f.name} is header-only; skipping")
            continue
        piece["source_file"] = f.name
        pieces.append(piece)
    df = pd.concat(pieces, ignore_index=True, sort=False) if pieces else pd.DataFrame()
    if not df.empty:
        df["start_timestamp"] = pd.to_datetime(df["start_timestamp"], errors="coerce", utc=True).dt.tz_convert("Africa/Tunis")
        df["end_timestamp"] = pd.to_datetime(df["end_timestamp"], errors="coerce", utc=True).dt.tz_convert("Africa/Tunis")
        bad_ts = int(df[["start_timestamp", "end_timestamp"]].isna().any(axis=1).sum())
        if bad_ts:
            raise ValueError(f"Incident data contains {bad_ts} rows with unparsable start/end timestamps")
        df = df.sort_values(["node_id", "start_timestamp", "end_timestamp"]).reset_index(drop=True)
    per_file_df = pd.DataFrame(per_file)
    per_file_df["kind"] = "incident"
    return df, per_file_df

def summarize_qos_quality(qos: pd.DataFrame) -> pd.DataFrame:
    """Create a compact audit of QoS completeness, duplicates, time range, and flags."""
    duplicate_keys = int(qos.duplicated(subset=["timestamp", "node_id", "cell_id", "zone_id"]).sum())
    missing_pct = qos.isna().mean().sort_values(ascending=False).head(12)
    return pd.DataFrame([
        {"metric": "qos_rows", "value": len(qos)},
        {"metric": "qos_files", "value": len(qos_files)},
        {"metric": "duplicate_time_node_cell_zone_keys", "value": duplicate_keys},
        {"metric": "unique_nodes", "value": qos["node_id"].nunique()},
        {"metric": "unique_cells", "value": qos["cell_id"].nunique()},
        {"metric": "unique_zones", "value": qos["zone_id"].nunique()},
        {"metric": "time_start", "value": str(qos["timestamp"].min())},
        {"metric": "time_end", "value": str(qos["timestamp"].max())},
        {"metric": "rows_flagged_skip_for_training", "value": int(qos.get("skip_for_training", pd.Series(False, index=qos.index)).fillna(False).astype(bool).sum())},
        {"metric": "rows_quality_poor", "value": int(qos.get("data_quality_rating", pd.Series("", index=qos.index)).astype(str).str.lower().eq("poor").sum())},
        {"metric": "worst_missing_column", "value": str(missing_pct.index[0]) if len(missing_pct) else "none"},
        {"metric": "worst_missing_pct", "value": round(float(missing_pct.iloc[0]), 4) if len(missing_pct) else 0.0},
    ])

def summarize_incident_quality(incidents: pd.DataFrame, incident_file_audit: pd.DataFrame) -> pd.DataFrame:
    """Create an incident audit focused on duration, headers, and temporal validity."""
    if incidents.empty:
        return pd.DataFrame([{"metric": "incident_rows", "value": 0}])
    durations = (incidents["end_timestamp"] - incidents["start_timestamp"]).dt.total_seconds()
    return pd.DataFrame([
        {"metric": "incident_rows", "value": len(incidents)},
        {"metric": "incident_files", "value": len(incident_file_audit)},
        {"metric": "header_only_incident_files", "value": int(incident_file_audit["header_only"].sum())},
        {"metric": "unique_incident_ids", "value": incidents["incident_id"].nunique()},
        {"metric": "unique_incident_nodes", "value": incidents["node_id"].nunique()},
        {"metric": "nonpositive_duration_incidents", "value": int((durations <= 0).sum())},
        {"metric": "median_duration_sec", "value": round(float(durations.clip(lower=0).median()), 2)},
        {"metric": "p90_duration_sec", "value": round(float(durations.clip(lower=0).quantile(0.90)), 2)},
    ])

def nearest_start_join_for_audit(qos: pd.DataFrame, incidents: pd.DataFrame,
                                 tolerance_s: int = 90) -> pd.DataFrame:
    """Reproduce the rejected nearest-start labeling strategy for audit comparison only."""
    out = qos.copy()
    if incidents.empty:
        out["incident_id"] = np.nan
        out["has_incident"] = False
        return out
    pieces = []
    for node, q in out.groupby("node_id", sort=False):
        i = incidents[incidents["node_id"] == node].sort_values("start_timestamp")
        if i.empty:
            piece = q.copy()
            piece["incident_id"] = np.nan
        else:
            piece = pd.merge_asof(
                q.sort_values("timestamp"),
                i[["start_timestamp", "incident_id"]],
                left_on="timestamp", right_on="start_timestamp",
                direction="nearest", tolerance=pd.Timedelta(seconds=tolerance_s),
            ).drop(columns="start_timestamp")
        pieces.append(piece)
    joined = pd.concat(pieces, ignore_index=True).sort_values("timestamp").reset_index(drop=True)
    joined["has_incident"] = joined["incident_id"].notna()
    return joined

def interval_join_incidents(qos: pd.DataFrame, incidents: pd.DataFrame,
                            instant_tolerance_s: int = 90) -> pd.DataFrame:
    """Attach incidents by node and active time interval, not nearest start time."""
    out = qos.copy()
    incident_cols = ["incident_id", "incident_type", "severity"]
    for col in incident_cols:
        out[col] = np.nan
    out["incident_start_timestamp"] = pd.NaT
    out["incident_end_timestamp"] = pd.NaT

    if incidents.empty:
        out["has_incident"] = False
        return out

    inc = incidents.copy()
    short = inc["end_timestamp"] <= inc["start_timestamp"]
    inc.loc[short, "end_timestamp"] = inc.loc[short, "start_timestamp"] + pd.Timedelta(seconds=instant_tolerance_s)

    for node, q_idx in out.groupby("node_id", sort=False).groups.items():
        node_inc = inc[inc["node_id"] == node]
        if node_inc.empty:
            continue
        node_mask = out.index.isin(q_idx)
        for _, alarm in node_inc.iterrows():
            mask = node_mask & out["timestamp"].between(alarm["start_timestamp"], alarm["end_timestamp"], inclusive="both")
            if not mask.any():
                continue
            assign_idx = out.loc[mask & out["incident_id"].isna()].index
            if len(assign_idx) == 0:
                continue
            for col in incident_cols:
                out.loc[assign_idx, col] = alarm.get(col, np.nan)
            out.loc[assign_idx, "incident_start_timestamp"] = alarm["start_timestamp"]
            out.loc[assign_idx, "incident_end_timestamp"] = alarm["end_timestamp"]

    out["has_incident"] = out["incident_id"].notna()
    return out.sort_values("timestamp").reset_index(drop=True)

qos_df, qos_file_audit = load_qos_frames(qos_files)
inc_df, incident_file_audit = load_incident_frames(inc_files)

qos_quality_audit = summarize_qos_quality(qos_df)
incident_quality_audit = summarize_incident_quality(inc_df, incident_file_audit)

nearest_join_df = nearest_start_join_for_audit(qos_df, inc_df)
df = interval_join_incidents(qos_df, inc_df)

join_audit = pd.DataFrame([
    {"labeling_protocol": "nearest_start_rejected", "rows_labeled_incident": int(nearest_join_df["has_incident"].sum()),
     "unique_incidents_matched": int(nearest_join_df["incident_id"].nunique())},
    {"labeling_protocol": "interval_active_window_selected", "rows_labeled_incident": int(df["has_incident"].sum()),
     "unique_incidents_matched": int(df["incident_id"].nunique())},
])
join_audit["incident_rows_delta_vs_nearest"] = join_audit["rows_labeled_incident"] - int(join_audit.loc[0, "rows_labeled_incident"])

print("QoS file audit:")
print(qos_file_audit[["file", "rows", "cols"]].to_string(index=False))
print("\nIncident file audit:")
print(incident_file_audit[["file", "rows", "cols", "header_only"]].to_string(index=False))
print("\nQoS quality audit:")
print(qos_quality_audit.to_string(index=False))
print("\nIncident quality audit:")
print(incident_quality_audit.to_string(index=False))
print("\nIncident labeling protocol comparison:")
print(join_audit.to_string(index=False))

fig_files = px.bar(qos_file_audit, x="file", y="rows", title="QoS Rows by Source File",
                   labels={"file": "QoS source file", "rows": "Rows"})
fig_files.update_xaxes(tickangle=-35)
fig_files.update_layout(width=1050, height=420)
save_fig(fig_files, "fig00_qos_rows_by_file"); fig_files.show()

missing = qos_df.isna().mean().sort_values(ascending=False).head(15).reset_index()
missing.columns = ["column", "missing_fraction"]
fig_missing = px.bar(missing, x="column", y="missing_fraction", title="Top QoS Missingness Rates",
                     labels={"column": "Column", "missing_fraction": "Missing fraction"})
fig_missing.update_xaxes(tickangle=-35)
fig_missing.update_layout(width=950, height=420)
save_fig(fig_missing, "fig00_qos_missingness"); fig_missing.show()

fig_join = px.bar(join_audit, x="labeling_protocol", y="rows_labeled_incident",
                  color="labeling_protocol", text="unique_incidents_matched",
                  title="Incident Labeling: Rejected Nearest Start vs Selected Active Interval",
                  labels={"labeling_protocol": "Protocol", "rows_labeled_incident": "QoS rows labeled as incident"})
fig_join.update_traces(texttemplate="matched incidents=%{text}", textposition="outside")
fig_join.update_layout(showlegend=False, width=850, height=430)
save_fig(fig_join, "fig00_incident_join_protocol_comparison"); fig_join.show()

print(f"\nSelected dataset after interval join: {len(df):,} QoS rows")
print(f"Rows inside incident intervals: {int(df['has_incident'].sum()):,}")
print(f"Unique incident_ids matched: {df['incident_id'].nunique():,} / {inc_df['incident_id'].nunique() if not inc_df.empty else 0:,}")

C:\Users\amani\AppData\Local\Temp\ipykernel_3900\106926304.py:41: UserWarning: incidents_choice_13_20260405.csv is header-only; skipping
  warnings.warn(f"{f.name} is header-only; skipping")


QoS file audit:
                                 file  rows  cols
qos_timeseries_choice_11_20260327.csv   250    82
qos_timeseries_choice_11_20260328.csv   416    82
qos_timeseries_choice_11_20260329.csv    15    82
qos_timeseries_choice_11_20260330.csv   267    83
qos_timeseries_choice_11_20260402.csv    82    83
qos_timeseries_choice_11_20260403.csv   406    83
qos_timeseries_choice_11_20260404.csv   555    83
qos_timeseries_choice_11_20260405.csv    91    83
qos_timeseries_choice_12_20260328.csv    64    82
qos_timeseries_choice_12_20260329.csv    51    82
qos_timeseries_choice_13_20260405.csv   818    83
 qos_timeseries_choice_7_20260329.csv    93    82
 qos_timeseries_choice_7_20260330.csv    20    82

Incident file audit:
                            file  rows  cols  header_only
incidents_choice_11_20260327.csv    80    10        False
incidents_choice_11_20260328.csv    42    10        False
incidents_choice_11_20260329.csv     3    10        False
incidents_choice_11_20260330.c


Selected dataset after interval join: 3,128 QoS rows
Rows inside incident intervals: 1,229
Unique incident_ids matched: 274 / 301


**Interpretation:** We treat the raw inputs as two different data objects because the outputs show they behave differently. The QoS side is a dense time series with 3,128 rows across 13 files, no duplicate `timestamp/node/cell/zone` keys, one node, one cell, and two zones. The row-volume figure shows a clear collection imbalance: `qos_timeseries_choice_13_20260405.csv` contributes 818 rows, while `qos_timeseries_choice_11_20260329.csv` contributes only 15 and `qos_timeseries_choice_7_20260330.csv` contributes 20. The missingness figure is reassuring rather than alarming: the worst missing field is `ho_success_rate_pct` at 0.0019, so we do not need broad imputation before modeling.

On the incident side, we see 301 incident rows and one header-only incident file. The duration audit explains why a simple nearest-start join is not appropriate: 243 incidents have non-positive duration and the p90 duration is 142.14 seconds, so incidents mix instantaneous alarms with interval events. We therefore keep the nearest-start method only as a rejected audit baseline. It labels 746 QoS rows and touches all 301 incident IDs, but it ignores the active incident window. The selected interval join labels 1,229 QoS rows, which is 483 more incident-context rows than nearest-start, while matching 274 incident IDs that actually overlap a QoS timestamp. We prefer this result because a time-series row should inherit an incident label only when it occurs inside that incident's active window for the same node, not merely because it is close to an incident start time.

<a id="sec-03"></a>

## 03 — Data Quality Drop

After establishing the incident alignment, we apply only the quality corrections supported by the audit. We remove unsafe training rows and post-hoc leakage fields while preserving the matched incident coverage needed for the benchmark.

In [3]:
quality_before = pd.DataFrame([
    {"stage": "before_quality_filter", "rows": len(df), "incident_rows": int(df["has_incident"].sum()),
     "unique_incidents": int(df["incident_id"].nunique()), "poor_quality_rows": int(df["data_quality_rating"].astype(str).str.lower().eq("poor").sum()),
     "skip_for_training_rows": int(df["skip_for_training"].fillna(False).astype(bool).sum())}
])

mask_skip = df["skip_for_training"].fillna(False).astype(bool)
df = df[~mask_skip].copy()
mask_poor = df["data_quality_rating"].astype(str).str.lower().eq("poor")
df = df[~mask_poor].copy()
df = df.reset_index(drop=True)

POST_HOC = ["hour_anomaly_rate", "incident_recovery_time",
            "duration_sec", "samples", "max_score", "time_to_detect_sec"]
dropped_post_hoc = [c for c in POST_HOC if c in df.columns]
df = df.drop(columns=dropped_post_hoc, errors="ignore")

quality_after = pd.DataFrame([
    {"stage": "after_quality_filter", "rows": len(df), "incident_rows": int(df["has_incident"].sum()),
     "unique_incidents": int(df["incident_id"].nunique()), "poor_quality_rows": int(df.get("data_quality_rating", pd.Series("", index=df.index)).astype(str).str.lower().eq("poor").sum()),
     "skip_for_training_rows": int(df.get("skip_for_training", pd.Series(False, index=df.index)).fillna(False).astype(bool).sum())}
])
quality_filter_audit = pd.concat([quality_before, quality_after], ignore_index=True)

print("Quality filter before/after:")
print(quality_filter_audit.to_string(index=False))
print("Dropped post-hoc leakage columns:", dropped_post_hoc if dropped_post_hoc else "none present")

fig_quality = px.bar(quality_filter_audit, x="stage", y=["rows", "incident_rows"],
                     barmode="group", title="Quality Filter Before/After",
                     labels={"value": "Rows", "stage": "Filter stage", "variable": "Count"})
fig_quality.update_layout(width=850, height=420)
save_fig(fig_quality, "fig00_quality_filter_before_after"); fig_quality.show()

Quality filter before/after:
                stage  rows  incident_rows  unique_incidents  poor_quality_rows  skip_for_training_rows
before_quality_filter  3128           1229               274                  1                       1
 after_quality_filter  3127           1228               274                  0                       0
Dropped post-hoc leakage columns: ['hour_anomaly_rate', 'incident_recovery_time']


**Interpretation:** We keep the quality correction narrow because the audit shows only a small quality problem. Before filtering, we have 3,128 rows, 1,229 incident-window rows, 274 matched incident IDs, one poor-quality row, and one `skip_for_training` row. After filtering, we have 3,127 rows and 1,228 incident-window rows, with the same 274 matched incident IDs. This indicates the filter removes one unsafe row without changing the incident coverage structure. We also drop exactly two post-hoc leakage columns, `hour_anomaly_rate` and `incident_recovery_time`, because those values describe incident evolution after the decision point. The before/after figure confirms that this step is a leakage correction, not an aggressive data reduction.

<a id="sec-04"></a>

## 04 - Data Engineering Protocol

Once the leakage fields are removed, we engineer only decision-time features. The added signals summarize cyclical time, KPI pressure, radio impairment, transport impairment, link dominance, and critical-KPI missingness without using recovery duration, future outcomes, oracle rewards, or post-action information.

In [4]:

ENGINEERED_NUMERIC_COLS = [
    "hour_sin", "hour_cos", "day_sin", "day_cos",
    "kpi_impairment_score", "transport_pressure_index",
    "radio_impairment_score", "link_signal_delta",
    "missing_critical_kpi_count",
]

def _safe_series(frame: pd.DataFrame, column: str, default: float = 0.0) -> pd.Series:
    """Return a numeric series with a default when the source column is absent."""
    if column not in frame.columns:
        return pd.Series(default, index=frame.index, dtype="float64")
    return pd.to_numeric(frame[column], errors="coerce")

def _minmax_score(series: pd.Series, lower: float, upper: float, higher_is_worse: bool = True) -> pd.Series:
    """Scale a numeric signal to [0, 1] using operational bounds."""
    values = pd.to_numeric(series, errors="coerce")
    if higher_is_worse:
        scaled = (values - lower) / max(upper - lower, 1e-9)
    else:
        scaled = (upper - values) / max(upper - lower, 1e-9)
    return scaled.clip(0.0, 1.0)

def engineer_decision_time_features(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Add leakage-safe time, pressure, radio, and missingness features."""
    out = frame.copy()
    ts = pd.to_datetime(out["timestamp"], errors="coerce")
    hour = ts.dt.hour.fillna(0).astype(float)
    day = ts.dt.dayofweek.fillna(0).astype(float)
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)
    out["day_sin"] = np.sin(2 * np.pi * day / 7.0)
    out["day_cos"] = np.cos(2 * np.pi * day / 7.0)

    latency_pressure = _minmax_score(_safe_series(out, "latency_ms"), 50, 800, higher_is_worse=True)
    jitter_pressure = _minmax_score(_safe_series(out, "jitter_ms"), 10, 250, higher_is_worse=True)
    loss_pressure = _minmax_score(_safe_series(out, "packet_loss_pct"), 0, 30, higher_is_worse=True)
    throughput_pressure = _minmax_score(_safe_series(out, "throughput_mbps"), 0, 10, higher_is_worse=False)
    queue_pressure = _minmax_score(_safe_series(out, "queue_length"), 0, 200, higher_is_worse=True)
    retrans_pressure = _minmax_score(_safe_series(out, "tcp_retransmit_rate"), 0, 20, higher_is_worse=True)
    sinr_pressure = _minmax_score(_safe_series(out, "sinr_db"), -5, 20, higher_is_worse=False)
    rssi_pressure = _minmax_score(_safe_series(out, "rssi_dbm"), -115, -45, higher_is_worse=False)
    bler_pressure = _minmax_score(_safe_series(out, "bler_proxy_pct"), 0, 30, higher_is_worse=True)

    out["kpi_impairment_score"] = pd.concat(
        [latency_pressure, jitter_pressure, loss_pressure, throughput_pressure], axis=1
    ).mean(axis=1)
    out["transport_pressure_index"] = pd.concat(
        [latency_pressure, queue_pressure, retrans_pressure, loss_pressure], axis=1
    ).mean(axis=1)
    out["radio_impairment_score"] = pd.concat(
        [sinr_pressure, rssi_pressure, bler_pressure], axis=1
    ).mean(axis=1)

    wifi_score = _safe_series(out, "wifi_signal_score")
    cell_score = _safe_series(out, "cellular_signal_score")
    out["link_signal_delta"] = (cell_score - wifi_score).fillna(0.0)

    critical = [
        "latency_ms", "jitter_ms", "packet_loss_pct", "throughput_mbps",
        "rssi_dbm", "sinr_db", "queue_length", "tcp_retransmit_rate",
    ]
    present_critical = [col for col in critical if col in out.columns]
    out["missing_critical_kpi_count"] = out[present_critical].isna().sum(axis=1) if present_critical else 0

    audit = pd.DataFrame([
        {"engineered_feature": "hour_sin/hour_cos", "purpose": "cyclical time-of-day signal", "uses_future_or_posthoc_data": False},
        {"engineered_feature": "day_sin/day_cos", "purpose": "cyclical day-of-week signal", "uses_future_or_posthoc_data": False},
        {"engineered_feature": "kpi_impairment_score", "purpose": "aggregate latency/jitter/loss/throughput impairment", "uses_future_or_posthoc_data": False},
        {"engineered_feature": "transport_pressure_index", "purpose": "queue/retransmission/latency transport pressure", "uses_future_or_posthoc_data": False},
        {"engineered_feature": "radio_impairment_score", "purpose": "SINR/RSSI/BLER radio impairment", "uses_future_or_posthoc_data": False},
        {"engineered_feature": "link_signal_delta", "purpose": "cellular-vs-Wi-Fi signal advantage", "uses_future_or_posthoc_data": False},
        {"engineered_feature": "missing_critical_kpi_count", "purpose": "model-visible data completeness signal", "uses_future_or_posthoc_data": False},
    ])
    return out, audit

df, engineering_audit = engineer_decision_time_features(df)

engineering_summary = df[ENGINEERED_NUMERIC_COLS].agg(["mean", "std", "min", "max"]).T.reset_index()
engineering_summary = engineering_summary.rename(columns={"index": "engineered_feature"}).round(4)

print("Data-engineering audit:")
print(engineering_audit.to_string(index=False))
print("\nEngineered feature summary:")
print(engineering_summary.to_string(index=False))

fig_eng = px.bar(engineering_summary, x="engineered_feature", y="mean",
                 error_y="std", title="Engineered Decision-Time Feature Summary",
                 labels={"engineered_feature": "Engineered feature", "mean": "Mean value"})
fig_eng.update_xaxes(tickangle=-35)
fig_eng.update_layout(width=1000, height=430)
save_fig(fig_eng, "fig00_engineered_feature_summary"); fig_eng.show()


Data-engineering audit:
        engineered_feature                                             purpose  uses_future_or_posthoc_data
         hour_sin/hour_cos                         cyclical time-of-day signal                        False
           day_sin/day_cos                         cyclical day-of-week signal                        False
      kpi_impairment_score aggregate latency/jitter/loss/throughput impairment                        False
  transport_pressure_index     queue/retransmission/latency transport pressure                        False
    radio_impairment_score                     SINR/RSSI/BLER radio impairment                        False
         link_signal_delta                  cellular-vs-Wi-Fi signal advantage                        False
missing_critical_kpi_count              model-visible data completeness signal                        False

Engineered feature summary:
        engineered_feature    mean     std      min     max
                  hour_

**Interpretation:** We use the engineered-feature audit to check leakage and scale before building episodes. Every engineered feature is marked as not using future or post-hoc data. The summary plot shows bounded cyclical time features between -1 and 1, a mean KPI impairment score of 0.1520, a mean transport pressure index of 0.1154, and a higher mean radio impairment score of 0.2980. The `link_signal_delta` has a wide range from -95 to 70, which is useful because it captures whether cellular or Wi-Fi has the stronger signal context. The `missing_critical_kpi_count` is 0 across the run, so missingness is not driving model behavior here. We keep these engineered signals because they summarize operational pressure without changing the label or leaking future recovery.

<a id="sec-05"></a>

## 05 - Exploratory Data Analysis (key figures)

Before constructing episodes, we examine the cleaned data distribution. The EDA figures identify common operating regimes, severe KPI tails, and rare anomaly patterns, which explains why the later benchmark uses chronological evaluation and bounded rare-class balancing instead of a naive random split.

In [5]:
# Fig 1 — KPI distributions
KPI_COLS = ["latency_ms", "jitter_ms", "throughput_mbps", "packet_loss_pct"]
fig = make_subplots(rows=2, cols=2, subplot_titles=KPI_COLS)
for i, col in enumerate(KPI_COLS):
    r, c = divmod(i, 2)
    series = df[col].dropna()
    fig.add_trace(go.Histogram(x=series, nbinsx=60, marker_color="#4C72B0",
                               showlegend=False), row=r+1, col=c+1)
    for q, dash in [(0.5,"solid"), (0.9,"dash"), (0.99,"dot")]:
        fig.add_vline(x=series.quantile(q), line=dict(color="#C44E52",dash=dash,width=1),
                      row=r+1, col=c+1)
    fig.update_xaxes(title_text=col, row=r+1, col=c+1)
style(fig, "KPI Distributions", "Long tails are the signal the agent must act on")
save_fig(fig, "fig01_kpi_distributions"); fig.show()

# Fig 2 — Anomaly distribution
rc_counts_raw = df["anomaly_type"].value_counts(dropna=False).reset_index()
rc_counts_raw.columns = ["anomaly_type", "rows"]
fig2 = px.bar(rc_counts_raw, x="anomaly_type", y="rows",
              color="rows", color_continuous_scale="Viridis",
              labels={"anomaly_type":"Anomaly type","rows":"QoS rows"})
fig2.update_xaxes(tickangle=-30)
style(fig2, "QoS Rows by Anomaly Type")
save_fig(fig2, "fig02_anomaly_distribution"); fig2.show()

**Interpretation:** We use the KPI histograms to identify the operational tails the optimizer must handle. Latency is usually modest, with a median of 45.0 ms and p90 of 115.1 ms, but the p99 rises to 628.09 ms and the maximum reaches 5,000 ms. Jitter behaves the same way: the median is 13.0 ms, p90 is 83.0 ms, p99 is 325.55 ms, and the maximum is 2,012.67 ms. Packet loss is even more skewed: the median and p90 are both 0.0%, but the p99 jumps to 69.84% and the maximum reaches 100.0%. Throughput is less pathological in the center, with a median of 6.2332 Mbps and p90 of 9.4775 Mbps, but the maximum of 31.39 Mbps shows that capacity context varies substantially across episodes.

The anomaly bar chart shows that the label space is not balanced. `weak_signal` dominates with 1,435 rows, followed by `normal` with 1,011 rows. Severe packet loss appears 203 times, while latency and jitter degradation each sit around 100 rows. Rare categories such as `congestion` with 13 rows and `statistical_outlier` with 8 rows are too small to support a naive random split or unbalanced learner. We use these plots to justify the later rare-RC balancing in train/validation and the decision to report reward/regret by policy instead of relying on raw anomaly frequency.

<a id="sec-06"></a>

## 06 - Root-Cause Vocabulary and Mock Diagnostic

The EDA motivates a bounded diagnostic contract for the bandits. We define the root-cause vocabulary, separate `RC_NONE` from actionable impairments, and audit the mock diagnostic so every policy receives a consistent root cause, confidence, causal chain, and safe action scope.

In [6]:
RC_VOCAB = [
    "RC_WEAK_SIGNAL", "RC_SINR_DEGRADED", "RC_HO_FAILURE", "RC_PRB_CONGESTION",
    "RC_TRANSPORT_DELAY", "RC_CQI_MISMATCH", "RC_COVERAGE_HOLE", "RC_CAPACITY_OVERLOAD",
]
RC_NO_PROBLEM = "RC_NONE"

RC_SCOPE = {
    "RC_WEAK_SIGNAL": "radio",
    "RC_SINR_DEGRADED": "radio",
    "RC_HO_FAILURE": "radio",
    "RC_CQI_MISMATCH": "radio",
    "RC_COVERAGE_HOLE": "radio",
    "RC_PRB_CONGESTION": "transport",
    "RC_TRANSPORT_DELAY": "transport",
    "RC_CAPACITY_OVERLOAD": "transport",
}

RC_SCHEMA = pd.DataFrame([
    {"root_cause": "RC_WEAK_SIGNAL", "scope": "radio",
     "operational_meaning": "Wi-Fi or access signal is weak but not a complete coverage hole.",
     "primary_signals": "low RSSI with still-viable cellular/RSRP context",
     "bandit_role": "select link steering, handover, or escalation actions"},
    {"root_cause": "RC_SINR_DEGRADED", "scope": "radio",
     "operational_meaning": "Radio interference degrades SINR and increases BLER.",
     "primary_signals": "SINR below 3 dB with BLER at or above 5%",
     "bandit_role": "select channel, MCS, handover, or escalation actions"},
    {"root_cause": "RC_HO_FAILURE", "scope": "radio",
     "operational_meaning": "Mobility/handover path is unreliable.",
     "primary_signals": "explicit handover failure, or low success rate when a handover was attempted",
     "bandit_role": "select neighbour reconfiguration, forced handover, or escalation"},
    {"root_cause": "RC_PRB_CONGESTION", "scope": "transport",
     "operational_meaning": "Access-resource congestion appears during peak demand; this optimizer mitigates it through traffic/load actions.",
     "primary_signals": "channel utilization at or above 70% during peak hour",
     "bandit_role": "select throttling, load balancing, or off-peak deferral"},
    {"root_cause": "RC_TRANSPORT_DELAY", "scope": "transport",
     "operational_meaning": "Queueing or path delay dominates while radio quality is not the primary limiter.",
     "primary_signals": "rolling latency at or above 200 ms with queue length at or above 50 and SINR at or above 5 dB",
     "bandit_role": "select queue management, throttling, reroute, or modem reset"},
    {"root_cause": "RC_CQI_MISMATCH", "scope": "radio",
     "operational_meaning": "CQI/MCS adaptation is too conservative for the observed radio quality.",
     "primary_signals": "low CQI and low MCS despite SINR at or above 8 dB",
     "bandit_role": "select MCS cap or channel change"},
    {"root_cause": "RC_COVERAGE_HOLE", "scope": "radio",
     "operational_meaning": "Both access signal indicators are below severe coverage thresholds.",
     "primary_signals": "RSSI at or below -100 dBm and RSRP at or below -110 dBm",
     "bandit_role": "select link switch or escalation"},
    {"root_cause": "RC_CAPACITY_OVERLOAD", "scope": "transport",
     "operational_meaning": "Aggregate demand saturates available bandwidth.",
     "primary_signals": "bandwidth utilization at or above 90% with at least 20 active connections",
     "bandit_role": "select throttling, deferral, load balancing, or escalation"},
    {"root_cause": RC_NO_PROBLEM, "scope": "none",
     "operational_meaning": "No actionable impairment is detected.",
     "primary_signals": "KPIs remain inside the nominal envelope",
     "bandit_role": "select no-op monitoring"},
])

assert len(RC_VOCAB) == len(set(RC_VOCAB)), "Root-cause vocabulary contains duplicates"
assert set(RC_VOCAB) == set(RC_SCOPE), "Every actionable root cause must have one scope"
assert RC_NO_PROBLEM not in RC_VOCAB, "RC_NONE is handled separately from actionable root causes"

@dataclass
class DiagnosticOutput:
    """Structured diagnosis contract consumed by the optimization policies."""
    primary_root_cause: str
    confidence: float
    alternatives: list[dict]
    diagnosis_scope: str
    evidence: list[str]
    causal_chain: str
    optimization_context: str
    diagnostic_trust_score: float

def _num(row: pd.Series, column: str, default: float = 0.0) -> float:
    """Read a numeric row value while treating missing and NaN values as defaults."""
    value = row.get(column, default)
    if value is None or pd.isna(value):
        return float(default)
    return float(value)

def _text(row: pd.Series, column: str, default: str = "") -> str:
    """Read a text row value while treating missing values as defaults."""
    value = row.get(column, default)
    if value is None or pd.isna(value):
        return default
    return str(value)

def _bool(row: pd.Series, column: str, default: bool = False) -> bool:
    """Read a boolean row value while treating missing values as defaults."""
    value = row.get(column, default)
    if value is None or pd.isna(value):
        return bool(default)
    return bool(value)

def _alts(*pairs):
    """Build bounded alternative root-cause confidence records."""
    return [{"cause": rc, "confidence": float(np.clip(c, 0.0, 1.0))} for rc, c in pairs]

def _sample_confidence(rng: np.random.Generator, mean: float, kappa: float = 30.0) -> float:
    """Sample a stable confidence around a threshold-rule mean."""
    a = max(1.0, mean * kappa)
    b = max(1.0, (1.0 - mean) * kappa)
    return float(np.clip(rng.beta(a, b), 0.05, 0.99))

def mock_diagnose(row: pd.Series, rng: np.random.Generator) -> DiagnosticOutput:
    """Map one decision-time KPI row to the diagnostic contract used by the bandits."""
    rssi = _num(row, "rssi_dbm")
    rsrp = _num(row, "rsrp_dbm")
    sinr = _num(row, "sinr_db")
    bler = _num(row, "bler_proxy_pct")
    cqi = _num(row, "cqi")
    mcs = _num(row, "mcs")
    channel_util = _num(row, "channel_util_pct")
    bandwidth_util = _num(row, "bandwidth_util_pct")
    queue_length = _num(row, "queue_length")
    latency_mean = _num(row, "latency_rolling_mean", _num(row, "latency_ms"))
    ho_success = _num(row, "ho_success_rate_pct", 100.0)
    ho_status = _text(row, "ho_status", "none").lower()
    handover_event = _bool(row, "handover_event")
    handover_count = _num(row, "handover_count")
    ho_attempted = handover_event or handover_count > 0 or ho_status not in {"none", "", "nan"}
    peak = _bool(row, "is_peak_hour")
    active_connections = _num(row, "active_connections")
    anomaly_flag = _bool(row, "anomaly_flag")
    anomaly_rate = _num(row, "anomaly_rate_recent")

    if rssi <= -100 and rsrp <= -110:
        rc = "RC_COVERAGE_HOLE"; conf = _sample_confidence(rng, 0.78)
        evidence = [f"RSSI={rssi:.0f} and RSRP={rsrp:.0f} are both below severe coverage thresholds"]
        chain = "Out-of-coverage -> both links degraded -> service interruption"
        alts = _alts(("RC_WEAK_SIGNAL", _sample_confidence(rng, 0.65)), ("RC_HO_FAILURE", _sample_confidence(rng, 0.30)))
        opt_ctx = "escalation_recommended"
    elif rssi <= -90 and rsrp >= -100:
        rc = "RC_WEAK_SIGNAL"; conf = _sample_confidence(rng, 0.85)
        evidence = [f"RSSI={rssi:.0f} is below -90 dBm while RSRP remains viable"]
        chain = "Weak access signal -> retransmission risk -> link steering or handover may help"
        alts = _alts(("RC_SINR_DEGRADED", _sample_confidence(rng, 0.45)), ("RC_COVERAGE_HOLE", _sample_confidence(rng, 0.25)))
        opt_ctx = "prefer_radio_first"
    elif sinr < 3 and bler >= 5:
        rc = "RC_SINR_DEGRADED"; conf = _sample_confidence(rng, 0.83)
        evidence = [f"SINR={sinr:.1f} dB is below 3 dB and BLER={bler:.1f}% is elevated"]
        chain = "Interference -> SINR drop -> BLER increase -> throughput erosion"
        alts = _alts(("RC_CQI_MISMATCH", _sample_confidence(rng, 0.50)), ("RC_WEAK_SIGNAL", _sample_confidence(rng, 0.30)))
        opt_ctx = "prefer_radio_first"
    elif ho_status == "failure" or (ho_attempted and ho_success < 80):
        rc = "RC_HO_FAILURE"; conf = _sample_confidence(rng, 0.80)
        evidence = [f"HO status={ho_status}, success rate={ho_success:.0f}%"]
        chain = "Neighbour/mobility issue -> failed handover -> dropped or unstable session"
        alts = _alts(("RC_WEAK_SIGNAL", _sample_confidence(rng, 0.40)), ("RC_COVERAGE_HOLE", _sample_confidence(rng, 0.20)))
        opt_ctx = "prefer_radio_first"
    elif channel_util >= 70 and peak:
        rc = "RC_PRB_CONGESTION"; conf = _sample_confidence(rng, 0.78)
        evidence = [f"Channel utilization={channel_util:.0f}% during peak hour"]
        chain = "Peak demand -> access-resource exhaustion -> queueing and scheduling delay"
        alts = _alts(("RC_CAPACITY_OVERLOAD", _sample_confidence(rng, 0.55)), ("RC_TRANSPORT_DELAY", _sample_confidence(rng, 0.25)))
        opt_ctx = "prefer_transport_first"
    elif latency_mean >= 200 and queue_length >= 50 and sinr >= 5:
        rc = "RC_TRANSPORT_DELAY"; conf = _sample_confidence(rng, 0.80)
        evidence = [f"Rolling latency={latency_mean:.0f} ms, queue={queue_length:.0f}, SINR={sinr:.1f} dB"]
        chain = "Queue build-up at gateway -> latency rise -> radio side not primary limiter"
        alts = _alts(("RC_PRB_CONGESTION", _sample_confidence(rng, 0.45)), ("RC_CAPACITY_OVERLOAD", _sample_confidence(rng, 0.25)))
        opt_ctx = "prefer_transport_first"
    elif cqi <= 6 and mcs <= 8 and sinr >= 8:
        rc = "RC_CQI_MISMATCH"; conf = _sample_confidence(rng, 0.72)
        evidence = [f"CQI={cqi:.0f} and MCS={mcs:.0f} are low despite SINR={sinr:.1f} dB"]
        chain = "CQI underestimate -> conservative MCS -> throughput left on table"
        alts = _alts(("RC_SINR_DEGRADED", _sample_confidence(rng, 0.40)),)
        opt_ctx = "prefer_radio_first"
    elif bandwidth_util >= 90 and active_connections >= 20:
        rc = "RC_CAPACITY_OVERLOAD"; conf = _sample_confidence(rng, 0.75)
        evidence = [f"Bandwidth utilization={bandwidth_util:.0f}% with {int(active_connections)} active connections"]
        chain = "Aggregate demand exceeds link capacity -> throughput cap and delay"
        alts = _alts(("RC_PRB_CONGESTION", _sample_confidence(rng, 0.55)),)
        opt_ctx = "prefer_transport_first"
    elif anomaly_flag and anomaly_rate > 0:
        rc = "RC_TRANSPORT_DELAY"; conf = _sample_confidence(rng, 0.40)
        evidence = ["Anomaly flag is set but KPI evidence is mixed"]
        chain = "Mixed signals -> low-confidence dispatch -> conservative transport action"
        alts = _alts(("RC_PRB_CONGESTION", _sample_confidence(rng, 0.35)), ("RC_SINR_DEGRADED", _sample_confidence(rng, 0.30)))
        opt_ctx = "defer_low_confidence"
    else:
        rc = RC_NO_PROBLEM; conf = _sample_confidence(rng, 0.92)
        evidence = ["KPIs remain inside the nominal envelope"]
        chain = "No anomaly evidence -> healthy operation"
        alts = []
        opt_ctx = "observe_only"

    scope = RC_SCOPE.get(rc, "none") if rc != RC_NO_PROBLEM else "none"
    completeness = _num(row, "data_completeness_pct", 100.0) / 100.0
    quality = {"excellent": 1.0, "fair": 0.7, "poor": 0.3}.get(
        _text(row, "data_quality_rating", "excellent").lower(), 0.7)
    trust = float(np.clip(0.6 * completeness + 0.4 * quality, 0.0, 1.0))
    return DiagnosticOutput(rc, conf, alts, scope, evidence, chain, opt_ctx, trust)

diag_rng = np.random.default_rng(RANDOM_STATE)
diag_records = [mock_diagnose(row, diag_rng) for _, row in df.iterrows()]
df["primary_rc"] = [d.primary_root_cause for d in diag_records]
df["rc_confidence"] = [d.confidence for d in diag_records]
df["diagnosis_scope"] = [d.diagnosis_scope for d in diag_records]
df["optimization_context"] = [d.optimization_context for d in diag_records]
df["diagnostic_trust_score"] = [d.diagnostic_trust_score for d in diag_records]

diagnostic_audit = (
    df["primary_rc"]
    .value_counts()
    .reindex(RC_VOCAB + [RC_NO_PROBLEM], fill_value=0)
    .rename_axis("root_cause")
    .reset_index(name="rows")
)
diagnostic_audit = diagnostic_audit.merge(RC_SCHEMA[["root_cause", "scope", "operational_meaning"]], on="root_cause", how="left")
diagnostic_confidence_audit = (
    df.groupby("primary_rc", dropna=False)
      .agg(rows=("primary_rc", "size"),
           mean_confidence=("rc_confidence", "mean"),
           mean_trust=("diagnostic_trust_score", "mean"))
      .reindex(RC_VOCAB + [RC_NO_PROBLEM])
      .fillna({"rows": 0, "mean_confidence": 0.0, "mean_trust": 0.0})
      .reset_index()
      .rename(columns={"primary_rc": "root_cause"})
)

assert set(df["primary_rc"].unique()).issubset(set(RC_VOCAB + [RC_NO_PROBLEM]))
assert df["rc_confidence"].between(0.0, 1.0).all()
assert df["diagnostic_trust_score"].between(0.0, 1.0).all()

print("Root-cause schema:")
print(RC_SCHEMA.to_string(index=False))
print("\nMock diagnostic distribution:")
print(diagnostic_audit[["root_cause", "scope", "rows", "operational_meaning"]].to_string(index=False))
print("\nDiagnostic confidence/trust audit:")
print(diagnostic_confidence_audit.round(4).to_string(index=False))

fig_rc = px.bar(diagnostic_audit, x="root_cause", y="rows", color="scope",
                title="Mock Diagnostic Root-Cause Distribution Before Episode Balancing",
                labels={"root_cause": "Root cause", "rows": "QoS rows"})
fig_rc.update_xaxes(tickangle=-35)
fig_rc.update_layout(width=1050, height=450)
save_fig(fig_rc, "fig02b_mock_diagnostic_rc_distribution"); fig_rc.show()

Root-cause schema:
          root_cause     scope                                                                                              operational_meaning                                                                               primary_signals                                                      bandit_role
      RC_WEAK_SIGNAL     radio                                                 Wi-Fi or access signal is weak but not a complete coverage hole.                                              low RSSI with still-viable cellular/RSRP context            select link steering, handover, or escalation actions
    RC_SINR_DEGRADED     radio                                                             Radio interference degrades SINR and increases BLER.                                                      SINR below 3 dB with BLER at or above 5%             select channel, MCS, handover, or escalation actions
       RC_HO_FAILURE     radio                                          

**Interpretation:** We now define the root-cause vocabulary as an explicit optimization contract rather than a loose label list. Each RC has a scope, operational meaning, primary KPI signals, and a bandit role, so the downstream policies know whether they are choosing from radio actions, transport actions, or `ACT_NO_OP`. We keep `RC_NONE` outside `RC_VOCAB` because it is not an impairment class; it is the no-action state.

The diagnostic distribution figure shows the raw mock diagnosis before episode balancing. The dominant class is `RC_TRANSPORT_DELAY` with 1,248 rows, followed by `RC_NONE` with 1,022 rows and `RC_COVERAGE_HOLE` with 834 rows. The sparse classes are visible rather than hidden: `RC_SINR_DEGRADED` has 12 rows, `RC_CQI_MISMATCH` has 7, `RC_CAPACITY_OVERLOAD` has 3, and `RC_HO_FAILURE` has 1. `RC_WEAK_SIGNAL` and `RC_PRB_CONGESTION` have 0 raw rows in this run. That imbalance is exactly why we do not rely on a naive random split and why we later add bounded synthetic examples only to train and validation. The audit also corrected a potential false-positive issue: We only treat low handover success as `RC_HO_FAILURE` when a handover was actually attempted or explicitly failed, so rows with no handover event are not mislabeled as mobility failures.

<a id="sec-07"></a>

## 07 - Action Catalogue and RC-to-Action Mask

After defining the root-cause contract, we define the remediation actions that policies may choose. The catalogue and mask validation establish the safety boundary before model training, so measured performance cannot come from unsafe or out-of-scope actions.

In [7]:
ACTIONS = [
    {"code": "ACT_NO_OP", "scope": "both", "reversible": True, "risky": False, "cost_weight": 0.00,
     "mechanism": "Take no action; observe further"},
    {"code": "ACT_BAND_STEER_5G", "scope": "radio", "reversible": True, "risky": False, "cost_weight": 0.10,
     "mechanism": "Steer client to 5 GHz band"},
    {"code": "ACT_CHANNEL_CHANGE", "scope": "radio", "reversible": True, "risky": False, "cost_weight": 0.15,
     "mechanism": "Switch Wi-Fi channel to escape interference"},
    {"code": "ACT_SWITCH_LINK_CELL", "scope": "radio", "reversible": True, "risky": False, "cost_weight": 0.20,
     "mechanism": "Set cellular as dominant link"},
    {"code": "ACT_SWITCH_LINK_WIFI", "scope": "radio", "reversible": True, "risky": False, "cost_weight": 0.20,
     "mechanism": "Set Wi-Fi as dominant link"},
    {"code": "ACT_FORCE_HO", "scope": "radio", "reversible": False, "risky": True, "cost_weight": 0.30,
     "mechanism": "Trigger handover to neighbour cell"},
    {"code": "ACT_NEIGHBOR_RECFG", "scope": "radio", "reversible": True, "risky": False, "cost_weight": 0.25,
     "mechanism": "Update neighbour list / A3 offset"},
    {"code": "ACT_MCS_CAP", "scope": "radio", "reversible": True, "risky": False, "cost_weight": 0.15,
     "mechanism": "Cap MCS to stabilise link adaptation"},
    {"code": "ACT_QOS_THROTTLE_BULK", "scope": "transport", "reversible": True, "risky": False, "cost_weight": 0.20,
     "mechanism": "Throttle bulk traffic to protect interactive flows"},
    {"code": "ACT_QUEUE_MANAGE", "scope": "transport", "reversible": True, "risky": False, "cost_weight": 0.15,
     "mechanism": "Apply AQM (CoDel) to bound queue delay"},
    {"code": "ACT_PATH_REROUTE", "scope": "transport", "reversible": True, "risky": False, "cost_weight": 0.30,
     "mechanism": "Reroute traffic via secondary path"},
    {"code": "ACT_LOAD_BALANCE", "scope": "both", "reversible": True, "risky": False, "cost_weight": 0.30,
     "mechanism": "Redistribute users across cells/APs"},
    {"code": "ACT_DEFER_OFFPEAK", "scope": "both", "reversible": True, "risky": False, "cost_weight": 0.05,
     "mechanism": "Schedule action to off-peak window"},
    {"code": "ACT_MODEM_SOFT_RESET", "scope": "transport", "reversible": False, "risky": True, "cost_weight": 0.70,
     "mechanism": "Soft-reset gateway modem (last resort)"},
    {"code": "ACT_ESCALATE", "scope": "both", "reversible": True, "risky": False, "cost_weight": 0.10,
     "mechanism": "Escalate to human operator"},
]
ACTION_BY_CODE = {a["code"]: a for a in ACTIONS}
ACTION_CODES = [a["code"] for a in ACTIONS]

RC_TO_ACTIONS = {
    "RC_WEAK_SIGNAL": ["ACT_BAND_STEER_5G", "ACT_SWITCH_LINK_CELL", "ACT_FORCE_HO", "ACT_ESCALATE"],
    "RC_SINR_DEGRADED": ["ACT_CHANNEL_CHANGE", "ACT_MCS_CAP", "ACT_FORCE_HO", "ACT_ESCALATE"],
    "RC_HO_FAILURE": ["ACT_NEIGHBOR_RECFG", "ACT_FORCE_HO", "ACT_ESCALATE"],
    "RC_PRB_CONGESTION": ["ACT_QOS_THROTTLE_BULK", "ACT_LOAD_BALANCE", "ACT_DEFER_OFFPEAK"],
    "RC_TRANSPORT_DELAY": ["ACT_QUEUE_MANAGE", "ACT_QOS_THROTTLE_BULK", "ACT_PATH_REROUTE", "ACT_MODEM_SOFT_RESET"],
    "RC_CQI_MISMATCH": ["ACT_MCS_CAP", "ACT_CHANNEL_CHANGE"],
    "RC_COVERAGE_HOLE": ["ACT_SWITCH_LINK_CELL", "ACT_SWITCH_LINK_WIFI", "ACT_ESCALATE"],
    "RC_CAPACITY_OVERLOAD": ["ACT_QOS_THROTTLE_BULK", "ACT_DEFER_OFFPEAK", "ACT_LOAD_BALANCE", "ACT_ESCALATE"],
    RC_NO_PROBLEM: ["ACT_NO_OP"],
}

def build_action_catalogue_frame(actions: list[dict]) -> pd.DataFrame:
    """Return a typed action catalogue table used by policies and audits."""
    frame = pd.DataFrame(actions).copy()
    frame["reversible"] = frame["reversible"].astype(bool)
    frame["risky"] = frame["risky"].astype(bool)
    frame["cost_weight"] = pd.to_numeric(frame["cost_weight"], errors="raise")
    return frame

def build_action_mask_frame(rc_to_actions: dict[str, list[str]]) -> pd.DataFrame:
    """Return one row per root-cause/action pair in the allowed action mask."""
    rows = []
    for rc, allowed in rc_to_actions.items():
        for action in allowed:
            action_meta = ACTION_BY_CODE[action]
            rows.append({
                "root_cause": rc,
                "rc_scope": RC_SCOPE.get(rc, "none") if rc != RC_NO_PROBLEM else "none",
                "action": action,
                "action_scope": action_meta["scope"],
                "cost_weight": action_meta["cost_weight"],
                "risky": action_meta["risky"],
                "reversible": action_meta["reversible"],
            })
    return pd.DataFrame(rows)

def validate_action_catalogue(action_frame: pd.DataFrame, mask_frame: pd.DataFrame) -> pd.DataFrame:
    """Validate action uniqueness, scope compatibility, mask coverage, and cost bounds."""
    checks = []
    checks.append({"check": "unique_action_codes", "passed": action_frame["code"].is_unique,
                   "detail": f"{action_frame['code'].nunique()} unique codes / {len(action_frame)} rows"})
    checks.append({"check": "cost_weight_in_0_1", "passed": action_frame["cost_weight"].between(0.0, 1.0).all(),
                   "detail": f"min={action_frame['cost_weight'].min():.2f}, max={action_frame['cost_weight'].max():.2f}"})
    checks.append({"check": "all_mask_actions_exist", "passed": set(mask_frame["action"]).issubset(set(action_frame["code"])),
                   "detail": f"{mask_frame['action'].nunique()} masked actions referenced"})
    actionable_rcs = set(RC_VOCAB)
    checks.append({"check": "all_actionable_rcs_have_mask", "passed": actionable_rcs.issubset(set(RC_TO_ACTIONS)),
                   "detail": f"{len(actionable_rcs)} actionable RCs checked"})
    checks.append({"check": "rc_none_only_no_op", "passed": RC_TO_ACTIONS.get(RC_NO_PROBLEM) == ["ACT_NO_OP"],
                   "detail": f"RC_NONE actions={RC_TO_ACTIONS.get(RC_NO_PROBLEM)}"})
    scope_violations = mask_frame[
        (mask_frame["root_cause"] != RC_NO_PROBLEM)
        & ~(mask_frame["action_scope"].eq(mask_frame["rc_scope"]) | mask_frame["action_scope"].eq("both"))
    ]
    checks.append({"check": "scope_compatible_mask", "passed": scope_violations.empty,
                   "detail": f"{len(scope_violations)} incompatible pairs"})
    return pd.DataFrame(checks)

action_catalogue_df = build_action_catalogue_frame(ACTIONS)
action_mask_df = build_action_mask_frame(RC_TO_ACTIONS)
action_validation = validate_action_catalogue(action_catalogue_df, action_mask_df)

if not action_validation["passed"].all():
    raise AssertionError(action_validation.loc[~action_validation["passed"]].to_string(index=False))

mask_summary = (
    action_mask_df.groupby(["root_cause", "rc_scope"], dropna=False)
    .agg(
        allowed_actions=("action", "nunique"),
        risky_actions=("risky", "sum"),
        mean_cost=("cost_weight", "mean"),
        max_cost=("cost_weight", "max"),
    )
    .reindex(pd.MultiIndex.from_tuples(
        [(rc, RC_SCOPE.get(rc, "none")) for rc in RC_VOCAB] + [(RC_NO_PROBLEM, "none")],
        names=["root_cause", "rc_scope"]
    ))
    .reset_index()
)

action_scope_summary = (
    action_catalogue_df.groupby("scope", dropna=False)
    .agg(actions=("code", "count"), risky_actions=("risky", "sum"), mean_cost=("cost_weight", "mean"))
    .reset_index()
)

mask_matrix = pd.crosstab(action_mask_df["root_cause"], action_mask_df["action"])
mask_matrix = mask_matrix.reindex(index=RC_VOCAB + [RC_NO_PROBLEM], columns=ACTION_CODES, fill_value=0)

print("Action catalogue:")
print(action_catalogue_df[["code", "scope", "reversible", "risky", "cost_weight", "mechanism"]].to_string(index=False))
print("\nAction catalogue validation:")
print(action_validation.to_string(index=False))
print("\nAction scope summary:")
print(action_scope_summary.round(4).to_string(index=False))
print("\nRC action-mask summary:")
print(mask_summary.round(4).to_string(index=False))

fig_cost = px.bar(
    action_catalogue_df.sort_values("cost_weight"),
    x="code", y="cost_weight", color="scope", pattern_shape="risky",
    title="Action Catalogue Cost and Risk Profile",
    labels={"code": "Action", "cost_weight": "Cost weight", "scope": "Action scope", "risky": "Risky"},
)
fig_cost.update_xaxes(tickangle=-35)
fig_cost.update_layout(width=1100, height=460)
save_fig(fig_cost, "fig02c_action_cost_risk_profile"); fig_cost.show()

fig_mask = px.imshow(
    mask_matrix.values,
    x=list(mask_matrix.columns),
    y=list(mask_matrix.index),
    color_continuous_scale=[[0, "#f5f5f5"], [1, "#2166ac"]],
    text_auto=True,
    aspect="auto",
    title="RC-to-Action Mask: Allowed Actions by Root Cause",
    labels={"x": "Action", "y": "Root cause", "color": "Allowed"},
)
fig_mask.update_layout(width=1200, height=560)
save_fig(fig_mask, "fig02d_rc_action_mask_heatmap"); fig_mask.show()

fig_mask_summary = px.bar(
    mask_summary,
    x="root_cause", y="allowed_actions", color="rc_scope",
    text="risky_actions",
    title="Allowed Action Count and Risk Exposure by Root Cause",
    labels={"root_cause": "Root cause", "allowed_actions": "Allowed actions", "risky_actions": "Risky actions"},
)
fig_mask_summary.update_traces(texttemplate="risky=%{text}", textposition="outside")
fig_mask_summary.update_xaxes(tickangle=-35)
fig_mask_summary.update_layout(width=1050, height=450)
save_fig(fig_mask_summary, "fig02e_rc_action_mask_summary"); fig_mask_summary.show()

Action catalogue:
                 code     scope  reversible  risky  cost_weight                                          mechanism
            ACT_NO_OP      both        True  False       0.0000                    Take no action; observe further
    ACT_BAND_STEER_5G     radio        True  False       0.1000                         Steer client to 5 GHz band
   ACT_CHANNEL_CHANGE     radio        True  False       0.1500        Switch Wi-Fi channel to escape interference
 ACT_SWITCH_LINK_CELL     radio        True  False       0.2000                      Set cellular as dominant link
 ACT_SWITCH_LINK_WIFI     radio        True  False       0.2000                         Set Wi-Fi as dominant link
         ACT_FORCE_HO     radio       False   True       0.3000                 Trigger handover to neighbour cell
   ACT_NEIGHBOR_RECFG     radio        True  False       0.2500                  Update neighbour list / A3 offset
          ACT_MCS_CAP     radio        True  False       0.150

**Interpretation:** We use the action catalogue as the safety boundary for every optimizer in the notebook. The validation table confirms the mask is internally consistent: We have 15 unique action codes, all cost weights are inside the allowed range from 0.00 to 0.70, every masked action exists in the catalogue, all 8 actionable root causes have a mask, `RC_NONE` is restricted to `ACT_NO_OP`, and there are 0 scope-incompatible RC/action pairs. This means the bandits and ranker cannot win by choosing an action that is outside the diagnosed root cause's operational scope.

The cost/risk profile figure shows that most actions are reversible and low-to-moderate cost. The cheapest action is `ACT_NO_OP` at 0.00, followed by `ACT_DEFER_OFFPEAK` at 0.05 and low-cost escalation or band steering at 0.10. The expensive end is intentionally narrow: `ACT_FORCE_HO` is risky with cost 0.30, and `ACT_MODEM_SOFT_RESET` is both risky and the most expensive action at 0.70. We interpret that as a deployment safeguard: disruptive actions exist, but only specific RC masks expose them.

The RC-to-action heatmap shows the exact choice set each model receives. Most actionable RCs have 3 or 4 actions, while `RC_CQI_MISMATCH` has only 2 (`ACT_MCS_CAP` and `ACT_CHANNEL_CHANGE`) and `RC_NONE` has only 1 (`ACT_NO_OP`). The mask-summary figure makes the risk exposure clear. `RC_TRANSPORT_DELAY` has the highest average allowed-action cost at 0.3375 and max cost 0.70 because it can select `ACT_MODEM_SOFT_RESET`. Radio classes that involve weak signal, SINR degradation, or handover failure each expose one risky action through `ACT_FORCE_HO`, while `RC_COVERAGE_HOLE`, `RC_PRB_CONGESTION`, and `RC_CAPACITY_OVERLOAD` expose no risky actions. We keep this mask because the benchmark is testing safe diagnosis-conditioned optimization, not unconstrained action search.

<a id="sec-08"></a>

## 08 - Reward Logic

With root causes and actions fixed, we define reward as an explicit benchmark contract rather than as an observed causal recovery claim. We audit three reward views: an intent-based rule oracle for policy training and evaluation, a recovery-cost-safety composite for sensitivity analysis, and an outcome-delta score used only when a pre/post state exists or is simulated.

In [8]:
DEFAULT_REWARD   = 0.25
DEFAULT_CITATION = "no rule fired; conservative default"

@dataclass(frozen=True)
class RewardRule:
    """One ordered KPI-conditional rule for a root-cause/action reward pair."""
    when:           Callable[[dict], bool]
    reward:         float
    citation:       str
    recovery_kpi:   float = 0.6
    escalation_cost:float = 0.2
    safety_margin:  float = 0.2
    def __post_init__(self):
        """Validate that objective metadata forms one complete rule profile."""
        total = self.recovery_kpi + self.escalation_cost + self.safety_margin
        if not (0.99 <= total <= 1.01):
            raise ValueError(f"Objective weights must sum to 1.0, got {total:.3f}")

def _always(_):
    """Return True for fallback reward rules."""
    return True
def _is_realtime(s):
    """Return whether traffic is latency-sensitive."""
    return s.get("traffic_type") in {"video_call","gaming","streaming"}
def _is_bulk(s):
    """Return whether traffic is bulk-transfer traffic."""
    return s.get("traffic_type") in {"file_transfer"}
def _is_peak(s):
    """Return whether the episode occurs during peak hour."""
    return bool(s.get("is_peak_hour", False))
def _link_cell(s):
    """Return whether cellular is the dominant link."""
    return str(s.get("signal_dominant_link","")).lower().startswith("cell")
def _link_wifi(s):
    """Return whether Wi-Fi is the dominant link."""
    return str(s.get("signal_dominant_link","")).lower().startswith("wi")

REWARD_RULES: dict[tuple[str,str], list[RewardRule]] = {
    # RC_WEAK_SIGNAL
    ("RC_WEAK_SIGNAL","ACT_BAND_STEER_5G"): [
        RewardRule(lambda s: s.get("rssi_dbm",0) > -100 and s.get("sinr_db",0) >= 3,
                   0.85, "3GPP TS 36.304: inter-band reselection viable when RSSI recoverable"),
        RewardRule(lambda s: s.get("rssi_dbm",0) <= -105,
                   0.15, "5G band steer futile when RSSI below -105 dBm"),
        RewardRule(_always, 0.45, "mid-range RSSI, band steer marginal"),
    ],
    ("RC_WEAK_SIGNAL","ACT_SWITCH_LINK_CELL"): [
        RewardRule(lambda s: _link_wifi(s) and s.get("rssi_dbm",0) < -100,
                   0.80, "IEEE 802.11: fallback to cellular when Wi-Fi RSSI collapses"),
        RewardRule(lambda s: _link_cell(s), 0.35, "already on cellular; no-op"),
        RewardRule(_always, 0.50, "neutral link switch"),
    ],
    ("RC_WEAK_SIGNAL","ACT_FORCE_HO"): [
        RewardRule(lambda s: s.get("rssi_dbm",0) <= -100 and s.get("neighbor_count",0) >= 1
                             and s.get("ho_success_rate_pct",100) >= 70,
                   0.88, "3GPP TS 36.331 A3: forced HO effective with viable neighbors"),
        RewardRule(lambda s: s.get("neighbor_count",0) == 0, 0.10, "no neighbour; HO will fail"),
        RewardRule(_always, 0.55, "HO plausibly useful"),
    ],
    ("RC_WEAK_SIGNAL","ACT_ESCALATE"): [
        RewardRule(lambda s: s.get("rssi_dbm",0) <= -112 and s.get("ho_success_rate_pct",100) < 30,
                   0.80, "radio dead + HO broken: mechanical options exhausted"),
        RewardRule(_always, 0.30, "escalation is a floor"),
    ],
    # RC_SINR_DEGRADED
    ("RC_SINR_DEGRADED","ACT_CHANNEL_CHANGE"): [
        RewardRule(lambda s: _link_wifi(s) and s.get("channel_util_pct",0) >= 70,
                   0.85, "IEEE 802.11 DCF: channel change resolves co-channel interference"),
        RewardRule(lambda s: _link_cell(s), 0.35, "cellular channel is operator-controlled"),
        RewardRule(_always, 0.45, "channel change mid-utility"),
    ],
    ("RC_SINR_DEGRADED","ACT_MCS_CAP"): [
        RewardRule(lambda s: s.get("bler_proxy_pct",0) >= 10 and s.get("mcs",0) >= 20,
                   0.82, "3GPP TR 36.942: capping MCS lowers BLER under interference"),
        RewardRule(lambda s: s.get("bler_proxy_pct",0) < 3, 0.25, "BLER already low"),
        RewardRule(_always, 0.50, "MCS cap moderate utility"),
    ],
    ("RC_SINR_DEGRADED","ACT_FORCE_HO"): [
        RewardRule(lambda s: s.get("neighbor_count",0) >= 1 and s.get("sinr_db",0) < 0,
                   0.78, "SINR below 0 dB: HO to cleaner neighbour"),
        RewardRule(_always, 0.40, "HO helpful only when SINR severely degraded"),
    ],
    ("RC_SINR_DEGRADED","ACT_ESCALATE"): [
        RewardRule(lambda s: s.get("sinr_db",0) < -2 and s.get("channel_util_pct",0) >= 85,
                   0.70, "channel saturated and SINR negative: network fix required"),
        RewardRule(_always, 0.28, "escalate floor"),
    ],
    # RC_HO_FAILURE
    ("RC_HO_FAILURE","ACT_NEIGHBOR_RECFG"): [
        RewardRule(lambda s: s.get("ho_success_rate_pct",100) < 60 and s.get("neighbor_count",0) >= 1,
                   0.85, "3GPP TS 36.331: updating neighbour list restores HO reliability"),
        RewardRule(_always, 0.50, "neighbour reconfig moderate utility"),
    ],
    ("RC_HO_FAILURE","ACT_FORCE_HO"): [
        RewardRule(lambda s: s.get("ho_success_rate_pct",100) >= 60, 0.60, "HO path still functional"),
        RewardRule(lambda s: s.get("ho_success_rate_pct",100) < 30, 0.15, "HO path broken"),
        RewardRule(_always, 0.40, "HO retry mid-utility"),
    ],
    ("RC_HO_FAILURE","ACT_ESCALATE"): [
        RewardRule(lambda s: s.get("ho_success_rate_pct",100) < 20, 0.78, "HO catastrophically broken"),
        RewardRule(_always, 0.30, "escalate floor"),
    ],
    # RC_PRB_CONGESTION
    ("RC_PRB_CONGESTION","ACT_QOS_THROTTLE_BULK"): [
        RewardRule(lambda s: _is_realtime(s), 0.88, "DiffServ EF: protect real-time by throttling bulk"),
        RewardRule(lambda s: _is_bulk(s), 0.30, "throttling self is counterproductive"),
        RewardRule(_always, 0.40, "throttle moderate"),
    ],
    ("RC_PRB_CONGESTION","ACT_LOAD_BALANCE"): [
        RewardRule(lambda s: s.get("bandwidth_util_pct",0) >= 80 and s.get("active_connections",0) >= 5,
                   0.82, "3GPP TS 36.300 MLB: redistribute UEs when load imbalanced"),
        RewardRule(_always, 0.45, "load balance moderate"),
    ],
    ("RC_PRB_CONGESTION","ACT_DEFER_OFFPEAK"): [
        RewardRule(lambda s: _is_bulk(s) and _is_peak(s), 0.88, "bulk traffic deferrable off-peak"),
        RewardRule(lambda s: _is_realtime(s), 0.10, "real-time cannot be deferred"),
        RewardRule(_always, 0.35, "deferral mid-utility"),
    ],
    # RC_TRANSPORT_DELAY
    ("RC_TRANSPORT_DELAY","ACT_QUEUE_MANAGE"): [
        RewardRule(lambda s: s.get("queue_length",0) >= 100 and s.get("jitter_ms",0) >= 100,
                   0.85, "RFC 7567 AQM: CoDel resolves bufferbloat when queue deep"),
        RewardRule(_always, 0.50, "queue tune moderate"),
    ],
    ("RC_TRANSPORT_DELAY","ACT_QOS_THROTTLE_BULK"): [
        RewardRule(lambda s: _is_realtime(s) and s.get("latency_ms",0) > 500,
                   0.78, "throttle bulk to free transport for real-time"),
        RewardRule(_always, 0.45, "throttle moderate"),
    ],
    ("RC_TRANSPORT_DELAY","ACT_PATH_REROUTE"): [
        RewardRule(lambda s: s.get("tcp_retransmit_rate",0) >= 10 and s.get("latency_ms",0) >= 800,
                   0.82, "RFC 4271 BGP alt-path: reroute when current path unreliable"),
        RewardRule(_always, 0.40, "reroute moderate"),
    ],
    ("RC_TRANSPORT_DELAY","ACT_MODEM_SOFT_RESET"): [
        RewardRule(lambda s: s.get("throughput_mbps",0) < 0.5 and s.get("packet_loss_pct",0) >= 50,
                   0.75, "modem hang signature: near-zero throughput + massive loss"),
        RewardRule(_always, 0.15, "modem reset high-cost; reserve for clear hang"),
    ],
    # RC_CQI_MISMATCH
    ("RC_CQI_MISMATCH","ACT_MCS_CAP"): [
        RewardRule(lambda s: s.get("bler_proxy_pct",0) >= 10 and s.get("mcs",0) >= 20,
                   0.85, "3GPP TR 36.942: CQI-MCS mismatch resolved by capping MCS"),
        RewardRule(_always, 0.45, "MCS cap moderate"),
    ],
    ("RC_CQI_MISMATCH","ACT_CHANNEL_CHANGE"): [
        RewardRule(lambda s: _link_wifi(s) and s.get("channel_util_pct",0) >= 70,
                   0.70, "Wi-Fi CQI-like mismatch from channel contention"),
        RewardRule(_always, 0.40, "channel change mid-utility"),
    ],
    # RC_COVERAGE_HOLE
    ("RC_COVERAGE_HOLE","ACT_SWITCH_LINK_CELL"): [
        RewardRule(lambda s: _link_wifi(s) and s.get("cellular_signal_score",0) >= 40,
                   0.80, "Wi-Fi hole; cellular fallback viable"),
        RewardRule(lambda s: _link_cell(s), 0.20, "already on cellular"),
        RewardRule(_always, 0.40, "link switch moderate"),
    ],
    ("RC_COVERAGE_HOLE","ACT_SWITCH_LINK_WIFI"): [
        RewardRule(lambda s: _link_cell(s) and s.get("wifi_signal_score",0) >= 40,
                   0.80, "cellular hole; Wi-Fi fallback viable"),
        RewardRule(lambda s: _link_wifi(s), 0.20, "already on Wi-Fi"),
        RewardRule(_always, 0.40, "link switch moderate"),
    ],
    ("RC_COVERAGE_HOLE","ACT_ESCALATE"): [
        RewardRule(lambda s: s.get("wifi_signal_score",0) < 30 and s.get("cellular_signal_score",0) < 30,
                   0.78, "both links dark: coverage gap requires network planning"),
        RewardRule(_always, 0.28, "escalate floor"),
    ],
    # RC_CAPACITY_OVERLOAD
    ("RC_CAPACITY_OVERLOAD","ACT_QOS_THROTTLE_BULK"): [
        RewardRule(lambda s: _is_realtime(s), 0.80, "protect RT flows under overload"),
        RewardRule(lambda s: _is_bulk(s), 0.35, "throttling bulk while caller is bulk harms throughput"),
        RewardRule(_always, 0.40, "throttle moderate"),
    ],
    ("RC_CAPACITY_OVERLOAD","ACT_DEFER_OFFPEAK"): [
        RewardRule(lambda s: _is_bulk(s) and _is_peak(s), 0.85, "deferral canonical bulk-overload response"),
        RewardRule(lambda s: _is_realtime(s), 0.10, "cannot defer real-time"),
        RewardRule(_always, 0.40, "deferral mid-utility"),
    ],
    ("RC_CAPACITY_OVERLOAD","ACT_LOAD_BALANCE"): [
        RewardRule(lambda s: s.get("bandwidth_util_pct",0) >= 95, 0.30, "cell saturated network-wide; MLB cannot find headroom"),
        RewardRule(lambda s: s.get("bandwidth_util_pct",0) >= 85 and s.get("active_connections",0) >= 8,
                   0.80, "MLB under true overload with multiple UEs"),
        RewardRule(_always, 0.45, "MLB moderate"),
    ],
    ("RC_CAPACITY_OVERLOAD","ACT_ESCALATE"): [
        RewardRule(lambda s: s.get("bandwidth_util_pct",0) >= 95 and s.get("active_connections",0) >= 10,
                   0.82, "sustained saturation: capacity planning required"),
        RewardRule(_always, 0.28, "escalate floor"),
    ],
    # RC_NONE
    ("RC_NONE","ACT_NO_OP"): [
        RewardRule(_always, 0.90, "no problem: no action is the correct action"),
    ],
}
def build_reward_rule_frame(reward_rules: dict[tuple[str, str], list[RewardRule]]) -> pd.DataFrame:
    """Flatten reward rules into one auditable row per ordered rule."""
    rows = []
    for (rc, action), rules in reward_rules.items():
        meta = ACTION_BY_CODE.get(action, {})
        for order, rule in enumerate(rules, start=1):
            rows.append({
                "root_cause": rc,
                "action": action,
                "rule_order": order,
                "reward": rule.reward,
                "citation": rule.citation,
                "recovery_kpi": rule.recovery_kpi,
                "escalation_cost": rule.escalation_cost,
                "safety_margin": rule.safety_margin,
                "objective_sum": rule.recovery_kpi + rule.escalation_cost + rule.safety_margin,
                "action_scope": meta.get("scope", "unknown"),
                "action_cost": meta.get("cost_weight", np.nan),
                "action_risky": meta.get("risky", False),
            })
    return pd.DataFrame(rows)

def validate_reward_rules(reward_rules: dict[tuple[str, str], list[RewardRule]]) -> pd.DataFrame:
    """Validate reward-rule coverage, bounds, objective weights, and citations."""
    allowed_pairs = {(rc, action) for rc, actions in RC_TO_ACTIONS.items() for action in actions}
    reward_pairs = set(reward_rules)
    rule_frame = build_reward_rule_frame(reward_rules)
    checks = [
        {"check": "all_allowed_pairs_have_rules", "passed": allowed_pairs.issubset(reward_pairs),
         "detail": f"missing={len(allowed_pairs - reward_pairs)}"},
        {"check": "no_rules_for_disallowed_pairs", "passed": reward_pairs.issubset(allowed_pairs),
         "detail": f"extra={len(reward_pairs - allowed_pairs)}"},
        {"check": "reward_values_in_0_1", "passed": rule_frame["reward"].between(0.0, 1.0).all(),
         "detail": f"min={rule_frame['reward'].min():.2f}, max={rule_frame['reward'].max():.2f}"},
        {"check": "objective_weights_sum_to_1", "passed": np.isclose(rule_frame["objective_sum"], 1.0, atol=0.01).all(),
         "detail": f"min={rule_frame['objective_sum'].min():.2f}, max={rule_frame['objective_sum'].max():.2f}"},
        {"check": "citations_present", "passed": rule_frame["citation"].astype(str).str.len().gt(0).all(),
         "detail": f"rules={len(rule_frame)}"},
        {"check": "default_reward_in_0_1", "passed": 0.0 <= DEFAULT_REWARD <= 1.0,
         "detail": f"default={DEFAULT_REWARD:.2f}"},
    ]
    return pd.DataFrame(checks)

reward_rule_frame = build_reward_rule_frame(REWARD_RULES)
reward_validation = validate_reward_rules(REWARD_RULES)
if not reward_validation["passed"].all():
    raise AssertionError(reward_validation.loc[~reward_validation["passed"]].to_string(index=False))

reward_pair_summary = (
    reward_rule_frame.groupby(["root_cause", "action"], dropna=False)
    .agg(
        rules=("reward", "size"),
        min_reward=("reward", "min"),
        max_reward=("reward", "max"),
        mean_reward=("reward", "mean"),
        action_cost=("action_cost", "first"),
        action_risky=("action_risky", "first"),
    )
    .reset_index()
)
reward_coverage_summary = (
    reward_pair_summary.groupby("root_cause", dropna=False)
    .agg(
        action_pairs=("action", "nunique"),
        total_rules=("rules", "sum"),
        max_pair_reward=("max_reward", "max"),
        min_pair_reward=("min_reward", "min"),
    )
    .reindex(RC_VOCAB + [RC_NO_PROBLEM])
    .reset_index()
    .rename(columns={"index": "root_cause"})
)

reward_envelope = reward_pair_summary.pivot(index="root_cause", columns="action", values="max_reward")
reward_envelope = reward_envelope.reindex(index=RC_VOCAB + [RC_NO_PROBLEM], columns=ACTION_CODES)

print("Reward-rule validation:")
print(reward_validation.to_string(index=False))
print("\nReward-rule coverage by root cause:")
print(reward_coverage_summary.round(4).to_string(index=False))
print("\nReward-rule pair summary:")
print(reward_pair_summary.round(4).to_string(index=False))

fig_rule_counts = px.bar(
    reward_coverage_summary,
    x="root_cause", y="total_rules", color="action_pairs",
    title="Reward Rule Coverage by Root Cause",
    labels={"root_cause": "Root cause", "total_rules": "Reward rules", "action_pairs": "Action pairs"},
)
fig_rule_counts.update_xaxes(tickangle=-35)
fig_rule_counts.update_layout(width=1050, height=450)
save_fig(fig_rule_counts, "fig02f_reward_rule_coverage"); fig_rule_counts.show()

fig_reward_env = px.imshow(
    reward_envelope.values,
    x=list(reward_envelope.columns),
    y=list(reward_envelope.index),
    color_continuous_scale="Viridis",
    text_auto=".2f",
    aspect="auto",
    title="Reward Envelope: Maximum Rule Reward for Each Allowed RC-Action Pair",
    labels={"x": "Action", "y": "Root cause", "color": "Max reward"},
)
fig_reward_env.update_layout(width=1200, height=560)
save_fig(fig_reward_env, "fig02g_reward_envelope_heatmap"); fig_reward_env.show()

fig_reward_cost = px.scatter(
    reward_pair_summary,
    x="action_cost", y="max_reward", color="root_cause", symbol="action_risky",
    hover_data=["action", "rules", "min_reward", "mean_reward"],
    title="Reward Envelope vs Action Cost",
    labels={"action_cost": "Action cost weight", "max_reward": "Maximum reward available"},
)
fig_reward_cost.update_layout(width=900, height=500)
save_fig(fig_reward_cost, "fig02h_reward_vs_action_cost"); fig_reward_cost.show()

Reward-rule validation:
                        check  passed             detail
 all_allowed_pairs_have_rules    True          missing=0
no_rules_for_disallowed_pairs    True            extra=0
         reward_values_in_0_1    True min=0.10, max=0.90
   objective_weights_sum_to_1    True min=1.00, max=1.00
            citations_present    True           rules=68
        default_reward_in_0_1    True       default=0.25

Reward-rule coverage by root cause:
          root_cause  action_pairs  total_rules  max_pair_reward  min_pair_reward
      RC_WEAK_SIGNAL             4           11           0.8800           0.1000
    RC_SINR_DEGRADED             4           10           0.8500           0.2500
       RC_HO_FAILURE             3            7           0.8500           0.1500
   RC_PRB_CONGESTION             3            8           0.8800           0.1000
  RC_TRANSPORT_DELAY             4            8           0.8500           0.1500
     RC_CQI_MISMATCH             2            4 

**Interpretation:** We audit the reward table before any model uses it. The validation table passes every check: the reward rules cover all allowed RC-action pairs with 0 missing pairs and 0 extra disallowed pairs, every reward is bounded between 0.10 and 0.90, the objective weights sum to 1.00, all 68 rules carry citations, and the conservative default reward is 0.25. This indicates the benchmark reward surface is explicit and auditable instead of being hidden inside model code.

The reward-coverage figure shows how much decision detail we give each root cause. `RC_WEAK_SIGNAL` and `RC_CAPACITY_OVERLOAD` have the richest reward descriptions with 11 ordered rules each, while `RC_CQI_MISMATCH` has 4 rules across 2 actions and `RC_NONE` has exactly 1 rule for `ACT_NO_OP`. We interpret that spread as intentional: complex and safety-sensitive classes need more conditional reward logic, while no-problem episodes must stay narrow.

The reward-envelope heatmap shows the maximum reward available inside each allowed mask. `RC_NONE` reaches 0.90 only through `ACT_NO_OP`, congestion can reach 0.88 through `ACT_QOS_THROTTLE_BULK` or `ACT_DEFER_OFFPEAK`, weak signal can reach 0.88 through `ACT_FORCE_HO`, and transport delay reaches 0.85 through `ACT_QUEUE_MANAGE`. The reward-versus-cost figure is the safety check: `ACT_MODEM_SOFT_RESET` is exposed only for transport delay, carries the highest cost at 0.70, and still has a limited reward range from 0.15 to 0.75. `ACT_FORCE_HO` is also marked risky, so we keep it available only where radio or handover logic can justify it.

In [9]:
def oracle_reward(rc: str, action: str, state: dict) -> tuple[float, str]:
    """Return the intent-based oracle reward and citation for one state/action pair."""
    rules = REWARD_RULES.get((rc, action))
    if rules is None:
        return DEFAULT_REWARD, DEFAULT_CITATION
    for rule in rules:
        if rule.when(state):
            return rule.reward, rule.citation
    return DEFAULT_REWARD, DEFAULT_CITATION

def action_cost_score(action: str) -> float:
    """Convert an action cost weight into a reward-style score where higher is better."""
    return float(np.clip(1.0 - ACTION_BY_CODE.get(action, {}).get("cost_weight", 1.0), 0.0, 1.0))

def action_safety_score(action: str) -> float:
    """Convert action risk/reversibility into a reward-style safety score."""
    meta = ACTION_BY_CODE.get(action, {})
    if meta.get("risky", False):
        return 0.25
    if not meta.get("reversible", True):
        return 0.60
    return 1.00

def oracle_reward_multiobjective(
    rc: str, action: str, state: dict,
    w_recovery: float = 0.6, w_cost: float = 0.2, w_safety: float = 0.2,
) -> tuple[float, str]:
    """Return a recovery/cost/safety composite reward for one state/action pair."""
    total = w_recovery + w_cost + w_safety
    if abs(total - 1.0) >= 0.01:
        raise ValueError("Objective weights must sum to 1")
    recovery_reward, citation = oracle_reward(rc, action, state)
    composite = (
        w_recovery * recovery_reward
        + w_cost * action_cost_score(action)
        + w_safety * action_safety_score(action)
    )
    return float(np.clip(composite, 0.0, 1.0)), citation

KPI_IMPROVEMENT_TARGETS = {
    "latency_ms": "lower", "jitter_ms": "lower", "packet_loss_pct": "lower",
    "queue_length": "lower", "bler_proxy_pct": "lower",
    "rssi_dbm": "higher", "sinr_db": "higher",
    "throughput_mbps": "higher", "ho_success_rate_pct": "higher",
}

def oracle_reward_outcome(pre_state: dict, post_state: dict) -> float:
    """Return the mean bounded KPI-improvement score between pre and post state."""
    delta_scores = []
    for kpi, direction in KPI_IMPROVEMENT_TARGETS.items():
        before = pre_state.get(kpi)
        after = post_state.get(kpi)
        if before is None or after is None:
            continue
        if direction == "lower":
            imp = max(0.0, min(1.0, (float(before) - float(after)) / max(abs(float(before)), 1e-6)))
        else:
            imp = max(0.0, min(1.0, (float(after) - float(before)) / max(abs(float(before)), 1e-6)))
        delta_scores.append(imp)
    return float(np.mean(delta_scores)) if delta_scores else DEFAULT_REWARD

_base = {"traffic_type": "video_call", "is_peak_hour": True, "bandwidth_util_pct": 82,
         "active_connections": 6, "rssi_dbm": -80, "sinr_db": 15,
         "signal_dominant_link": "cellular", "latency_ms": 60, "jitter_ms": 12,
         "packet_loss_pct": 0, "throughput_mbps": 8, "queue_length": 20,
         "tcp_retransmit_rate": 1, "bler_proxy_pct": 2, "ho_success_rate_pct": 95,
         "neighbor_count": 2, "cqi": 10, "mcs": 15, "channel_util_pct": 30,
         "wifi_signal_score": 60, "cellular_signal_score": 70}
r_int, cite = oracle_reward("RC_PRB_CONGESTION", "ACT_QOS_THROTTLE_BULK", _base)
r_mo, _ = oracle_reward_multiobjective("RC_PRB_CONGESTION", "ACT_QOS_THROTTLE_BULK", _base)
pre = {**_base, "latency_ms": 200}
post = {**_base, "latency_ms": 80}
r_ob = oracle_reward_outcome(pre, post)
reward_smoke_audit = pd.DataFrame([
    {"reward_view": "intent_oracle", "value": r_int, "meaning": "KPI-conditional recovery rule"},
    {"reward_view": "multi_objective", "value": r_mo, "meaning": "60% recovery + 20% low cost + 20% safety"},
    {"reward_view": "outcome_delta_demo", "value": r_ob, "meaning": "mean KPI improvement in a simulated post-state"},
]).round(4)

print("Reward smoke audit:")
print(reward_smoke_audit.to_string(index=False))
print(f"Intent citation: {cite[:90]}")
assert reward_smoke_audit["value"].between(0.0, 1.0).all()

fig_smoke = px.bar(
    reward_smoke_audit,
    x="reward_view", y="value", color="reward_view",
    title="Reward Smoke Check Across Reward Views",
    labels={"reward_view": "Reward view", "value": "Reward"},
)
fig_smoke.update_layout(showlegend=False, width=750, height=420)
save_fig(fig_smoke, "fig02i_reward_smoke_check"); fig_smoke.show()

Reward smoke audit:
       reward_view  value                                        meaning
     intent_oracle 0.8800                  KPI-conditional recovery rule
   multi_objective 0.8880       60% recovery + 20% low cost + 20% safety
outcome_delta_demo 0.0667 mean KPI improvement in a simulated post-state
Intent citation: DiffServ EF: protect real-time by throttling bulk


**Interpretation:** We use this smoke audit as a sanity check for the reward engine before the bandits see it. All three values are computed on the same PRB-congestion scenario, so the differences come from the reward definition rather than from a different episode. The intent oracle gives `ACT_QOS_THROTTLE_BULK` a high reward of 0.8800 because the state is a peak-hour `video_call`; in this context, throttling bulk traffic protects the latency-sensitive flow and matches the strongest congestion rule. This is the reward view we use for the main benchmark because it directly tests whether a policy can choose the action that the expert RC-action contract considers operationally appropriate.

The multi-objective reward is 0.8880, slightly higher than the intent score. That increase is expected: the action already has strong recovery value, its cost is moderate rather than disruptive, and it is not marked risky, so the cost and safety terms reinforce the same decision instead of changing it. We interpret this as a useful consistency check: the multi-objective view does not contradict the intent oracle on this example, but it would penalize expensive or risky actions in cases where two actions have similar recovery scores.

The outcome-delta demo is much lower at 0.0667 because it measures a small simulated KPI improvement, not an observed intervention result. We do not train or rank the bandits with this value because the notebook has no real post-action logs. This keeps the scientific claim honest: the benchmark evaluates expert-aligned decision quality under a clean reward rule engine, while causal recovery remains future work that requires real before/after action outcomes.

<a id="sec-09"></a>

## 09 - Episode Construction and Temporal Split

The cleaned data, diagnostic contract, action mask, and reward engine now become contextual-bandit episodes. We create one peak incident episode per matched incident, add matched no-problem episodes, balance only train and validation for rare root causes, and keep the final test set as future real data.

In [10]:
EPISODE_STATE_KEYS = [
    "latency_ms","jitter_ms","packet_loss_pct","throughput_mbps",
    "rssi_dbm","rsrp_dbm","rsrq_db","sinr_db","cqi","mcs",
    "bler_proxy_pct","ho_success_rate_pct","neighbor_count",
    "channel_util_pct","queue_length","tcp_retransmit_rate",
    "bandwidth_util_pct","active_connections",
    "wifi_signal_score","cellular_signal_score",
    "signal_dominant_link","traffic_type","is_peak_hour",
    "zone_id","hour_of_day","day_of_week",
    "hour_sin","hour_cos","day_sin","day_cos",
    "kpi_impairment_score","transport_pressure_index",
    "radio_impairment_score","link_signal_delta",
    "missing_critical_kpi_count",
]

@dataclass
class Episode:
    """One contextual-bandit decision point with diagnosis, action mask, and state."""
    episode_id:     str
    timestamp:      pd.Timestamp
    zone_id:        str
    primary_rc:     str
    rc_confidence:  float
    allowed_actions:list
    state:          dict
    severity:       str
    source_file:    str
    is_incident:    bool
    is_synthetic:   bool = False

def _row_to_state(row: pd.Series) -> dict:
    """Extract only decision-time state fields from a QoS row."""
    out = {}
    for k in EPISODE_STATE_KEYS:
        if k not in row.index:
            continue
        v = row[k]
        out[k] = None if pd.isna(v) else (v.item() if hasattr(v, "item") else v)
    return out

INCIDENT_TYPE_TO_RC = {
    "weak_signal":"RC_WEAK_SIGNAL","latency_degradation":"RC_TRANSPORT_DELAY",
    "high_latency":"RC_TRANSPORT_DELAY","jitter_degradation":"RC_TRANSPORT_DELAY",
    "high_jitter":"RC_TRANSPORT_DELAY","high_retransmission":"RC_SINR_DEGRADED",
    "low_throughput":"RC_CAPACITY_OVERLOAD","severe_packet_loss":"RC_TRANSPORT_DELAY",
    "congestion":"RC_PRB_CONGESTION","statistical_outlier":"RC_NONE",
}
FALLBACK_CONFIDENCE = 0.65

def _peak_tick(grp: pd.DataFrame) -> pd.Series:
    """Select the highest-severity decision tick inside one incident window."""
    return grp.sort_values(["anomaly_score", "timestamp"], ascending=[False, True]).iloc[0]

def build_incident_episodes(frame: pd.DataFrame, rng: np.random.Generator) -> tuple[list[Episode], int]:
    """Build one incident episode per matched incident using the peak anomaly tick."""
    eps, fallback_count = [], 0
    inc_rows = frame[frame["has_incident"] & frame["incident_id"].notna()].copy()
    for inc_id, grp in inc_rows.groupby("incident_id", sort=False):
        tick = _peak_tick(grp)
        diag = mock_diagnose(tick, rng)
        rc = diag.primary_root_cause
        conf = diag.confidence
        if rc == "RC_NONE":
            fallback = INCIDENT_TYPE_TO_RC.get(str(tick.get("incident_type", "")), "RC_NONE")
            if fallback != "RC_NONE":
                rc, conf = fallback, FALLBACK_CONFIDENCE
                fallback_count += 1
        eps.append(Episode(
            episode_id=str(inc_id), timestamp=tick["timestamp"],
            zone_id=str(tick["zone_id"]), primary_rc=rc,
            rc_confidence=conf,
            allowed_actions=list(RC_TO_ACTIONS.get(rc, ["ACT_NO_OP"])),
            state=_row_to_state(tick), severity=str(tick.get("severity", "")),
            source_file=str(tick["source_file"]), is_incident=True,
        ))
    return eps, fallback_count

def build_nil_episodes(frame: pd.DataFrame, n: int, rng: np.random.Generator) -> list[Episode]:
    """Sample no-incident, non-anomalous rows as matched no-problem episodes."""
    eligible = frame[(~frame["has_incident"]) & (frame["anomaly_flag"] == False)].copy()
    n = min(n, len(eligible))
    idx = rng.choice(eligible.index, size=n, replace=False)
    picked = eligible.loc[idx].sort_values("timestamp")
    eps = []
    for row_idx, row in picked.iterrows():
        eps.append(Episode(
            episode_id=f"nil_{row_idx}", timestamp=row["timestamp"],
            zone_id=str(row["zone_id"]), primary_rc=RC_NO_PROBLEM, rc_confidence=1.0,
            allowed_actions=["ACT_NO_OP"], state=_row_to_state(row),
            severity="", source_file=str(row["source_file"]), is_incident=False,
        ))
    return eps

def episode_frame(episodes: list[Episode]) -> pd.DataFrame:
    """Convert episodes into an audit table without expanding state vectors."""
    rows = []
    for e in episodes:
        rows.append({
            "episode_id": e.episode_id,
            "timestamp": e.timestamp,
            "episode_kind": "incident_peak" if e.is_incident else "matched_no_problem",
            "primary_rc": e.primary_rc,
            "rc_confidence": e.rc_confidence,
            "allowed_action_count": len(e.allowed_actions),
            "allowed_actions": ", ".join(e.allowed_actions),
            "zone_id": e.zone_id,
            "source_file": e.source_file,
            "missing_state_features": sum(e.state.get(k) is None for k in EPISODE_STATE_KEYS),
            "is_synthetic": e.is_synthetic,
        })
    return pd.DataFrame(rows)

def validate_episode_pool(episodes: list[Episode], source_frame: pd.DataFrame, fallback_count: int) -> pd.DataFrame:
    """Validate episode uniqueness, state completeness, action masks, and incident coverage."""
    ep_df = episode_frame(episodes)
    matched_incident_ids = set(source_frame.loc[
        source_frame["has_incident"] & source_frame["incident_id"].notna(), "incident_id"
    ].astype(str))
    incident_episode_ids = {e.episode_id for e in episodes if e.is_incident}
    action_codes = set(ACTION_BY_CODE)
    rows = [
        {"check": "episode_ids_unique", "passed": ep_df["episode_id"].is_unique,
         "detail": f"episodes={len(ep_df)}, unique_ids={ep_df['episode_id'].nunique()}"},
        {"check": "one_episode_per_matched_incident", "passed": incident_episode_ids == matched_incident_ids,
         "detail": f"incident_episodes={len(incident_episode_ids)}, matched_incidents={len(matched_incident_ids)}"},
        {"check": "nil_count_matches_incident_count", "passed": int((~ep_df['episode_kind'].eq('incident_peak')).sum()) == len(incident_episode_ids),
         "detail": f"nil={int((ep_df['episode_kind'] == 'matched_no_problem').sum())}, incident={len(incident_episode_ids)}"},
        {"check": "all_actions_exist_in_catalogue", "passed": all(set(e.allowed_actions).issubset(action_codes) for e in episodes),
         "detail": f"catalogue_actions={len(action_codes)}"},
        {"check": "actions_follow_rc_mask", "passed": all(set(e.allowed_actions) == set(RC_TO_ACTIONS.get(e.primary_rc, ['ACT_NO_OP'])) for e in episodes),
         "detail": f"rc_masks_checked={ep_df['primary_rc'].nunique()}"},
        {"check": "confidence_bounded", "passed": ep_df["rc_confidence"].between(0.0, 1.0).all(),
         "detail": f"min={ep_df['rc_confidence'].min():.3f}, max={ep_df['rc_confidence'].max():.3f}"},
        {"check": "state_features_available", "passed": set(EPISODE_STATE_KEYS).issubset(source_frame.columns),
         "detail": f"state_keys={len(EPISODE_STATE_KEYS)}, missing_columns={len(set(EPISODE_STATE_KEYS) - set(source_frame.columns))}"},
        {"check": "fallbacks_explicit", "passed": fallback_count >= 0,
         "detail": f"fallback_rc_assignments={fallback_count}"},
    ]
    return pd.DataFrame(rows)

ep_rng       = np.random.default_rng(RANDOM_STATE + 2)
diag_rng_eps = np.random.default_rng(RANDOM_STATE + 3)
incident_eps, n_fallback = build_incident_episodes(df, diag_rng_eps)
nil_eps = build_nil_episodes(df, n=len(incident_eps), rng=ep_rng)
all_eps = sorted(incident_eps + nil_eps, key=lambda e: e.timestamp)

episode_audit_df = episode_frame(all_eps)
episode_validation = validate_episode_pool(all_eps, df, n_fallback)
episode_rc_summary = (
    episode_audit_df.groupby(["episode_kind", "primary_rc"], dropna=False)
    .agg(episodes=("episode_id", "size"),
         mean_confidence=("rc_confidence", "mean"),
         mean_allowed_actions=("allowed_action_count", "mean"),
         mean_missing_state_features=("missing_state_features", "mean"))
    .reset_index()
)
episode_source_summary = (
    episode_audit_df.groupby("source_file", dropna=False)
    .agg(episodes=("episode_id", "size"),
         incidents=("episode_kind", lambda s: int((s == "incident_peak").sum())),
         nil=("episode_kind", lambda s: int((s == "matched_no_problem").sum())))
    .reset_index()
    .sort_values("episodes", ascending=False)
)

if not episode_validation["passed"].all():
    raise AssertionError(episode_validation.loc[~episode_validation["passed"]].to_string(index=False))

print("Episode pool audit:")
print(episode_validation.to_string(index=False))
print(f"\nEpisode pool: {len(all_eps)} ({len(incident_eps)} incident peaks + {len(nil_eps)} matched RC_NONE)")
print(f"Fallbacks applied: {n_fallback}")
print("\nEpisode RC summary:")
print(episode_rc_summary.round(4).to_string(index=False))
print("\nTop episode source files:")
print(episode_source_summary.head(10).to_string(index=False))

fig_episode_kind = px.histogram(
    episode_audit_df, x="episode_kind", color="primary_rc", barmode="stack",
    title="Constructed Episode Pool by Type and Root Cause",
    labels={"episode_kind": "Episode type", "count": "Episodes", "primary_rc": "Root cause"},
)
fig_episode_kind.update_layout(width=900, height=430)
save_fig(fig_episode_kind, "fig03a_episode_pool_composition"); fig_episode_kind.show()

fig_episode_conf = px.box(
    episode_audit_df, x="primary_rc", y="rc_confidence", color="episode_kind",
    title="Diagnosis Confidence Across Constructed Episodes",
    labels={"primary_rc": "Root cause", "rc_confidence": "Diagnosis confidence", "episode_kind": "Episode type"},
)
fig_episode_conf.update_xaxes(tickangle=-35)
fig_episode_conf.update_layout(width=1050, height=460)
save_fig(fig_episode_conf, "fig03b_episode_confidence_by_rc"); fig_episode_conf.show()

fig_episode_actions = px.bar(
    episode_rc_summary, x="primary_rc", y="mean_allowed_actions", color="episode_kind",
    text="episodes", barmode="group",
    title="Allowed Action Breadth by Episode Root Cause",
    labels={"primary_rc": "Root cause", "mean_allowed_actions": "Mean allowed actions", "episode_kind": "Episode type"},
)
fig_episode_actions.update_xaxes(tickangle=-35)
fig_episode_actions.update_layout(width=1050, height=430)
save_fig(fig_episode_actions, "fig03c_episode_allowed_action_breadth"); fig_episode_actions.show()


Episode pool audit:
                           check  passed                                       detail
              episode_ids_unique    True                 episodes=548, unique_ids=548
one_episode_per_matched_incident    True incident_episodes=274, matched_incidents=274
nil_count_matches_incident_count    True                        nil=274, incident=274
  all_actions_exist_in_catalogue    True                         catalogue_actions=15
          actions_follow_rc_mask    True                           rc_masks_checked=7
              confidence_bounded    True                         min=0.209, max=1.000
        state_features_available    True             state_keys=35, missing_columns=0
              fallbacks_explicit    True                   fallback_rc_assignments=36

Episode pool: 548 (274 incident peaks + 274 matched RC_NONE)
Fallbacks applied: 36

Episode RC summary:
      episode_kind           primary_rc  episodes  mean_confidence  mean_allowed_actions  mean_missin

**Interpretation:** We convert the cleaned row-level data into contextual-bandit episodes by taking one peak decision tick per matched incident and then sampling the same number of clean no-problem episodes. The audit passes every integrity check: there are 548 unique episodes, made of 274 incident peaks and 274 matched `RC_NONE` examples, and the 274 incident episodes match the 274 incident IDs that survived interval matching. The action catalogue check passes for all 15 actions, and the RC-mask check passes across the 7 root causes present before balancing, so no bandit can receive an action that falls outside the diagnosed operational scope.

The episode-pool composition figure shows that the real incident pool is highly concentrated. `RC_TRANSPORT_DELAY` contributes 238 incident episodes, while `RC_SINR_DEGRADED` has 13, `RC_CAPACITY_OVERLOAD` has 9, `RC_COVERAGE_HOLE` has 9, `RC_WEAK_SIGNAL` has 4, and `RC_PRB_CONGESTION` has only 1. The matched no-problem class contributes 274 `RC_NONE` episodes. We applied 36 explicit fallback assignments when the mock diagnostic returned `RC_NONE` on an incident row but the incident type supported a more specific root cause. We keep those fallbacks visible because they are part of the diagnostic contract, not hidden relabeling.

The confidence and action-breadth figures explain why this construction is appropriate for bandits. `RC_NONE` has confidence 1.0 and only one allowed action, so no-problem episodes act as a stable `ACT_NO_OP` anchor. Actionable incident classes have wider action masks, usually 3 or 4 allowed actions, and lower confidence; for example, `RC_TRANSPORT_DELAY` has mean confidence 0.4247 and 4 allowed actions. That creates the real exploration problem. The state audit also passes with all 35 configured state keys available and 0 missing source columns, so the bandits receive a consistent decision context rather than an ad hoc feature mix.

In [11]:
MIN_SYNTH_PER_RC = 20
RC_SYNTH_OVERRIDES = {
    "RC_WEAK_SIGNAL":       {"rssi_dbm":-92.0,"rsrp_dbm":-97.0,"sinr_db":8.0,
                             "bler_proxy_pct":2.0,"signal_dominant_link":"wifi",
                             "wifi_signal_score":35.0,"cellular_signal_score":65.0},
    "RC_HO_FAILURE":        {"ho_success_rate_pct":35.0,"rssi_dbm":-88.0,"sinr_db":5.0,
                             "neighbor_count":2.0,"signal_dominant_link":"cellular"},
    "RC_PRB_CONGESTION":    {"bandwidth_util_pct":90.0,"channel_util_pct":85.0,
                             "active_connections":14.0,"is_peak_hour":True,
                             "latency_ms":220.0,"queue_length":60.0},
    "RC_CAPACITY_OVERLOAD": {"bandwidth_util_pct":95.0,"channel_util_pct":92.0,
                             "active_connections":18.0,"is_peak_hour":True,
                             "latency_ms":350.0,"queue_length":80.0},
    "RC_SINR_DEGRADED":     {"sinr_db":1.5,"rssi_dbm":-89.0,"rsrp_dbm":-103.0,
                             "bler_proxy_pct":12.0,"mcs":4.0,"cqi":3.0},
    "RC_CQI_MISMATCH":      {"cqi":2.0,"mcs":2.0,"throughput_mbps":1.2,
                             "sinr_db":4.0,"bler_proxy_pct":15.0},
}

def augment_pool(pool: list[Episode], overrides: dict, min_count: int,
                 rng: np.random.Generator, split_name: str) -> list[Episode]:
    """Add bounded synthetic rare-RC episodes to a train or validation pool only."""
    rc_existing = {}
    for e in pool:
        rc_existing.setdefault(e.primary_rc, []).append(e)
    donor_pool = [e for e in pool if e.primary_rc != RC_NO_PROBLEM] or pool
    synth = []
    for rc, kpi_patch in overrides.items():
        n_have = len(rc_existing.get(rc, []))
        n_need = max(0, min_count - n_have)
        for k in range(n_need):
            donor = donor_pool[int(rng.integers(0, len(donor_pool)))]
            new_state = dict(donor.state)
            for key, val in kpi_patch.items():
                if isinstance(val, (bool, str)):
                    new_state[key] = val
                else:
                    jitter = float(rng.normal(0.0, max(abs(float(val)) * 0.05, 1e-6)))
                    new_state[key] = float(val) + jitter
            conf_mu = {"RC_WEAK_SIGNAL":0.82,"RC_HO_FAILURE":0.78,"RC_PRB_CONGESTION":0.80}.get(rc, 0.75)
            kappa = 25.0
            conf = float(np.clip(rng.beta(max(1.0, conf_mu*kappa), max(1.0, (1-conf_mu)*kappa)), 0.05, 0.99))
            synth.append(Episode(
                episode_id=f"syn_{split_name}_{rc}_{k}",
                timestamp=donor.timestamp,
                zone_id=donor.zone_id, primary_rc=rc, rc_confidence=conf,
                allowed_actions=list(RC_TO_ACTIONS[rc]),
                state=new_state, severity=donor.severity or "medium",
                source_file=donor.source_file, is_incident=True, is_synthetic=True,
            ))
    if synth:
        n_by_rc = pd.Series([e.primary_rc for e in synth]).value_counts().to_dict()
        print(f"Synthetic augmentation for {split_name}: +{len(synth)} episodes -> {n_by_rc}")
    return sorted(pool + synth, key=lambda e: e.timestamp)

**Interpretation:** We use synthetic episodes only as a bounded balancing aid after the real chronological split has already been made. The synthetic generator does not create a new holdout and it does not alter the real test set. Instead, it patches realistic KPI signatures onto donor training or validation episodes for rare RCs that would otherwise have too few examples for the bandits to learn an action preference. This makes the training signal less brittle while keeping the final benchmark conservative.

In [12]:
def chronological_split(episodes, train_frac=0.60, val_frac=0.20):
    """Split real episodes in timestamp order to mimic deployment on future data."""
    order = sorted(range(len(episodes)), key=lambda i: episodes[i].timestamp)
    n = len(order); n_tr = int(np.floor(train_frac*n)); n_va = int(np.floor(val_frac*n))
    return order[:n_tr], order[n_tr:n_tr+n_va], order[n_tr+n_va:]

real_all_eps = sorted(all_eps, key=lambda e: e.timestamp)
real_train_idx, real_val_idx, real_test_idx = chronological_split(real_all_eps)
real_train_eps = [real_all_eps[i] for i in real_train_idx]
real_val_eps   = [real_all_eps[i] for i in real_val_idx]
test_eps       = [real_all_eps[i] for i in real_test_idx]

split_protocol_audit = pd.DataFrame([
    {"candidate_protocol": "random_row_split", "implemented": False,
     "reason_rejected": "Rows and incidents are time-dependent; random rows leak future operating regimes into training."},
    {"candidate_protocol": "group_kfold_by_incident", "implemented": False,
     "reason_rejected": "Useful for offline diagnosis models, but it breaks the deployment chronology required by an optimizer trained on the past."},
    {"candidate_protocol": "node_or_zone_holdout", "implemented": False,
     "reason_rejected": "Measures site transfer, not the temporal deployment question of this notebook; rare RC coverage is too sparse for a stable holdout."},
    {"candidate_protocol": "chronological_train_val_test", "implemented": True,
     "reason_rejected": "Selected: matches deployment, keeps test in the future, and prevents synthetic examples from entering the holdout."},
])

pre_balance_rows = []
for split, eps_list in [("real_train_pre_balance", real_train_eps), ("real_val_pre_balance", real_val_eps), ("real_test_holdout", test_eps)]:
    for e in eps_list:
        pre_balance_rows.append({"split": split, "primary_rc": e.primary_rc, "is_synthetic": e.is_synthetic, "timestamp": e.timestamp})
pre_balance_df = pd.DataFrame(pre_balance_rows)

aug_rng = np.random.default_rng(RANDOM_STATE + 42)
train_eps = augment_pool(real_train_eps, RC_SYNTH_OVERRIDES, MIN_SYNTH_PER_RC, aug_rng, "train")
val_eps   = augment_pool(real_val_eps,   RC_SYNTH_OVERRIDES, MIN_SYNTH_PER_RC, aug_rng, "val")

all_eps = train_eps + val_eps + test_eps
train_idx = list(range(0, len(train_eps)))
val_idx   = list(range(len(train_eps), len(train_eps) + len(val_eps)))
test_idx  = list(range(len(train_eps) + len(val_eps), len(all_eps)))

ids_tr = {e.episode_id for e in train_eps}
ids_va = {e.episode_id for e in val_eps}
ids_te = {e.episode_id for e in test_eps}
assert not (ids_tr & ids_va), "train/val overlap"
assert not (ids_tr & ids_te), "train/test overlap"
assert not (ids_va & ids_te), "val/test overlap"
assert max(e.timestamp for e in train_eps) <= min(e.timestamp for e in val_eps)
assert max(e.timestamp for e in val_eps) <= min(e.timestamp for e in test_eps)
assert not any(e.is_synthetic for e in test_eps), "test set must remain real-only"

post_split_rows = []
for split, eps_list in [("train", train_eps), ("val", val_eps), ("test", test_eps)]:
    for e in eps_list:
        post_split_rows.append({"split": split, "primary_rc": e.primary_rc, "is_synthetic": e.is_synthetic, "timestamp": e.timestamp})
split_df = pd.DataFrame(post_split_rows)

print("Split protocol audit:")
print(split_protocol_audit.to_string(index=False))
print(f"\nSplit: train={len(train_eps)}  val={len(val_eps)}  test={len(test_eps)} real-only")
print("\nSynthetic by split after balancing:")
print(pd.crosstab(split_df["split"], split_df["is_synthetic"]))
print("\nRC by split after balancing:")
rc_split_counts = pd.crosstab(split_df["primary_rc"], split_df["split"])
print(rc_split_counts)

split_window_summary = (
    split_df.groupby("split", dropna=False)
    .agg(episodes=("primary_rc", "size"),
         real_episodes=("is_synthetic", lambda s: int((~s).sum())),
         synthetic_episodes=("is_synthetic", lambda s: int(s.sum())),
         start=("timestamp", "min"),
         end=("timestamp", "max"),
         root_causes=("primary_rc", "nunique"))
    .reindex(["train", "val", "test"])
    .reset_index()
)
split_window_summary["duration_hours"] = (
    (split_window_summary["end"] - split_window_summary["start"]).dt.total_seconds() / 3600.0
).round(2)
split_leakage_audit = pd.DataFrame([
    {"check": "no_train_val_id_overlap", "passed": len(ids_tr & ids_va) == 0, "detail": f"overlap={len(ids_tr & ids_va)}"},
    {"check": "no_train_test_id_overlap", "passed": len(ids_tr & ids_te) == 0, "detail": f"overlap={len(ids_tr & ids_te)}"},
    {"check": "no_val_test_id_overlap", "passed": len(ids_va & ids_te) == 0, "detail": f"overlap={len(ids_va & ids_te)}"},
    {"check": "train_before_val", "passed": max(e.timestamp for e in train_eps) <= min(e.timestamp for e in val_eps),
     "detail": f"train_end={max(e.timestamp for e in train_eps)}, val_start={min(e.timestamp for e in val_eps)}"},
    {"check": "val_before_test", "passed": max(e.timestamp for e in val_eps) <= min(e.timestamp for e in test_eps),
     "detail": f"val_end={max(e.timestamp for e in val_eps)}, test_start={min(e.timestamp for e in test_eps)}"},
    {"check": "test_real_only", "passed": not any(e.is_synthetic for e in test_eps),
     "detail": f"test_synthetic={sum(e.is_synthetic for e in test_eps)}"},
])
if not split_leakage_audit["passed"].all():
    raise AssertionError(split_leakage_audit.loc[~split_leakage_audit["passed"]].to_string(index=False))

print("\nSplit leakage audit:")
print(split_leakage_audit.to_string(index=False))
print("\nSplit window summary:")
print(split_window_summary.to_string(index=False))

fig_pre = px.histogram(pre_balance_df, x="primary_rc", color="split", barmode="group",
                       title="RC Distribution Before Train/Val Balancing",
                       labels={"primary_rc": "Root cause", "count": "Episodes"})
fig_pre.update_xaxes(tickangle=-35)
fig_pre.update_layout(width=1050, height=430)
save_fig(fig_pre, "fig03d_rc_distribution_before_balancing"); fig_pre.show()

fig_post = px.histogram(split_df, x="primary_rc", color="split", barmode="group",
                        pattern_shape="is_synthetic",
                        title="RC Distribution After Train/Val Balancing and Real-Only Test",
                        labels={"primary_rc": "Root cause", "count": "Episodes"})
fig_post.update_xaxes(tickangle=-35)
fig_post.update_layout(width=1050, height=430)
save_fig(fig_post, "fig03e_rc_distribution_after_balancing"); fig_post.show()

heatmap_counts = rc_split_counts.reindex(index=RC_VOCAB + [RC_NO_PROBLEM], columns=["train", "val", "test"]).fillna(0).astype(int)
fig_rc_heatmap = px.imshow(
    heatmap_counts.values,
    x=list(heatmap_counts.columns), y=list(heatmap_counts.index),
    text_auto=True, aspect="auto", color_continuous_scale="Viridis",
    title="Root-Cause Counts by Chronological Split",
    labels={"x": "Split", "y": "Root cause", "color": "Episodes"},
)
fig_rc_heatmap.update_layout(width=760, height=520)
save_fig(fig_rc_heatmap, "fig03f_rc_split_heatmap"); fig_rc_heatmap.show()

fig_split_stack = px.bar(
    split_window_summary, x="split", y=["real_episodes", "synthetic_episodes"],
    title="Real vs Synthetic Episodes by Split",
    labels={"value": "Episodes", "split": "Split", "variable": "Episode source"},
)
fig_split_stack.update_layout(width=760, height=420)
save_fig(fig_split_stack, "fig03g_real_vs_synthetic_by_split"); fig_split_stack.show()

fig_timeline = px.scatter(split_df, x="timestamp", y="primary_rc", color="split", symbol="is_synthetic",
                          color_discrete_map={"train":"#1f77b4","val":"#ff7f0e","test":"#2ca02c"},
                          title="Episode Timeline - Chronological Train / Val / Real Holdout Test")
fig_timeline.update_layout(width=1100, height=460)
fig_timeline.update_traces(marker=dict(size=7, opacity=0.75))
save_fig(fig_timeline, "fig03h_episode_timeline"); fig_timeline.show()

Synthetic augmentation for train: +109 episodes -> {'RC_WEAK_SIGNAL': 20, 'RC_HO_FAILURE': 20, 'RC_PRB_CONGESTION': 20, 'RC_CQI_MISMATCH': 20, 'RC_CAPACITY_OVERLOAD': 16, 'RC_SINR_DEGRADED': 13}
Synthetic augmentation for val: +116 episodes -> {'RC_WEAK_SIGNAL': 20, 'RC_HO_FAILURE': 20, 'RC_CQI_MISMATCH': 20, 'RC_PRB_CONGESTION': 19, 'RC_CAPACITY_OVERLOAD': 19, 'RC_SINR_DEGRADED': 18}
Split protocol audit:
          candidate_protocol  implemented                                                                                                                     reason_rejected
            random_row_split        False                                     Rows and incidents are time-dependent; random rows leak future operating regimes into training.
     group_kfold_by_incident        False          Useful for offline diagnosis models, but it breaks the deployment chronology required by an optimizer trained on the past.
        node_or_zone_holdout        False Measures site transfer, no

**Interpretation:** The split protocol is explicit and leak-checked. We reject random row splitting because rows and incidents are time-dependent, reject incident GroupKFold because it answers an offline generalization question rather than deployment, and reject a node/zone holdout because rare RC coverage is too sparse here. The implemented split trains on the past, validates on the middle window, and tests on future real episodes only. The leakage audit confirms 0 train/validation overlap, 0 train/test overlap, 0 validation/test overlap, and 0 synthetic episodes in the test set.

The split-window table gives the actual deployment chronology. Training contains 437 episodes from 2026-03-27 18:22:08+01:00 to 2026-03-30 00:38:24+01:00. Validation contains 225 episodes from 2026-03-30 00:39:33+01:00 to 2026-03-30 15:35:36+01:00. The real holdout test starts later, at 2026-03-30 15:40:38+01:00, and runs until 2026-04-05 03:21:06+01:00. This ordering matters because the benchmark evaluates how a policy trained on earlier operating conditions behaves on future episodes.

The pre-balance RC figure shows why balancing is needed: several rare operational RCs are absent or nearly absent from train and validation. After balancing, train has 328 real and 109 synthetic episodes, validation has 109 real and 116 synthetic episodes, and test stays fully real with 111 real and 0 synthetic episodes. The RC split heatmap shows what the benchmark can and cannot claim. The test set is dominated by `RC_TRANSPORT_DELAY` with 56 episodes and `RC_NONE` with 34 episodes, while `RC_CAPACITY_OVERLOAD`, `RC_SINR_DEGRADED`, and `RC_WEAK_SIGNAL` have only 4 real test examples each. `RC_CQI_MISMATCH`, `RC_HO_FAILURE`, and `RC_PRB_CONGESTION` have no real test examples in this run, so their balanced train/validation examples help policies learn safe action preferences but do not support separate real-holdout claims for those classes. The timeline figure confirms the final rule we care about most: synthetic points stay before the real-only future test window.

<a id="sec-10"></a>

## 10 - Feature Encoder (single source of truth)

After the split is fixed, we define the shared state vector used by every contextual policy. Numeric scaling is fitted only on training episodes, categorical diagnostic fields are encoded explicitly, and the final vector dimension is audited so model comparisons use the same information boundary.

In [13]:
# Column lists for the single canonical encoder.
LIN_NUM_COLS = [
    "latency_ms","jitter_ms","packet_loss_pct","throughput_mbps",
    "rssi_dbm","rsrp_dbm","sinr_db","cqi","mcs",
    "bler_proxy_pct","ho_success_rate_pct","neighbor_count",
    "channel_util_pct","queue_length","tcp_retransmit_rate",
    "bandwidth_util_pct","active_connections",
    "wifi_signal_score","cellular_signal_score","hour_of_day",
    "hour_sin","hour_cos","day_sin","day_cos",
    "kpi_impairment_score","transport_pressure_index",
    "radio_impairment_score","link_signal_delta",
    "missing_critical_kpi_count",
]
LIN_TRAFFIC_VOCAB = ["browsing","streaming","file_transfer","video_call",
                     "gaming","congestion","normal","packet_loss"]
LIN_LINK_VOCAB    = ["cellular","wifi"]

def compute_train_stats(episodes: list[Episode]) -> dict:
    """Compute numeric encoder means and standard deviations using training episodes only."""
    arr = np.array([[e.state.get(c) if e.state.get(c) is not None else np.nan
                     for c in LIN_NUM_COLS] for e in episodes], dtype=np.float64)
    stats = {}
    for i, c in enumerate(LIN_NUM_COLS):
        col = arr[:, i]
        col = col[~np.isnan(col)]
        if col.size == 0:
            stats[c] = (0.0, 1.0)
        else:
            sd = float(col.std(ddof=1)) if col.size > 1 else 1.0
            stats[c] = (float(col.mean()), sd if sd > 1e-8 else 1.0)
    return stats

def encoder_feature_names() -> list[str]:
    """Return feature names in the exact order emitted by `_encode_state`."""
    return (
        [f"num::{c}" for c in LIN_NUM_COLS]
        + [f"rc::{rc}" for rc in RC_VOCAB]
        + [f"traffic::{t}" for t in LIN_TRAFFIC_VOCAB]
        + [f"link::{l}" for l in LIN_LINK_VOCAB]
        + ["flag::is_peak_hour", "bias::constant"]
    )

def _encode_state(state: dict, stats: dict | None = None) -> np.ndarray:
    """Project `Episode.state` into the canonical fixed-length feature vector."""
    num = []
    for c in LIN_NUM_COLS:
        v = state.get(c)
        v = 0.0 if v is None or (isinstance(v, float) and np.isnan(v)) else float(v)
        if stats is not None and c in stats:
            mu, sd = stats[c]
            v = (v - mu) / (sd if sd > 1e-8 else 1.0)
        num.append(v)
    rc_oh = [1.0 if state.get("__rc__") == rc else 0.0 for rc in RC_VOCAB]
    tt_oh = [1.0 if state.get("traffic_type") == t else 0.0 for t in LIN_TRAFFIC_VOCAB]
    lk = str(state.get("signal_dominant_link") or "").lower()
    lk_oh = [1.0 if lk.startswith(l[:3]) else 0.0 for l in LIN_LINK_VOCAB]
    peak = [1.0 if state.get("is_peak_hour") else 0.0, 1.0]
    return np.array(num + rc_oh + tt_oh + lk_oh + peak, dtype=np.float64)

def encode_episode_matrix(episodes: list[Episode], stats: dict) -> pd.DataFrame:
    """Encode a list of episodes into a dataframe with named feature columns."""
    rows = []
    names = encoder_feature_names()
    for e in episodes:
        x = _encode_state({**e.state, "__rc__": e.primary_rc}, stats)
        rows.append(dict(zip(names, x)))
    return pd.DataFrame(rows)

def numeric_missingness_frame(split_to_episodes: dict[str, list[Episode]]) -> pd.DataFrame:
    """Calculate numeric input missingness for each split before imputation/scaling."""
    rows = []
    for split, episodes in split_to_episodes.items():
        for col in LIN_NUM_COLS:
            values = [e.state.get(col) for e in episodes]
            missing = sum(v is None or (isinstance(v, float) and np.isnan(v)) for v in values)
            rows.append({"split": split, "feature": col, "missing": missing,
                         "episodes": len(episodes), "missing_pct": missing / max(1, len(episodes))})
    return pd.DataFrame(rows)

def validate_encoder_contract(train_stats: dict, sample_vector: np.ndarray) -> pd.DataFrame:
    """Validate feature dimensions, train-stat coverage, and numeric finiteness."""
    names = encoder_feature_names()
    rows = [
        {"check": "dimension_matches_feature_names", "passed": len(sample_vector) == len(names),
         "detail": f"vector={len(sample_vector)}, names={len(names)}"},
        {"check": "expected_dimension_formula", "passed": len(sample_vector) == EXPECTED_D,
         "detail": f"D={len(sample_vector)}, expected={EXPECTED_D}"},
        {"check": "all_numeric_columns_have_train_stats", "passed": set(LIN_NUM_COLS) == set(train_stats),
         "detail": f"numeric_cols={len(LIN_NUM_COLS)}, stats={len(train_stats)}"},
        {"check": "train_stats_finite", "passed": all(np.isfinite(mu) and np.isfinite(sd) and sd > 0 for mu, sd in train_stats.values()),
         "detail": f"stats_checked={len(train_stats)}"},
        {"check": "sample_vector_finite", "passed": bool(np.isfinite(sample_vector).all()),
         "detail": f"finite_values={int(np.isfinite(sample_vector).sum())}/{len(sample_vector)}"},
    ]
    return pd.DataFrame(rows)

# Compute once on train set. Validation and test are transformed with these train-only statistics.
train_stats = compute_train_stats(train_eps)
_sample_state = {**train_eps[0].state, "__rc__": train_eps[0].primary_rc}
EXPECTED_D = len(LIN_NUM_COLS) + len(RC_VOCAB) + len(LIN_TRAFFIC_VOCAB) + len(LIN_LINK_VOCAB) + 2
_sample_vector = _encode_state(_sample_state, train_stats)
D = len(_sample_vector)
encoder_validation = validate_encoder_contract(train_stats, _sample_vector)
if not encoder_validation["passed"].all():
    raise AssertionError(encoder_validation.loc[~encoder_validation["passed"]].to_string(index=False))

split_to_eps = {"train": train_eps, "val": val_eps, "test": test_eps}
encoded_by_split = {split: encode_episode_matrix(eps, train_stats) for split, eps in split_to_eps.items()}
encoded_scale_rows = []
for split, matrix in encoded_by_split.items():
    numeric_matrix = matrix[[f"num::{c}" for c in LIN_NUM_COLS]]
    encoded_scale_rows.append({
        "split": split,
        "features": numeric_matrix.shape[1],
        "episodes": numeric_matrix.shape[0],
        "mean_abs_encoded_numeric_mean": float(numeric_matrix.mean().abs().mean()),
        "median_encoded_numeric_std": float(numeric_matrix.std(ddof=1).median()),
        "max_abs_encoded_value": float(np.nanmax(np.abs(numeric_matrix.to_numpy(dtype=float)))),
    })
encoded_scale_summary = pd.DataFrame(encoded_scale_rows)

encoder_component_df = pd.DataFrame([
    {"component": "numeric_kpis_and_engineered", "dimensions": len(LIN_NUM_COLS)},
    {"component": "root_cause_one_hot", "dimensions": len(RC_VOCAB)},
    {"component": "traffic_type_one_hot", "dimensions": len(LIN_TRAFFIC_VOCAB)},
    {"component": "link_one_hot", "dimensions": len(LIN_LINK_VOCAB)},
    {"component": "peak_flag_and_bias", "dimensions": 2},
])
train_stats_df = pd.DataFrame([
    {"feature": col, "train_mean": train_stats[col][0], "train_std": train_stats[col][1]}
    for col in LIN_NUM_COLS
]).sort_values("train_std", ascending=False)
missingness_df = numeric_missingness_frame(split_to_eps)
missingness_summary = (
    missingness_df.groupby("split", dropna=False)
    .agg(total_missing=("missing", "sum"),
         mean_missing_pct=("missing_pct", "mean"),
         max_feature_missing_pct=("missing_pct", "max"))
    .reindex(["train", "val", "test"])
    .reset_index()
)

_TRAINVAL = [all_eps[i] for i in train_idx] + [all_eps[i] for i in val_idx]

print("Encoder validation:")
print(encoder_validation.to_string(index=False))
print(f"\nFeature dimension D = {D}  (expected {EXPECTED_D})")
print(f"_TRAINVAL size: {len(_TRAINVAL)} episodes")
print("\nEncoder component dimensions:")
print(encoder_component_df.to_string(index=False))
print("\nNumeric missingness by split:")
print(missingness_summary.round(4).to_string(index=False))
print("\nEncoded numeric scale by split:")
print(encoded_scale_summary.round(4).to_string(index=False))
print("\nTop train-scaling standard deviations:")
print(train_stats_df.head(10).round(4).to_string(index=False))

fig_encoder_components = px.bar(
    encoder_component_df, x="component", y="dimensions", text="dimensions",
    title="Encoder Component Dimensions",
    labels={"component": "Encoder component", "dimensions": "Dimensions"},
)
fig_encoder_components.update_xaxes(tickangle=-25)
fig_encoder_components.update_layout(width=850, height=430)
save_fig(fig_encoder_components, "fig03i_encoder_component_dimensions"); fig_encoder_components.show()

missing_heat = missingness_df.pivot(index="feature", columns="split", values="missing_pct").reindex(columns=["train", "val", "test"])
fig_missing = px.imshow(
    missing_heat.values,
    x=list(missing_heat.columns), y=list(missing_heat.index),
    text_auto=".2f", aspect="auto", color_continuous_scale="Reds",
    title="Numeric Encoder Input Missingness by Split",
    labels={"x": "Split", "y": "Numeric feature", "color": "Missing fraction"},
)
fig_missing.update_layout(width=780, height=720)
save_fig(fig_missing, "fig03j_encoder_numeric_missingness"); fig_missing.show()

scale_long = encoded_scale_summary.melt(
    id_vars=["split", "episodes", "features"],
    value_vars=["mean_abs_encoded_numeric_mean", "median_encoded_numeric_std", "max_abs_encoded_value"],
    var_name="scale_metric", value_name="value",
)
fig_scale = px.bar(
    scale_long, x="split", y="value", color="scale_metric", barmode="group",
    title="Encoded Numeric Scale by Split Using Train-Only Statistics",
    labels={"split": "Split", "value": "Metric value", "scale_metric": "Scale metric"},
)
fig_scale.update_layout(width=900, height=430)
save_fig(fig_scale, "fig03k_encoder_encoded_scale_by_split"); fig_scale.show()

fig_stats = px.bar(
    train_stats_df.head(12).sort_values("train_std"),
    x="train_std", y="feature", orientation="h", text="train_std",
    title="Largest Train-Only Scaling Standard Deviations",
    labels={"train_std": "Training standard deviation", "feature": "Numeric feature"},
)
fig_stats.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig_stats.update_layout(width=850, height=520)
save_fig(fig_stats, "fig03l_encoder_train_scaling_std"); fig_stats.show()


Encoder validation:
                               check  passed                    detail
     dimension_matches_feature_names    True       vector=49, names=49
          expected_dimension_formula    True         D=49, expected=49
all_numeric_columns_have_train_stats    True numeric_cols=29, stats=29
                  train_stats_finite    True          stats_checked=29
                sample_vector_finite    True       finite_values=49/49

Feature dimension D = 49  (expected 49)
_TRAINVAL size: 662 episodes

Encoder component dimensions:
                  component  dimensions
numeric_kpis_and_engineered          29
         root_cause_one_hot           8
       traffic_type_one_hot           8
               link_one_hot           2
         peak_flag_and_bias           2

Numeric missingness by split:
split  total_missing  mean_missing_pct  max_feature_missing_pct
train              7            0.0006                   0.0137
  val              0            0.0000                

**Interpretation:** We treat the encoder as a single source of truth because every contextual policy uses the same vector. The validation table passes all checks: the vector has 49 values, the feature-name list also has 49 entries, all 29 numeric inputs have train-only mean/std statistics, the statistics are finite, and the sample vector has 49 finite values. The component figure shows where the 49 dimensions come from: 29 numeric KPI/engineered inputs, 8 root-cause one-hot features, 8 traffic-type one-hot features, 2 link features, and 2 final flag/bias terms. This removes the earlier ambiguity around feature dimension and makes the model comparison fair.

The missingness heatmap checks the raw numeric inputs before imputation. Missingness is very low: the train split has 7 missing numeric values across 437 episodes and 29 numeric features, while validation and test both have 0. The highest feature-level missing fraction in train is only 0.0137, so the encoder is not hiding a data-quality problem behind zero imputation. The encoded-scale figure then shows the effect of using train-only scaling. Train has a mean absolute encoded numeric mean of 0.0005 and a median encoded numeric standard deviation of 1.0000 by construction. Validation drifts to a mean absolute encoded numeric mean of 0.4506, and test drifts further to 0.9475 because we do not refit statistics on future data. We interpret that drift as a useful deployment signal, not a bug.

The train-scaling standard-deviation figure identifies the features that dominate raw scale before normalization. `latency_ms` has the largest training standard deviation at 423.8711, followed by `queue_length` at 212.4678, `jitter_ms` at 71.7702, and `ho_success_rate_pct` at 48.1672. Those values explain why a shared normalizer is necessary for LinUCB: otherwise high-scale KPIs would dominate the linear score. After this audit, we use `_encode_state` everywhere: baselines, M6, M7, M8, reward-mode checks, and robustness tests all consume the same context contract.

<a id="sec-11"></a>

## 11 - Reference Baselines

Before introducing learned policies, we set the benchmark anchors on the real chronological holdout. The oracle ceiling, seeded random-in-mask baseline, and non-oracle rule lookup policy define the reward scale that every later model must be compared against.

In [14]:
def evaluate_policy_episodes(policy, episodes: list[Episode], run_id: int = 0) -> dict:
    """Evaluate one policy on episodes and return aggregate plus per-episode decisions."""
    rewards, regrets, chosen, rows = [], [], [], []
    invalid_actions = 0
    for ep in episodes:
        action = policy.select(ep)
        if action not in ep.allowed_actions:
            invalid_actions += 1
            action = ep.allowed_actions[0]
        reward, _ = oracle_reward(ep.primary_rc, action, ep.state)
        best_reward = max(oracle_reward(ep.primary_rc, candidate, ep.state)[0] for candidate in ep.allowed_actions)
        regret = best_reward - reward
        rewards.append(reward)
        regrets.append(regret)
        chosen.append(action)
        rows.append({
            "policy": policy.name,
            "run_id": run_id,
            "episode_id": ep.episode_id,
            "primary_rc": ep.primary_rc,
            "chosen_action": action,
            "allowed_action_count": len(ep.allowed_actions),
            "reward": reward,
            "best_reward": best_reward,
            "regret": regret,
            "is_synthetic": ep.is_synthetic,
        })
        if hasattr(policy, "update"):
            policy.update(ep, action, reward)
    return {
        "mean_reward": float(np.mean(rewards)) if rewards else float("nan"),
        "std_reward": float(np.std(rewards, ddof=1)) if len(rewards) > 1 else 0.0,
        "mean_regret": float(np.mean(regrets)) if regrets else float("nan"),
        "std_regret": float(np.std(regrets, ddof=1)) if len(regrets) > 1 else 0.0,
        "action_counts": dict(pd.Series(chosen).value_counts()),
        "invalid_actions": invalid_actions,
        "_rewards": rewards,
        "_decisions": pd.DataFrame(rows),
    }

class OraclePolicy:
    """Upper-bound baseline that selects the highest reward allowed action."""
    name = "oracle"
    uses_oracle_selection = True
    stochastic = False

    def select(self, ep: Episode) -> str:
        """Select the allowed action with maximum intent-oracle reward."""
        return max(ep.allowed_actions, key=lambda action: oracle_reward(ep.primary_rc, action, ep.state)[0])

    def update(self, ep: Episode, action: str, reward: float) -> None:
        """Keep the policy interface compatible with online evaluators."""
        return None

class RandomScopePolicy:
    """Lower-anchor baseline that samples uniformly from the allowed action mask."""
    name = "random_scope"
    uses_oracle_selection = False
    stochastic = True

    def __init__(self, seed: int = RANDOM_STATE):
        """Create a seeded random allowed-action selector."""
        self._rng = np.random.default_rng(seed)

    def select(self, ep: Episode) -> str:
        """Select a random action from the episode's allowed action mask."""
        return self._rng.choice(np.array(ep.allowed_actions)).item()

    def update(self, ep: Episode, action: str, reward: float) -> None:
        """Keep the policy interface compatible with online evaluators."""
        return None

class RuleLookupPolicy:
    """Structured deterministic baseline that selects the first action for each RC mask."""
    name = "rule_lookup"
    uses_oracle_selection = False
    stochastic = False

    def select(self, ep: Episode) -> str:
        """Select the canonical first action in the diagnosed RC action list."""
        return ep.allowed_actions[0]

    def update(self, ep: Episode, action: str, reward: float) -> None:
        """Keep the policy interface compatible with online evaluators."""
        return None

def evaluate_baseline_family(policy_name: str, factory: Callable[[int], object],
                             episodes: list[Episode], n_runs: int) -> tuple[dict, pd.DataFrame]:
    """Evaluate a deterministic or stochastic baseline family over one or more runs."""
    run_rows, decision_frames = [], []
    for run_id in range(n_runs):
        policy = factory(run_id)
        result = evaluate_policy_episodes(policy, episodes, run_id=run_id)
        run_rows.append({
            "policy": policy_name,
            "run_id": run_id,
            "mean_reward": result["mean_reward"],
            "mean_regret": result["mean_regret"],
            "invalid_actions": result["invalid_actions"],
        })
        decision_frames.append(result["_decisions"])
    run_df = pd.DataFrame(run_rows)
    decisions = pd.concat(decision_frames, ignore_index=True)
    action_share = decisions["chosen_action"].value_counts(normalize=True)
    entropy = float(-(action_share * np.log2(action_share + 1e-12)).sum())
    summary = {
        "policy": policy_name,
        "n_runs": n_runs,
        "mean_reward": float(run_df["mean_reward"].mean()),
        "std_reward": float(run_df["mean_reward"].std(ddof=1)) if n_runs > 1 else 0.0,
        "mean_regret": float(run_df["mean_regret"].mean()),
        "std_regret": float(run_df["mean_regret"].std(ddof=1)) if n_runs > 1 else 0.0,
        "invalid_actions": int(run_df["invalid_actions"].sum()),
        "action_entropy": entropy,
    }
    return summary, decisions

baseline_specs = [
    {"policy": "oracle", "factory": lambda run_id: OraclePolicy(), "n_runs": 1,
     "role": "upper_bound", "uses_oracle_selection": True, "stochastic": False},
    {"policy": "random_scope", "factory": lambda run_id: RandomScopePolicy(seed=RANDOM_STATE + 5000 + run_id),
     "n_runs": 100, "role": "lower_anchor", "uses_oracle_selection": False, "stochastic": True},
    {"policy": "rule_lookup", "factory": lambda run_id: RuleLookupPolicy(), "n_runs": 1,
     "role": "structured_prior", "uses_oracle_selection": False, "stochastic": False},
]

baseline_rows, baseline_decision_frames = [], []
for spec in baseline_specs:
    summary, decisions = evaluate_baseline_family(spec["policy"], spec["factory"], test_eps, spec["n_runs"])
    summary.update({
        "role": spec["role"],
        "uses_oracle_selection": spec["uses_oracle_selection"],
        "stochastic": spec["stochastic"],
    })
    baseline_rows.append(summary)
    baseline_decision_frames.append(decisions)

baseline_df = pd.DataFrame(baseline_rows)
baseline_decisions = pd.concat(baseline_decision_frames, ignore_index=True)

# Anchors for normalisation. The random anchor is averaged over 100 seeded runs.
R_RANDOM_TEST = baseline_df.loc[baseline_df.policy == "random_scope", "mean_reward"].values[0]
R_ORACLE_TEST = baseline_df.loc[baseline_df.policy == "oracle", "mean_reward"].values[0]

def norm_reward(r: float) -> float:
    """Normalize reward between random-scope baseline and oracle ceiling."""
    return (r - R_RANDOM_TEST) / max(R_ORACLE_TEST - R_RANDOM_TEST, 1e-12)

baseline_df["norm_reward"] = baseline_df["mean_reward"].map(norm_reward)

baseline_contract = pd.DataFrame([
    {"check": "test_set_real_only", "passed": not any(e.is_synthetic for e in test_eps),
     "detail": f"test_synthetic={sum(e.is_synthetic for e in test_eps)}"},
    {"check": "all_baseline_actions_valid", "passed": int(baseline_df["invalid_actions"].sum()) == 0,
     "detail": f"invalid_actions={int(baseline_df['invalid_actions'].sum())}"},
    {"check": "oracle_is_upper_anchor", "passed": R_ORACLE_TEST >= baseline_df["mean_reward"].max() - 1e-12,
     "detail": f"oracle={R_ORACLE_TEST:.4f}, max_policy={baseline_df['mean_reward'].max():.4f}"},
    {"check": "random_is_lower_anchor", "passed": R_RANDOM_TEST <= baseline_df.loc[baseline_df.policy != "random_scope", "mean_reward"].min() + 1e-12,
     "detail": f"random={R_RANDOM_TEST:.4f}"},
    {"check": "rule_lookup_non_oracle", "passed": not bool(baseline_df.loc[baseline_df.policy == "rule_lookup", "uses_oracle_selection"].iloc[0]),
     "detail": "rule_lookup uses only RC action ordering"},
])
if not baseline_contract["passed"].all():
    raise AssertionError(baseline_contract.loc[~baseline_contract["passed"]].to_string(index=False))

baseline_rc_summary = (
    baseline_decisions.groupby(["policy", "primary_rc"], dropna=False)
    .agg(mean_reward=("reward", "mean"),
         mean_regret=("regret", "mean"),
         episodes=("episode_id", "nunique"),
         decisions=("episode_id", "size"))
    .reset_index()
)
baseline_action_summary = (
    baseline_decisions.groupby(["policy", "chosen_action"], dropna=False)
    .agg(decisions=("episode_id", "size"),
         mean_reward=("reward", "mean"),
         mean_regret=("regret", "mean"))
    .reset_index()
)
baseline_action_summary["decision_share"] = baseline_action_summary["decisions"] / baseline_action_summary.groupby("policy")["decisions"].transform("sum")

print("Baseline contract audit:")
print(baseline_contract.to_string(index=False))
print("\nReference baseline summary on real chronological holdout:")
print(baseline_df[["policy", "role", "n_runs", "mean_reward", "std_reward", "mean_regret", "std_regret", "norm_reward", "action_entropy"]].round(4).to_string(index=False))
print(f"\nNormalisation anchors: random={R_RANDOM_TEST:.4f}  oracle={R_ORACLE_TEST:.4f}")
print("\nBaseline regret by root cause:")
print(baseline_rc_summary.pivot_table(index="primary_rc", columns="policy", values="mean_regret").round(4).fillna("").to_string())
print("\nBaseline action shares:")
print(baseline_action_summary.sort_values(["policy", "decision_share"], ascending=[True, False]).round(4).to_string(index=False))

fig_baseline_reward = px.bar(
    baseline_df, x="policy", y="mean_reward", error_y="std_reward", color="role",
    text="mean_reward", title="Reference Baseline Reward on Real Chronological Holdout",
    labels={"policy": "Baseline", "mean_reward": "Mean reward", "role": "Baseline role"},
)
fig_baseline_reward.add_hline(y=R_RANDOM_TEST, line_dash="dash", line_color="#666666",
                              annotation_text=f"random anchor {R_RANDOM_TEST:.3f}")
fig_baseline_reward.add_hline(y=R_ORACLE_TEST, line_dash="dot", line_color="#7b3294",
                              annotation_text=f"oracle ceiling {R_ORACLE_TEST:.3f}")
fig_baseline_reward.update_traces(texttemplate="%{text:.4f}", textposition="outside")
fig_baseline_reward.update_layout(width=820, height=450)
save_fig(fig_baseline_reward, "fig03m_reference_baseline_reward"); fig_baseline_reward.show()

fig_baseline_regret = px.bar(
    baseline_rc_summary, x="primary_rc", y="mean_regret", color="policy", barmode="group",
    hover_data=["episodes", "decisions"],
    title="Reference Baseline Regret by Root Cause",
    labels={"primary_rc": "Root cause", "mean_regret": "Mean regret"},
)
fig_baseline_regret.update_xaxes(tickangle=-35)
fig_baseline_regret.update_layout(width=1050, height=470)
save_fig(fig_baseline_regret, "fig03n_reference_baseline_regret_by_rc"); fig_baseline_regret.show()

fig_baseline_actions = px.bar(
    baseline_action_summary, x="chosen_action", y="decision_share", color="policy", barmode="group",
    hover_data=["decisions", "mean_reward", "mean_regret"],
    title="Reference Baseline Action Shares",
    labels={"chosen_action": "Chosen action", "decision_share": "Decision share"},
)
fig_baseline_actions.update_xaxes(tickangle=-35)
fig_baseline_actions.update_layout(width=1100, height=470)
save_fig(fig_baseline_actions, "fig03o_reference_baseline_action_shares"); fig_baseline_actions.show()

regret_heat = baseline_rc_summary.pivot(index="primary_rc", columns="policy", values="mean_regret")
regret_heat = regret_heat.reindex(index=RC_VOCAB + [RC_NO_PROBLEM], columns=["random_scope", "rule_lookup", "oracle"])
fig_regret_heat = px.imshow(
    regret_heat.values,
    x=list(regret_heat.columns), y=list(regret_heat.index),
    text_auto=".3f", aspect="auto", color_continuous_scale="Reds",
    title="Reference Baseline Regret Heatmap",
    labels={"x": "Baseline", "y": "Root cause", "color": "Mean regret"},
)
fig_regret_heat.update_layout(width=760, height=520)
save_fig(fig_regret_heat, "fig03p_reference_baseline_regret_heatmap"); fig_regret_heat.show()


Baseline contract audit:
                     check  passed                                   detail
        test_set_real_only    True                         test_synthetic=0
all_baseline_actions_valid    True                        invalid_actions=0
    oracle_is_upper_anchor    True         oracle=0.6685, max_policy=0.6685
    random_is_lower_anchor    True                            random=0.5520
    rule_lookup_non_oracle    True rule_lookup uses only RC action ordering

Reference baseline summary on real chronological holdout:
      policy             role  n_runs  mean_reward  std_reward  mean_regret  std_regret  norm_reward  action_entropy
      oracle      upper_bound       1       0.6685      0.0000       0.0000      0.0000       1.0000          1.9494
random_scope     lower_anchor     100       0.5520      0.0116       0.1165      0.0116       0.0000          2.9973
 rule_lookup structured_prior       1       0.6351      0.0000       0.0333      0.0000       0.7139         

**Interpretation:** We use the reference baselines to set the benchmark scale before any learned model appears. The contract audit passes: the test set has 0 synthetic episodes, all baseline choices stay inside the RC-action mask, the oracle is the upper anchor, the random-scope policy is the lower anchor, and the rule lookup does not use oracle reward at selection time. We also average the random baseline over 100 seeded runs, so the normalization anchor is not determined by one lucky random draw.

The baseline reward figure shows the scale of the task. The random-scope anchor reaches 0.5520 mean reward with seed-level standard deviation 0.0116, the oracle ceiling is 0.6685, and the rule lookup reaches 0.6351. That means the simple RC action ordering closes 71.39% of the distance between random and oracle. We interpret this as a hard baseline, not a weak strawman. A learned bandit has to improve on an informed operational prior, not merely beat random action selection.

The regret-by-root-cause and regret heatmap figures show where the baselines disagree. Oracle regret is 0 by definition. Rule lookup has low overall regret at 0.0333, but its errors are concentrated: it has 0.3000 regret on `RC_COVERAGE_HOLE`, 0.1500 on `RC_SINR_DEGRADED`, and 0.0500 on both `RC_CAPACITY_OVERLOAD` and `RC_WEAK_SIGNAL`. It has 0 regret on `RC_TRANSPORT_DELAY` and `RC_NONE`, which matters because those two classes dominate the real holdout. Random-scope regret is higher at 0.1165 because it spreads probability across all allowed actions, including low-reward choices such as `ACT_MODEM_SOFT_RESET` for transport delay. The action-share figure confirms the intended behavior: `RC_NONE` episodes push all baselines toward `ACT_NO_OP`, while actionable RCs expose wider action choices. We use these figures as the reference frame for M6, M7, and M8: a model is scientifically useful only if it beats random and can justify any improvement over the strong rule baseline.

<a id="sec-12"></a>

## 12 - M6: Vanilla LinUCB (per-RC dispatcher)

M6 is the first contextual learner in the benchmark. It uses per-arm Ridge regression with UCB exploration (Li et al. 2010), shares the executed encoder dimension `D=49`, and warm-starts each arm with the diagnosis prior before enough observations are available to separate actions by KPI context.

In [15]:
@dataclass
class LinUCBArm:
    """One ridge-regression arm for a single root-cause/action pair."""
    d:   int
    lam: float = 1.0
    A:   np.ndarray = field(init=False)
    b:   np.ndarray = field(init=False)
    n_updates: int  = field(default=0, init=False)

    def __post_init__(self):
        """Initialize the ridge design matrix and reward vector."""
        self.A = self.lam * np.eye(self.d, dtype=np.float64)
        self.b = np.zeros(self.d, dtype=np.float64)

    def update(self, x: np.ndarray, r: float) -> None:
        """Apply one LinUCB ridge update for context vector `x` and reward `r`."""
        x = x.astype(np.float64).ravel()
        assert x.shape[0] == self.d, f"x dim {x.shape[0]} != arm dim {self.d}"
        self.A += np.outer(x, x)
        self.b += r * x
        self.n_updates += 1

    def theta(self) -> np.ndarray:
        """Return the current ridge coefficient estimate."""
        return np.linalg.solve(self.A, self.b)

    def ucb_score(self, x: np.ndarray, alpha: float) -> float:
        """Return the linear prediction plus the LinUCB uncertainty bonus."""
        x = x.astype(np.float64).ravel()
        assert x.shape[0] == self.d, f"x dim {x.shape[0]} != arm dim {self.d}"
        theta  = self.theta()
        A_inv  = np.linalg.inv(self.A)
        return float(theta @ x) + float(alpha * np.sqrt(x @ A_inv @ x))


def diagnosis_prior(rc: str, action: str) -> float:
    """Return 1.0 for the canonical first action of an RC mask, otherwise 0.0."""
    canonical = RC_TO_ACTIONS.get(rc, ["ACT_NO_OP"])
    return 1.0 if (canonical and action == canonical[0]) else 0.0


class LinUCBDispatcher:
    """Per-root-cause LinUCB dispatcher over the allowed RC-action mask.

    The dispatcher owns one arm for every allowed `(root_cause, action)` pair.
    Every arm uses the same encoder dimension `D`, and selection is restricted to
    the episode's allowed action list.
    """
    def __init__(self, rc_to_actions, D, alpha=0.4, lam=3.0, beta0=2.0, rng=None):
        """Create one LinUCB arm per allowed root-cause/action pair."""
        self.D     = D
        self.alpha = alpha
        self.lam   = lam
        self.beta0 = beta0
        self.rng   = rng or np.random.default_rng(42)
        self.rc_to_actions = rc_to_actions
        self.arms: dict[tuple[str, str], LinUCBArm] = {}
        for rc, actions in rc_to_actions.items():
            for action in actions:
                self.arms[(rc, action)] = LinUCBArm(d=D, lam=lam)

    def _ucb_score_for(self, rc: str, action: str, x: np.ndarray) -> float:
        """Return the LinUCB score plus a decaying diagnosis-prior warm start."""
        arm = self.arms.get((rc, action))
        if arm is None:
            return float("-inf")
        prior = self.beta0 / np.sqrt(1 + arm.n_updates) * diagnosis_prior(rc, action)
        return arm.ucb_score(x, self.alpha) + prior

    def select(self, rc: str, x: np.ndarray, allowed_actions: list[str]) -> int:
        """Return the index of the highest-scoring allowed action."""
        scores = np.array([self._ucb_score_for(rc, action, x) for action in allowed_actions], dtype=float)
        best = np.flatnonzero(np.isclose(scores, scores.max(), rtol=1e-12, atol=1e-12))
        return int(self.rng.choice(best))

    def update(self, rc: str, action_code: str, r: float, x: np.ndarray) -> None:
        """Update the arm for one observed root-cause/action/reward tuple."""
        arm = self.arms.get((rc, action_code))
        if arm is not None:
            arm.update(x, r)

    def ucb(self, rc: str, action: str, x: np.ndarray) -> float:
        """Return the score for one action, used later by the M7 arbiter."""
        return self._ucb_score_for(rc, action, x)

    def arm_audit_frame(self) -> pd.DataFrame:
        """Return one audit row per LinUCB arm."""
        rows = []
        for (rc, action), arm in self.arms.items():
            rows.append({
                "root_cause": rc,
                "action": action,
                "dimension": arm.d,
                "lambda": arm.lam,
                "updates": arm.n_updates,
                "diagnosis_prior": diagnosis_prior(rc, action),
                "action_cost": ACTION_BY_CODE[action]["cost_weight"],
                "action_risky": ACTION_BY_CODE[action]["risky"],
            })
        return pd.DataFrame(rows)

def validate_m6_dispatcher(dispatcher: LinUCBDispatcher) -> pd.DataFrame:
    """Validate arm coverage, dimensions, masks, and hyperparameter bounds."""
    expected_pairs = {(rc, action) for rc, actions in RC_TO_ACTIONS.items() for action in actions}
    actual_pairs = set(dispatcher.arms)
    arm_df = dispatcher.arm_audit_frame()
    rows = [
        {"check": "all_allowed_pairs_have_arms", "passed": expected_pairs.issubset(actual_pairs),
         "detail": f"missing={len(expected_pairs - actual_pairs)}"},
        {"check": "no_arms_for_disallowed_pairs", "passed": actual_pairs.issubset(expected_pairs),
         "detail": f"extra={len(actual_pairs - expected_pairs)}"},
        {"check": "all_arms_match_encoder_dimension", "passed": bool((arm_df["dimension"] == dispatcher.D).all()),
         "detail": f"D={dispatcher.D}, arms={len(arm_df)}"},
        {"check": "exploration_alpha_nonnegative", "passed": dispatcher.alpha >= 0,
         "detail": f"alpha={dispatcher.alpha}"},
        {"check": "ridge_lambda_positive", "passed": dispatcher.lam > 0,
         "detail": f"lambda={dispatcher.lam}"},
        {"check": "diagnosis_prior_nonnegative", "passed": dispatcher.beta0 >= 0,
         "detail": f"beta0={dispatcher.beta0}"},
    ]
    return pd.DataFrame(rows)

m6_audit_dispatcher = LinUCBDispatcher(RC_TO_ACTIONS, D, alpha=0.4, lam=3.0, beta0=2.0,
                                       rng=np.random.default_rng(RANDOM_STATE + 600))
m6_dispatcher_validation = validate_m6_dispatcher(m6_audit_dispatcher)
if not m6_dispatcher_validation["passed"].all():
    raise AssertionError(m6_dispatcher_validation.loc[~m6_dispatcher_validation["passed"]].to_string(index=False))

m6_arm_audit = m6_audit_dispatcher.arm_audit_frame()
m6_arm_rc_summary = (
    m6_arm_audit.groupby("root_cause", dropna=False)
    .agg(arms=("action", "size"),
         diagnosis_prior_actions=("diagnosis_prior", "sum"),
         mean_action_cost=("action_cost", "mean"),
         risky_actions=("action_risky", "sum"))
    .reindex(RC_VOCAB + [RC_NO_PROBLEM])
    .reset_index()
)

print("M6 dispatcher validation:")
print(m6_dispatcher_validation.to_string(index=False))
print(f"\nLinUCBDispatcher ready. All {len(m6_arm_audit)} arms are D={D}-dimensional.")
print("\nM6 arm summary by root cause:")
print(m6_arm_rc_summary.round(4).to_string(index=False))

fig_m6_arms = px.bar(
    m6_arm_rc_summary, x="root_cause", y="arms", color="risky_actions", text="arms",
    title="M6 LinUCB Arm Coverage by Root Cause",
    labels={"root_cause": "Root cause", "arms": "Arms / allowed actions", "risky_actions": "Risky arms"},
)
fig_m6_arms.update_xaxes(tickangle=-35)
fig_m6_arms.update_layout(width=950, height=430)
save_fig(fig_m6_arms, "fig04a_m6_arm_coverage"); fig_m6_arms.show()

fig_m6_prior = px.scatter(
    m6_arm_audit, x="action_cost", y="diagnosis_prior", color="root_cause", symbol="action_risky",
    hover_data=["action"], title="M6 Diagnosis-Prior Warm Start by Arm",
    labels={"action_cost": "Action cost", "diagnosis_prior": "Initial diagnosis prior"},
)
fig_m6_prior.update_layout(width=900, height=430)
save_fig(fig_m6_prior, "fig04b_m6_diagnosis_prior_by_arm"); fig_m6_prior.show()


M6 dispatcher validation:
                           check  passed        detail
     all_allowed_pairs_have_arms    True     missing=0
    no_arms_for_disallowed_pairs    True       extra=0
all_arms_match_encoder_dimension    True D=49, arms=28
   exploration_alpha_nonnegative    True     alpha=0.4
           ridge_lambda_positive    True    lambda=3.0
     diagnosis_prior_nonnegative    True     beta0=2.0

LinUCBDispatcher ready. All 28 arms are D=49-dimensional.

M6 arm summary by root cause:
          root_cause  arms  diagnosis_prior_actions  mean_action_cost  risky_actions
      RC_WEAK_SIGNAL     4                   1.0000            0.1750              1
    RC_SINR_DEGRADED     4                   1.0000            0.1750              1
       RC_HO_FAILURE     3                   1.0000            0.2167              1
   RC_PRB_CONGESTION     3                   1.0000            0.1833              0
  RC_TRANSPORT_DELAY     4                   1.0000            0.3375     

**Interpretation:** M6 is a standard contextual bandit over the same RC-action safety mask used by every other policy. The dispatcher validation passes: all 28 allowed RC-action pairs have arms, there are 0 missing arms and 0 extra disallowed arms, every arm uses the executed `D=49` encoder dimension, `alpha=0.4` is nonnegative, ridge `lambda=3.0` is positive, and the diagnosis-prior weight `beta0=2.0` is nonnegative. This means M6 cannot win by seeing a different feature vector or by selecting actions outside the catalogue.

The arm-coverage figure shows the operational shape of the dispatcher. `RC_WEAK_SIGNAL`, `RC_SINR_DEGRADED`, `RC_TRANSPORT_DELAY`, and `RC_CAPACITY_OVERLOAD` each create 4 arms; `RC_HO_FAILURE`, `RC_PRB_CONGESTION`, and `RC_COVERAGE_HOLE` create 3 arms; `RC_CQI_MISMATCH` creates 2 arms; and `RC_NONE` creates only the no-op arm. The risk exposure is also bounded: the risky action count is 1 for weak signal, SINR degradation, handover failure, and transport delay, and 0 for the other root causes. The diagnosis-prior figure shows the warm start clearly: exactly one canonical action per root cause receives prior score 1.0 before data arrive, while the other allowed actions start at 0.0 and must earn selection through observed reward. We read this as a conservative online baseline: it starts from the expert action ordering, then lets KPI context move it away from that ordering.

In [16]:
def run_linucb_episode(dispatcher: LinUCBDispatcher, episodes: list[Episode],
                       stats: dict, update: bool = True, collect: bool = False) -> dict:
    """Run one chronological LinUCB pass and optionally collect per-step decisions."""
    rewards, regrets, decision_rows = [], [], []
    for step, ep in enumerate(episodes):
        if not ep.primary_rc or ep.primary_rc is None:
            continue
        x = _encode_state({**ep.state, "__rc__": ep.primary_rc}, stats)
        assert x.shape[0] == dispatcher.D, (
            f"Encoder dim {x.shape[0]} != dispatcher.D {dispatcher.D}. "
            "Ensure _encode_state and LinUCBDispatcher use the same D."
        )
        idx = dispatcher.select(ep.primary_rc, x, ep.allowed_actions)
        action = ep.allowed_actions[idx]
        reward, _ = oracle_reward(ep.primary_rc, action, ep.state)
        best = max(oracle_reward(ep.primary_rc, candidate, ep.state)[0] for candidate in ep.allowed_actions)
        regret = best - reward
        rewards.append(reward)
        regrets.append(regret)
        if collect:
            decision_rows.append({
                "step": step,
                "episode_id": ep.episode_id,
                "timestamp": ep.timestamp,
                "primary_rc": ep.primary_rc,
                "chosen_action": action,
                "reward": reward,
                "best_reward": best,
                "regret": regret,
                "allowed_action_count": len(ep.allowed_actions),
                "is_synthetic": ep.is_synthetic,
                "updated": update,
            })
        if update:
            dispatcher.update(ep.primary_rc, action, reward, x)
    return {
        "mean_reward": float(np.mean(rewards)) if rewards else float("nan"),
        "std_reward": float(np.std(rewards, ddof=1)) if len(rewards) > 1 else 0.0,
        "mean_regret": float(np.mean(regrets)) if regrets else float("nan"),
        "per_step_regret": regrets,
        "decisions": pd.DataFrame(decision_rows),
    }

cum_traces = []
train_diag_frames = []
for seed in range(10):
    rng = np.random.default_rng(RANDOM_STATE + 100 + seed)
    bandit = LinUCBDispatcher(RC_TO_ACTIONS, D, alpha=0.4, lam=3.0, beta0=2.0, rng=rng)
    result = run_linucb_episode(bandit, train_eps, train_stats, update=True, collect=True)
    cum_traces.append(np.cumsum(result["per_step_regret"]))
    frame = result["decisions"].copy()
    frame["seed"] = seed
    frame["cumulative_regret"] = frame["regret"].cumsum()
    train_diag_frames.append(frame)

cum_mat = np.array(cum_traces)
cum_mean = cum_mat.mean(axis=0)
cum_lo = np.percentile(cum_mat, 10, axis=0)
cum_hi = np.percentile(cum_mat, 90, axis=0)
m6_train_diagnostics = pd.concat(train_diag_frames, ignore_index=True)
m6_train_regret_by_rc = (
    m6_train_diagnostics.groupby("primary_rc", dropna=False)
    .agg(mean_regret=("regret", "mean"),
         mean_reward=("reward", "mean"),
         decisions=("episode_id", "size"),
         episodes=("episode_id", "nunique"))
    .reset_index()
)
m6_cum_summary = pd.DataFrame({
    "episode": np.arange(len(cum_mean)),
    "cum_regret_mean": cum_mean,
    "cum_regret_p10": cum_lo,
    "cum_regret_p90": cum_hi,
})

print("M6 training diagnostic summary across 10 seeds:")
print(pd.DataFrame([{
    "train_episodes": len(train_eps),
    "mean_final_cum_regret": float(cum_mean[-1]),
    "p10_final_cum_regret": float(cum_lo[-1]),
    "p90_final_cum_regret": float(cum_hi[-1]),
    "mean_per_step_regret": float(cum_mean[-1] / max(1, len(train_eps))),
}]).round(4).to_string(index=False))
print("\nM6 training regret by root cause:")
print(m6_train_regret_by_rc.round(4).to_string(index=False))

fig = go.Figure()
fig.add_trace(go.Scatter(y=cum_hi, line=dict(width=0), showlegend=False))
fig.add_trace(go.Scatter(y=cum_lo, line=dict(width=0), fill="tonexty",
                         fillcolor="rgba(31,119,180,0.2)", name="10-90 pct", showlegend=False))
fig.add_trace(go.Scatter(y=cum_mean, mode="lines",
                         line=dict(width=2, color="#1f77b4"), name="mean"))
fig.update_layout(title_text="M6 LinUCB - Cumulative Regret on Train (10 seeds)",
                  xaxis_title="episode (chronological)", yaxis_title="cumulative regret",
                  width=900, height=400, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig, "fig04_m6_cumregret"); fig.show()

fig_train_rc = px.bar(
    m6_train_regret_by_rc, x="primary_rc", y="mean_regret", color="mean_reward",
    text="episodes", title="M6 Training Regret by Root Cause Across 10 Seeds",
    labels={"primary_rc": "Root cause", "mean_regret": "Mean regret", "mean_reward": "Mean reward"},
)
fig_train_rc.update_xaxes(tickangle=-35)
fig_train_rc.update_layout(width=950, height=430)
save_fig(fig_train_rc, "fig04c_m6_train_regret_by_rc"); fig_train_rc.show()


M6 training diagnostic summary across 10 seeds:
 train_episodes  mean_final_cum_regret  p10_final_cum_regret  p90_final_cum_regret  mean_per_step_regret
            437                17.0630               16.4580               17.5720                0.0390

M6 training regret by root cause:
          primary_rc  mean_regret  mean_reward  decisions  episodes
RC_CAPACITY_OVERLOAD       0.1974       0.5696        200        20
     RC_CQI_MISMATCH       0.0075       0.4425        200        20
       RC_HO_FAILURE       0.0000       0.8500        200        20
             RC_NONE       0.0000       0.9000       1730       173
   RC_PRB_CONGESTION       0.1634       0.6926        200        20
    RC_SINR_DEGRADED       0.0740       0.4010        200        20
  RC_TRANSPORT_DELAY       0.0571       0.5003       1440       144
      RC_WEAK_SIGNAL       0.0000       0.8500        200        20


**Interpretation:** The cumulative-regret plot checks whether M6 learns useful action preferences on the training chronology before we trust its holdout score. Across 10 seeded training passes, final cumulative regret averages 17.0630 over 437 train episodes, with a 10th-to-90th percentile range from 16.4580 to 17.5720. The mean per-step training regret is 0.0390, so the bandit is not randomly wandering, but it still pays an exploration and adaptation cost.

The training root-cause regret figure shows where that cost appears. `RC_CAPACITY_OVERLOAD` has the highest training regret at 0.1974, followed by `RC_PRB_CONGESTION` at 0.1634, `RC_SINR_DEGRADED` at 0.0740, and `RC_TRANSPORT_DELAY` at 0.0571. `RC_HO_FAILURE`, `RC_WEAK_SIGNAL`, and `RC_NONE` have 0.0000 training regret in this diagnostic. We interpret this as evidence that M6 learns the easy anchored cases quickly, while congestion and capacity decisions remain harder because several allowed actions can look plausible before enough reward evidence accumulates. These plots are learning diagnostics only; the real deployment comparison remains the future chronological holdout.

In [17]:
def seed_sweep_m6(n_seeds: int = 50, alpha: float = 0.4, lam: float = 3.0, beta0: float = 2.0) -> pd.DataFrame:
    """Train and evaluate M6 across seeds with frozen and online test views."""
    rows = []
    for seed in range(n_seeds):
        rng_train = np.random.default_rng(RANDOM_STATE + 200 + seed)
        frozen_bandit = LinUCBDispatcher(RC_TO_ACTIONS, D, alpha=alpha, lam=lam, beta0=beta0, rng=rng_train)
        train_result = run_linucb_episode(frozen_bandit, _TRAINVAL, train_stats, update=True)
        frozen_result = run_linucb_episode(frozen_bandit, test_eps, train_stats, update=False)

        rng_online = np.random.default_rng(RANDOM_STATE + 200 + seed)
        online_bandit = LinUCBDispatcher(RC_TO_ACTIONS, D, alpha=alpha, lam=lam, beta0=beta0, rng=rng_online)
        run_linucb_episode(online_bandit, _TRAINVAL, train_stats, update=True)
        online_result = run_linucb_episode(online_bandit, test_eps, train_stats, update=True)
        rows.append({
            "seed": seed,
            "train_reward": train_result["mean_reward"],
            "train_regret": train_result["mean_regret"],
            "test_frozen_reward": frozen_result["mean_reward"],
            "test_frozen_regret": frozen_result["mean_regret"],
            "test_online_reward": online_result["mean_reward"],
            "test_online_regret": online_result["mean_regret"],
        })
    return pd.DataFrame(rows)

def evaluate_m6_holdout_decisions(seed: int = RANDOM_STATE + 200) -> pd.DataFrame:
    """Return one representative online M6 decision trace on the real holdout."""
    bandit = LinUCBDispatcher(RC_TO_ACTIONS, D, alpha=0.4, lam=3.0, beta0=2.0,
                              rng=np.random.default_rng(seed))
    run_linucb_episode(bandit, _TRAINVAL, train_stats, update=True)
    result = run_linucb_episode(bandit, test_eps, train_stats, update=True, collect=True)
    decisions = result["decisions"].copy()
    decisions["cumulative_regret"] = decisions["regret"].cumsum()
    return decisions

print("Running M6 seed sweep (50 seeds) ...")
m6_sweep = seed_sweep_m6(n_seeds=50)
m6_online = m6_sweep["test_online_reward"]
m6_holdout_decisions = evaluate_m6_holdout_decisions()
m6_holdout_rc = (
    m6_holdout_decisions.groupby("primary_rc", dropna=False)
    .agg(episodes=("episode_id", "nunique"),
         mean_reward=("reward", "mean"),
         mean_regret=("regret", "mean"),
         total_regret=("regret", "sum"))
    .reset_index()
)
m6_holdout_actions = (
    m6_holdout_decisions.groupby(["primary_rc", "chosen_action"], dropna=False)
    .agg(decisions=("episode_id", "size"),
         mean_reward=("reward", "mean"),
         mean_regret=("regret", "mean"))
    .reset_index()
)

print("M6 seed sweep summary:")
print(m6_sweep[["train_reward", "test_frozen_reward", "test_online_reward", "test_online_regret"]].agg(["mean", "std", "min", "max"]).round(4).to_string())
print("\nM6 representative online holdout regret by root cause:")
print(m6_holdout_rc.round(4).to_string(index=False))
print("\nM6 representative online holdout action choices:")
print(m6_holdout_actions.sort_values(["primary_rc", "decisions"], ascending=[True, False]).round(4).to_string(index=False))

fig_m6_seed = px.box(
    m6_sweep.melt(id_vars="seed", value_vars=["test_frozen_reward", "test_online_reward"],
                  var_name="test_mode", value_name="reward"),
    x="test_mode", y="reward", color="test_mode",
    title="M6 Holdout Reward Stability Across 50 Seeds",
    labels={"test_mode": "Test mode", "reward": "Mean reward"},
)
fig_m6_seed.update_layout(width=760, height=430, showlegend=False)
save_fig(fig_m6_seed, "fig04d_m6_seed_reward_distribution"); fig_m6_seed.show()

fig_m6_frozen_online = px.scatter(
    m6_sweep, x="test_frozen_reward", y="test_online_reward", text="seed",
    title="M6 Frozen vs Online Holdout Reward by Seed",
    labels={"test_frozen_reward": "Frozen holdout reward", "test_online_reward": "Online holdout reward"},
)
fig_m6_frozen_online.add_shape(type="line", x0=m6_sweep["test_frozen_reward"].min(), y0=m6_sweep["test_frozen_reward"].min(),
                               x1=m6_sweep["test_online_reward"].max(), y1=m6_sweep["test_online_reward"].max(),
                               line=dict(color="#666666", dash="dash"))
fig_m6_frozen_online.update_traces(textposition="top center", marker=dict(size=8, opacity=0.75))
fig_m6_frozen_online.update_layout(width=760, height=520)
save_fig(fig_m6_frozen_online, "fig04e_m6_frozen_vs_online"); fig_m6_frozen_online.show()

fig_m6_holdout_rc = px.bar(
    m6_holdout_rc, x="primary_rc", y="mean_regret", color="mean_reward", text="episodes",
    title="M6 Representative Online Holdout Regret by Root Cause",
    labels={"primary_rc": "Root cause", "mean_regret": "Mean regret", "mean_reward": "Mean reward"},
)
fig_m6_holdout_rc.update_xaxes(tickangle=-35)
fig_m6_holdout_rc.update_layout(width=950, height=430)
save_fig(fig_m6_holdout_rc, "fig04f_m6_holdout_regret_by_rc"); fig_m6_holdout_rc.show()

fig_m6_actions = px.bar(
    m6_holdout_actions, x="primary_rc", y="decisions", color="chosen_action",
    title="M6 Representative Online Holdout Actions by Root Cause",
    labels={"primary_rc": "Root cause", "decisions": "Decisions", "chosen_action": "Chosen action"},
)
fig_m6_actions.update_xaxes(tickangle=-35)
fig_m6_actions.update_layout(width=1050, height=470)
save_fig(fig_m6_actions, "fig04g_m6_holdout_actions_by_rc"); fig_m6_actions.show()


Running M6 seed sweep (50 seeds) ...
M6 seed sweep summary:
      train_reward  test_frozen_reward  test_online_reward  test_online_regret
mean        0.6826              0.5206              0.6158              0.0526
std         0.0020              0.0255              0.0042              0.0042
min         0.6780              0.4816              0.6092              0.0446
max         0.6851              0.5581              0.6239              0.0593

M6 representative online holdout regret by root cause:
          primary_rc  episodes  mean_reward  mean_regret  total_regret
RC_CAPACITY_OVERLOAD         4       0.3825       0.0675        0.2700
    RC_COVERAGE_HOLE         9       0.3178       0.2933        2.6400
             RC_NONE        34       0.9000       0.0000        0.0000
    RC_SINR_DEGRADED         4       0.3825       0.1175        0.4700
  RC_TRANSPORT_DELAY        56       0.5304       0.0321        1.8000
      RC_WEAK_SIGNAL         4       0.5125       0.1875       

**Interpretation:** M6 is evaluated in two holdout modes. The frozen mode trains on train+validation and then scores the future test without updating; the online mode keeps updating as test episodes arrive, which matches a continuously learning deployment. Across 50 seeds, M6 reaches mean online reward 0.6158 with standard deviation 0.0042 and mean online regret 0.0526. The frozen reward is much lower at 0.5206 with standard deviation 0.0255, so the frozen-versus-online figure shows that M6 depends on online adaptation in this benchmark.

The representative online holdout diagnostics show what the aggregate reward hides. M6 has 0.0000 regret on `RC_NONE`, low regret on `RC_TRANSPORT_DELAY` at 0.0321, and moderate regret on `RC_CAPACITY_OVERLOAD` at 0.0675 and `RC_SINR_DEGRADED` at 0.1175. The main weakness is `RC_COVERAGE_HOLE`, where mean regret is 0.2933 across 9 episodes, followed by `RC_WEAK_SIGNAL` at 0.1875 across 4 episodes. The action plot explains why: M6 often chooses `ACT_QUEUE_MANAGE` for transport delay, which works well, but it spreads coverage-hole choices across cellular switch, Wi-Fi switch, and escalation, and the Wi-Fi switch has high regret in this run. We therefore read M6 as a useful online baseline that beats random, but not as a replacement for the strong rule lookup or the offline reward ranker.

<a id="sec-13"></a>

## 13 - EpsilonGreedy Bandit

We then test a simpler online learner that keeps empirical reward means for each allowed root-cause/action arm. EpsilonGreedy serves as the adaptation baseline because it updates directly from observed reward without assuming a linear context model.

In [18]:
class EpsilonGreedyArm:
    """Incremental mean-reward estimate for one root-cause/action arm."""
    def __init__(self):
        """Initialize an empty empirical reward mean."""
        self.mean = 0.0
        self.n = 0

    def update(self, reward: float) -> None:
        """Update the running empirical mean with one observed reward."""
        self.n += 1
        self.mean += (reward - self.mean) / self.n


class EpsilonGreedyDispatcher:
    """Per-root-cause epsilon-greedy dispatcher with a decaying exploration rate."""
    def __init__(self, rc_to_actions: dict[str, list[str]], epsilon_0: float = 0.3, rng=None):
        """Create one empirical-mean arm per allowed root-cause/action pair."""
        if epsilon_0 < 0:
            raise ValueError("epsilon_0 must be nonnegative")
        self.rc_to_actions = rc_to_actions
        self.epsilon_0 = float(epsilon_0)
        self.rng = rng or np.random.default_rng(42)
        self.arms = {
            (rc, action): EpsilonGreedyArm()
            for rc, actions in rc_to_actions.items()
            for action in actions
        }
        self.t = 0
        self.last_selection: dict = {}

    def epsilon(self) -> float:
        """Return the current decayed exploration probability."""
        return float(self.epsilon_0 / np.sqrt(1 + self.t))

    def select(self, rc: str, allowed_actions: list[str]) -> int:
        """Select an allowed action index using epsilon exploration or greedy means."""
        eps = self.epsilon()
        self.t += 1
        explore = bool(self.rng.random() < eps)
        if explore:
            idx = int(self.rng.integers(0, len(allowed_actions)))
            action_means = {action: self.arms[(rc, action)].mean for action in allowed_actions}
            self.last_selection = {"epsilon": eps, "explore": True, "action_means": action_means}
            return idx
        means = np.array([self.arms[(rc, action)].mean for action in allowed_actions], dtype=float)
        best = np.flatnonzero(np.isclose(means, means.max(), rtol=1e-12, atol=1e-12))
        idx = int(self.rng.choice(best))
        self.last_selection = {
            "epsilon": eps,
            "explore": False,
            "action_means": {action: float(mean) for action, mean in zip(allowed_actions, means)},
        }
        return idx

    def update(self, rc: str, action: str, reward: float) -> None:
        """Update the empirical-mean arm for one observed decision."""
        arm = self.arms.get((rc, action))
        if arm is not None:
            arm.update(reward)

    def arm_audit_frame(self) -> pd.DataFrame:
        """Return one audit row per epsilon-greedy arm."""
        rows = []
        for (rc, action), arm in self.arms.items():
            rows.append({
                "root_cause": rc,
                "action": action,
                "updates": arm.n,
                "mean_reward_estimate": arm.mean,
                "action_cost": ACTION_BY_CODE[action]["cost_weight"],
                "action_risky": ACTION_BY_CODE[action]["risky"],
            })
        return pd.DataFrame(rows)


def validate_eg_dispatcher(dispatcher: EpsilonGreedyDispatcher) -> pd.DataFrame:
    """Validate arm coverage, action masks, and epsilon-greedy hyperparameters."""
    expected_pairs = {(rc, action) for rc, actions in RC_TO_ACTIONS.items() for action in actions}
    actual_pairs = set(dispatcher.arms)
    rows = [
        {"check": "all_allowed_pairs_have_arms", "passed": expected_pairs.issubset(actual_pairs),
         "detail": f"missing={len(expected_pairs - actual_pairs)}"},
        {"check": "no_arms_for_disallowed_pairs", "passed": actual_pairs.issubset(expected_pairs),
         "detail": f"extra={len(actual_pairs - expected_pairs)}"},
        {"check": "epsilon_0_nonnegative", "passed": dispatcher.epsilon_0 >= 0,
         "detail": f"epsilon_0={dispatcher.epsilon_0:.3f}"},
        {"check": "initial_epsilon_bounded", "passed": 0.0 <= dispatcher.epsilon() <= 1.0,
         "detail": f"epsilon_t0={dispatcher.epsilon():.3f}"},
        {"check": "all_actions_exist_in_catalogue", "passed": all(action in ACTION_BY_CODE for _, action in actual_pairs),
         "detail": f"catalogue_actions={len(ACTION_BY_CODE)}"},
    ]
    return pd.DataFrame(rows)


def run_eg_episode(dispatcher: EpsilonGreedyDispatcher, episodes: list[Episode],
                   update: bool = True, collect: bool = False) -> dict:
    """Run one chronological epsilon-greedy pass and optionally collect decisions."""
    rewards, regrets, decision_rows = [], [], []
    for step, ep in enumerate(episodes):
        idx = dispatcher.select(ep.primary_rc, ep.allowed_actions)
        action = ep.allowed_actions[idx]
        reward, _ = oracle_reward(ep.primary_rc, action, ep.state)
        best = max(oracle_reward(ep.primary_rc, candidate, ep.state)[0] for candidate in ep.allowed_actions)
        regret = best - reward
        rewards.append(reward)
        regrets.append(regret)
        if collect:
            meta = dispatcher.last_selection
            decision_rows.append({
                "step": step,
                "episode_id": ep.episode_id,
                "timestamp": ep.timestamp,
                "primary_rc": ep.primary_rc,
                "chosen_action": action,
                "reward": reward,
                "best_reward": best,
                "regret": regret,
                "epsilon": meta.get("epsilon", np.nan),
                "explore": bool(meta.get("explore", False)),
                "allowed_action_count": len(ep.allowed_actions),
                "is_synthetic": ep.is_synthetic,
                "updated": update,
            })
        if update:
            dispatcher.update(ep.primary_rc, action, reward)
    return {
        "mean_reward": float(np.mean(rewards)) if rewards else float("nan"),
        "std_reward": float(np.std(rewards, ddof=1)) if len(rewards) > 1 else 0.0,
        "mean_regret": float(np.mean(regrets)) if regrets else float("nan"),
        "per_step_regret": regrets,
        "decisions": pd.DataFrame(decision_rows),
    }


def seed_sweep_eg(n_seeds: int = 50, epsilon_0: float = 0.3) -> pd.DataFrame:
    """Train and evaluate epsilon-greedy across seeds with frozen and online test views."""
    rows = []
    for seed in range(n_seeds):
        rng_frozen = np.random.default_rng(RANDOM_STATE + 300 + seed)
        frozen_bandit = EpsilonGreedyDispatcher(RC_TO_ACTIONS, epsilon_0=epsilon_0, rng=rng_frozen)
        train_result = run_eg_episode(frozen_bandit, _TRAINVAL, update=True)
        frozen_result = run_eg_episode(frozen_bandit, test_eps, update=False)

        rng_online = np.random.default_rng(RANDOM_STATE + 300 + seed)
        online_bandit = EpsilonGreedyDispatcher(RC_TO_ACTIONS, epsilon_0=epsilon_0, rng=rng_online)
        run_eg_episode(online_bandit, _TRAINVAL, update=True)
        online_result = run_eg_episode(online_bandit, test_eps, update=True)
        rows.append({
            "seed": seed,
            "train_reward": train_result["mean_reward"],
            "train_regret": train_result["mean_regret"],
            "test_frozen_reward": frozen_result["mean_reward"],
            "test_frozen_regret": frozen_result["mean_regret"],
            "test_online_reward": online_result["mean_reward"],
            "test_online_regret": online_result["mean_regret"],
        })
    return pd.DataFrame(rows)


def evaluate_eg_holdout_decisions(seed: int = RANDOM_STATE + 300, epsilon_0: float = 0.3) -> pd.DataFrame:
    """Return one representative online epsilon-greedy decision trace on holdout."""
    bandit = EpsilonGreedyDispatcher(RC_TO_ACTIONS, epsilon_0=epsilon_0, rng=np.random.default_rng(seed))
    run_eg_episode(bandit, _TRAINVAL, update=True)
    result = run_eg_episode(bandit, test_eps, update=True, collect=True)
    decisions = result["decisions"].copy()
    decisions["cumulative_regret"] = decisions["regret"].cumsum()
    return decisions


eg_audit_dispatcher = EpsilonGreedyDispatcher(RC_TO_ACTIONS, epsilon_0=0.3, rng=np.random.default_rng(RANDOM_STATE + 700))
eg_validation = validate_eg_dispatcher(eg_audit_dispatcher)
if not eg_validation["passed"].all():
    raise AssertionError(eg_validation.loc[~eg_validation["passed"]].to_string(index=False))

eg_arm_audit = eg_audit_dispatcher.arm_audit_frame()
eg_arm_rc_summary = (
    eg_arm_audit.groupby("root_cause", dropna=False)
    .agg(arms=("action", "size"),
         mean_action_cost=("action_cost", "mean"),
         risky_actions=("action_risky", "sum"))
    .reindex(RC_VOCAB + [RC_NO_PROBLEM])
    .reset_index()
)
eg_schedule = pd.DataFrame({
    "decision_step": np.arange(0, len(_TRAINVAL) + len(test_eps) + 1),
})
eg_schedule["epsilon"] = 0.3 / np.sqrt(1 + eg_schedule["decision_step"])
eg_schedule_sample = eg_schedule.loc[eg_schedule["decision_step"].isin([0, 1, 10, 50, 100, len(_TRAINVAL), len(_TRAINVAL) + len(test_eps)])]

print("EpsilonGreedy dispatcher validation:")
print(eg_validation.to_string(index=False))
print(f"\nEpsilonGreedy ready. Arms={len(eg_arm_audit)}, epsilon_0=0.300")
print("\nEpsilonGreedy arm summary by root cause:")
print(eg_arm_rc_summary.round(4).to_string(index=False))
print("\nEpsilon schedule checkpoints:")
print(eg_schedule_sample.round(4).to_string(index=False))

fig_eg_arms = px.bar(
    eg_arm_rc_summary, x="root_cause", y="arms", color="risky_actions", text="arms",
    title="EpsilonGreedy Arm Coverage by Root Cause",
    labels={"root_cause": "Root cause", "arms": "Arms / allowed actions", "risky_actions": "Risky arms"},
)
fig_eg_arms.update_xaxes(tickangle=-35)
fig_eg_arms.update_layout(width=950, height=430)
save_fig(fig_eg_arms, "fig04h_eg_arm_coverage"); fig_eg_arms.show()

fig_eg_schedule = px.line(
    eg_schedule, x="decision_step", y="epsilon",
    title="EpsilonGreedy Exploration Schedule",
    labels={"decision_step": "Decision step", "epsilon": "Exploration probability"},
)
fig_eg_schedule.add_vline(x=len(_TRAINVAL), line_dash="dash", line_color="#666666",
                          annotation_text="test starts")
fig_eg_schedule.update_layout(width=850, height=430)
save_fig(fig_eg_schedule, "fig04i_eg_epsilon_schedule"); fig_eg_schedule.show()

print("\nRunning EpsilonGreedy seed sweep (50 seeds) ...")
eg_sweep = seed_sweep_eg(n_seeds=50, epsilon_0=0.3)
eg_online = eg_sweep["test_online_reward"]
eg_holdout_decisions = evaluate_eg_holdout_decisions(epsilon_0=0.3)
eg_holdout_rc = (
    eg_holdout_decisions.groupby("primary_rc", dropna=False)
    .agg(episodes=("episode_id", "nunique"),
         mean_reward=("reward", "mean"),
         mean_regret=("regret", "mean"),
         total_regret=("regret", "sum"),
         exploration_rate=("explore", "mean"))
    .reset_index()
)
eg_holdout_actions = (
    eg_holdout_decisions.groupby(["primary_rc", "chosen_action"], dropna=False)
    .agg(decisions=("episode_id", "size"),
         mean_reward=("reward", "mean"),
         mean_regret=("regret", "mean"),
         exploration_rate=("explore", "mean"))
    .reset_index()
)

print("EpsilonGreedy seed sweep summary:")
print(eg_sweep[["train_reward", "test_frozen_reward", "test_online_reward", "test_online_regret"]].agg(["mean", "std", "min", "max"]).round(4).to_string())
print("\nEpsilonGreedy representative online holdout regret by root cause:")
print(eg_holdout_rc.round(4).to_string(index=False))
print("\nEpsilonGreedy representative online holdout action choices:")
print(eg_holdout_actions.sort_values(["primary_rc", "decisions"], ascending=[True, False]).round(4).to_string(index=False))
print(f"\nEpsilonGreedy online: mean={eg_online.mean():.4f}  std={eg_online.std():.4f}")

fig_eg_seed = px.box(
    eg_sweep.melt(id_vars="seed", value_vars=["test_frozen_reward", "test_online_reward"],
                  var_name="test_mode", value_name="reward"),
    x="test_mode", y="reward", color="test_mode",
    title="EpsilonGreedy Holdout Reward Stability Across 50 Seeds",
    labels={"test_mode": "Test mode", "reward": "Mean reward"},
)
fig_eg_seed.update_layout(width=760, height=430, showlegend=False)
save_fig(fig_eg_seed, "fig04j_eg_seed_reward_distribution"); fig_eg_seed.show()

fig_eg_frozen_online = px.scatter(
    eg_sweep, x="test_frozen_reward", y="test_online_reward", text="seed",
    title="EpsilonGreedy Frozen vs Online Holdout Reward by Seed",
    labels={"test_frozen_reward": "Frozen holdout reward", "test_online_reward": "Online holdout reward"},
)
fig_eg_frozen_online.add_shape(type="line", x0=eg_sweep["test_frozen_reward"].min(), y0=eg_sweep["test_frozen_reward"].min(),
                               x1=eg_sweep["test_online_reward"].max(), y1=eg_sweep["test_online_reward"].max(),
                               line=dict(color="#666666", dash="dash"))
fig_eg_frozen_online.update_traces(textposition="top center", marker=dict(size=8, opacity=0.75))
fig_eg_frozen_online.update_layout(width=760, height=520)
save_fig(fig_eg_frozen_online, "fig04k_eg_frozen_vs_online"); fig_eg_frozen_online.show()

fig_eg_rc = px.bar(
    eg_holdout_rc, x="primary_rc", y="mean_regret", color="exploration_rate", text="episodes",
    title="EpsilonGreedy Representative Online Holdout Regret by Root Cause",
    labels={"primary_rc": "Root cause", "mean_regret": "Mean regret", "exploration_rate": "Exploration rate"},
)
fig_eg_rc.update_xaxes(tickangle=-35)
fig_eg_rc.update_layout(width=950, height=430)
save_fig(fig_eg_rc, "fig04l_eg_holdout_regret_by_rc"); fig_eg_rc.show()

fig_eg_actions = px.bar(
    eg_holdout_actions, x="primary_rc", y="decisions", color="chosen_action",
    title="EpsilonGreedy Representative Online Holdout Actions by Root Cause",
    labels={"primary_rc": "Root cause", "decisions": "Decisions", "chosen_action": "Chosen action"},
)
fig_eg_actions.update_xaxes(tickangle=-35)
fig_eg_actions.update_layout(width=1050, height=470)
save_fig(fig_eg_actions, "fig04m_eg_holdout_actions_by_rc"); fig_eg_actions.show()


EpsilonGreedy dispatcher validation:
                         check  passed               detail
   all_allowed_pairs_have_arms    True            missing=0
  no_arms_for_disallowed_pairs    True              extra=0
         epsilon_0_nonnegative    True      epsilon_0=0.300
       initial_epsilon_bounded    True     epsilon_t0=0.300
all_actions_exist_in_catalogue    True catalogue_actions=15

EpsilonGreedy ready. Arms=28, epsilon_0=0.300

EpsilonGreedy arm summary by root cause:
          root_cause  arms  mean_action_cost  risky_actions
      RC_WEAK_SIGNAL     4            0.1750              1
    RC_SINR_DEGRADED     4            0.1750              1
       RC_HO_FAILURE     3            0.2167              1
   RC_PRB_CONGESTION     3            0.1833              0
  RC_TRANSPORT_DELAY     4            0.3375              1
     RC_CQI_MISMATCH     2            0.1500              0
    RC_COVERAGE_HOLE     3            0.1667              0
RC_CAPACITY_OVERLOAD     4        


Running EpsilonGreedy seed sweep (50 seeds) ...
EpsilonGreedy seed sweep summary:
      train_reward  test_frozen_reward  test_online_reward  test_online_regret
mean        0.6490              0.6227              0.6240              0.0445
std         0.0265              0.0246              0.0255              0.0255
min         0.5883              0.5690              0.5650              0.0079
max         0.7014              0.6483              0.6605              0.1034

EpsilonGreedy representative online holdout regret by root cause:
          primary_rc  episodes  mean_reward  mean_regret  total_regret  exploration_rate
RC_CAPACITY_OVERLOAD         4       0.4500       0.0000        0.0000            0.0000
    RC_COVERAGE_HOLE         9       0.2889       0.3222        2.9000            0.0000
             RC_NONE        34       0.9000       0.0000        0.0000            0.0294
    RC_SINR_DEGRADED         4       0.5000       0.0000        0.0000            0.0000
  RC_TRANS

**Interpretation:** We use EpsilonGreedy as the simplest online learning baseline: it keeps one empirical mean reward for each allowed RC-action arm, chooses randomly with a decaying probability, and otherwise chooses the currently best empirical arm. The dispatcher audit passes: all 28 allowed RC-action pairs have arms, there are 0 missing arms and 0 extra disallowed arms, `epsilon_0=0.300` is valid, initial epsilon is 0.300, and every action exists in the catalogue. The arm-coverage figure mirrors the safety mask exactly, so EpsilonGreedy has the same action space as M6 but no linear context model.

The epsilon schedule figure shows the exploration policy we actually use. Exploration starts at 0.3000, drops to 0.0905 by decision step 10, 0.0299 by step 100, 0.0117 when the real test window begins after 662 train+validation decisions, and 0.0108 by the final evaluated decision. This means most holdout choices are exploitative, but the policy still explores occasionally. We read that as a practical deployment compromise: EpsilonGreedy remains adaptive, but it does not keep injecting heavy random action noise into the future test window.

The seed distribution figure shows why this baseline is strong but noisy. Across 50 seeds, online reward averages 0.6240 with standard deviation 0.0255, and online regret averages 0.0445. The frozen mean is similar at 0.6227, which is very different from M6: EpsilonGreedy relies less on test-time updating because its empirical means already encode the dominant action choices from train+validation. The reward range is wide, though: online reward moves from 0.5650 to 0.6605 across seeds, so random tie-breaking and rare exploration still matter.

The representative online holdout figures show what the aggregate score hides. EpsilonGreedy gets 0.0000 regret on `RC_TRANSPORT_DELAY`, `RC_NONE`, `RC_CAPACITY_OVERLOAD`, and `RC_SINR_DEGRADED`; for example, it chooses `ACT_QUEUE_MANAGE` for all 56 transport-delay episodes and `ACT_NO_OP` for all 34 no-problem episodes. The failures are concentrated in smaller classes: `RC_COVERAGE_HOLE` has mean regret 0.3222 because the policy chooses `ACT_SWITCH_LINK_WIFI` for all 9 coverage-hole episodes, and `RC_WEAK_SIGNAL` has mean regret 0.4000 because it chooses `ACT_ESCALATE` for all 4 weak-signal episodes. We therefore treat EpsilonGreedy as the best simple online baseline, but not as a robust per-RC solution.

<a id="sec-14"></a>

## 14 - M7: LLM-Guided LinUCB (Qwen2.5-3B via Ollama)

M7 adds a constrained reasoning layer on top of LinUCB. LinUCB proposes a top-3 shortlist, the local Qwen model receives the root cause, confidence, causal chain, KPI snapshot, and shortlist, and the returned JSON choice is used only when it is valid; otherwise the policy falls back to the UCB argmax. The notebook records coverage, fallbacks, overrides, rewards, regret, and reasoning text from the executed cache rather than assuming full model availability.

In [19]:
import hashlib as _hl, json as _js
OLLAMA_URL = "http://localhost:11434/api/generate"
LLM_MODEL = "qwen2.5:3b"
LLM_TIMEOUT_S = 30
LLM_CACHE_PATH = INTERIM_DIR / "llm_arbiter_cache_v2.json"
PROMPT_VERSION = "v4_score_blend"
CACHE_SCHEMA_VERSION = 2
LLM_BLEND_WEIGHT = 0.65
LLM_TARGET_VALID = 24

_CAUSAL_CHAINS = {
    "RC_WEAK_SIGNAL":       "Weak Wi-Fi radio -> throughput loss -> link switch or HO may recover.",
    "RC_SINR_DEGRADED":     "Interference rise -> SINR drop -> BLER growth -> MCS cap or channel change.",
    "RC_HO_FAILURE":        "Broken handover -> mobility drops -> neighbour-list reconfig needed.",
    "RC_PRB_CONGESTION":    "Peak demand -> PRB exhaustion -> throttle bulk or defer off-peak.",
    "RC_TRANSPORT_DELAY":   "Queue build-up -> latency rise -> AQM (CoDel) or path reroute.",
    "RC_CQI_MISMATCH":      "CQI misreport -> wrong MCS selected -> cap MCS or change channel.",
    "RC_COVERAGE_HOLE":     "Location out of coverage -> links degraded -> link swap or escalate.",
    "RC_CAPACITY_OVERLOAD": "Cell saturated -> many UEs competing -> load balance or defer.",
    "RC_NONE":              "No problem detected; no corrective action required.",
}

def _json_digest(obj: Any, n: int = 16) -> str:
    """Return a stable short SHA-256 digest for a JSON-serializable object."""
    payload = _js.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)
    return _hl.sha256(payload.encode("utf-8")).hexdigest()[:n]

ACTION_CATALOG_DIGEST = _json_digest({"actions": ACTIONS, "rc_to_actions": RC_TO_ACTIONS}, n=20)

def _state_summary(state: dict) -> dict:
    """Compress one episode state into the KPI fields shown to the LLM."""
    def rnd(v, d=1):
        """Round a scalar for compact prompt display while preserving missing values."""
        if v is None or (isinstance(v, float) and np.isnan(v)): return None
        try: return round(float(v), d)
        except Exception: return v
    return {
        "latency_ms": rnd(state.get("latency_ms"), 0),
        "jitter_ms": rnd(state.get("jitter_ms"), 0),
        "packet_loss_pct": rnd(state.get("packet_loss_pct"), 1),
        "throughput_mbps": rnd(state.get("throughput_mbps"), 1),
        "rssi_dbm": rnd(state.get("rssi_dbm"), 0),
        "sinr_db": rnd(state.get("sinr_db"), 0),
        "bler_proxy_pct": rnd(state.get("bler_proxy_pct"), 1),
        "bandwidth_util_pct": rnd(state.get("bandwidth_util_pct"), 0),
        "queue_length": rnd(state.get("queue_length"), 0),
        "active_connections": rnd(state.get("active_connections"), 0),
        "ho_success_rate_pct": rnd(state.get("ho_success_rate_pct"), 0),
        "traffic_type": state.get("traffic_type"),
        "is_peak_hour": bool(state.get("is_peak_hour", False)),
        "signal_dominant_link": state.get("signal_dominant_link"),
    }

def _build_prompt(rc, conf, state_summary, shortlist, ucb_scores) -> str:
    """Build the constrained JSON-only LLM arbitration prompt."""
    chain = _CAUSAL_CHAINS.get(rc, "Unspecified.")
    s = state_summary; ev = []
    if s.get("rssi_dbm") is not None and s["rssi_dbm"] <= -100:
        ev.append(f"RSSI {s['rssi_dbm']} dBm is below -100 dBm.")
    if s.get("sinr_db") is not None and s["sinr_db"] < 5:
        ev.append(f"SINR {s['sinr_db']} dB indicates interference.")
    if s.get("latency_ms") is not None and s["latency_ms"] >= 200:
        ev.append(f"Latency {s['latency_ms']} ms exceeds 200 ms.")
    if s.get("packet_loss_pct") is not None and s["packet_loss_pct"] >= 5:
        ev.append(f"Packet loss {s['packet_loss_pct']}% is abnormal.")
    if s.get("bandwidth_util_pct") is not None and s["bandwidth_util_pct"] >= 80:
        ev.append(f"Bandwidth utilization {s['bandwidth_util_pct']}% is near saturation.")
    ev_block = "\n".join(f"  - {e}" for e in ev) if ev else "  - No strong threshold breach."
    shortlist_block = "\n".join(
        f"  {i+1}. {a} (bandit_UCB={ucb_scores.get(a, 0.0):.3f}) -- {ACTION_BY_CODE[a]['mechanism']}"
        for i, a in enumerate(shortlist)
    )
    return "\n".join([
        "You are a telecom RAN/SRE decision arbiter. Score the shortlisted safe actions using the causal diagnosis and KPI evidence.",
        f"Root cause: {rc} (diagnostic confidence {conf:.2f}).",
        f"Causal chain: {chain}",
        f"Evidence:\n{ev_block}",
        f"KPI snapshot: {_js.dumps(s, sort_keys=True)}",
        "",
        "Shortlisted actions:",
        shortlist_block,
        "",
        "Return ONLY valid JSON with this schema:",
        '{"chosen":"<one shortlisted action>","confidence":0.0,"action_scores":{"<action>":0.0},"reasoning":"one or two sentences"}',
        "Scores must be calibrated from 0.0 to 1.0 and include every shortlisted action.",
    ])

def _cache_key_for_prompt(ep, prompt: str) -> str:
    """Return the prompt-keyed cache key for one episode arbitration call."""
    payload = {
        "schema": CACHE_SCHEMA_VERSION,
        "episode_id": ep.episode_id,
        "rc": ep.primary_rc,
        "model": LLM_MODEL,
        "prompt_version": PROMPT_VERSION,
        "prompt_hash": _hl.sha256(prompt.encode("utf-8")).hexdigest(),
        "action_catalog": ACTION_CATALOG_DIGEST,
    }
    return _json_digest(payload, n=24)

def _call_ollama(prompt: str) -> str | None:
    """Call the local Ollama JSON-generation endpoint and return raw text."""
    try:
        r = requests.post(OLLAMA_URL, json={"model": LLM_MODEL, "prompt": prompt,
                          "stream": False, "format": "json",
                          "options": {"temperature": 0.0, "num_predict": 350}},
                          timeout=LLM_TIMEOUT_S)
        r.raise_for_status()
        return r.json().get("response", "")
    except Exception as exc:
        print(f"  Ollama error: {type(exc).__name__}: {exc}")
        return None

def _parse_response(text, shortlist):
    """Parse and validate one LLM JSON response against the safe shortlist."""
    if not text:
        return {"chosen": shortlist[0], "confidence": 0.0, "action_scores": {},
                "reasoning": "LLM backend unavailable; UCB fallback used.", "ok": False}
    try:
        obj = _js.loads(text)
    except Exception:
        return {"chosen": shortlist[0], "confidence": 0.0, "action_scores": {},
                "reasoning": f"JSON parse error; raw={text[:80]}", "ok": False}
    chosen = str(obj.get("chosen", "")).strip()
    if chosen not in shortlist:
        return {"chosen": shortlist[0], "confidence": 0.0, "action_scores": {},
                "reasoning": f"Chosen action '{chosen}' was not shortlisted; UCB fallback used.", "ok": False}
    raw_scores = obj.get("action_scores", {}) or {}
    scores = {}
    for a in shortlist:
        try:
            scores[a] = float(np.clip(float(raw_scores.get(a, 1.0 if a == chosen else 0.0)), 0.0, 1.0))
        except Exception:
            scores[a] = 1.0 if a == chosen else 0.0
    try:
        conf = float(np.clip(float(obj.get("confidence", max(scores.values()))), 0.0, 1.0))
    except Exception:
        conf = max(scores.values())
    return {"chosen": chosen, "confidence": conf, "action_scores": scores,
            "reasoning": str(obj.get("reasoning", "")).strip(), "ok": True}

print(f"M7 LLM backend : {LLM_MODEL} @ {OLLAMA_URL}")
print(f"Cache path     : {LLM_CACHE_PATH}")
print(f"Action catalogue digest: {ACTION_CATALOG_DIGEST}")

def validate_m7_prompt_contract() -> pd.DataFrame:
    """Validate the prompt, cache, blend, and fallback contracts for M7."""
    sample_ep = test_eps[0]
    sample_bandit = LinUCBDispatcher(RC_TO_ACTIONS, D, alpha=0.4, lam=3.0, beta0=2.0,
                                     rng=np.random.default_rng(RANDOM_STATE + 1700))
    run_linucb_episode(sample_bandit, _TRAINVAL, train_stats, update=True)
    _, ucb_scores, shortlist, prompt, key = _shortlist_for_episode(sample_bandit, sample_ep, train_stats) if "_shortlist_for_episode" in globals() else (None, {}, sample_ep.allowed_actions[:1], "", "")
    rows = [
        {"check": "llm_weight_bounded", "passed": 0.0 <= LLM_BLEND_WEIGHT <= 1.0,
         "detail": f"weight={LLM_BLEND_WEIGHT:.2f}"},
        {"check": "target_valid_not_above_test_size", "passed": 0 <= LLM_TARGET_VALID <= len(test_eps),
         "detail": f"target={LLM_TARGET_VALID}, test={len(test_eps)}"},
        {"check": "cache_schema_positive", "passed": CACHE_SCHEMA_VERSION > 0,
         "detail": f"schema={CACHE_SCHEMA_VERSION}"},
        {"check": "action_catalog_digest_present", "passed": bool(ACTION_CATALOG_DIGEST),
         "detail": ACTION_CATALOG_DIGEST},
        {"check": "prompt_requests_json_reasoning", "passed": ("Return ONLY valid JSON" in prompt and "reasoning" in prompt) if prompt else True,
         "detail": "prompt includes JSON schema and reasoning field"},
    ]
    return pd.DataFrame(rows)


M7 LLM backend : qwen2.5:3b @ http://localhost:11434/api/generate
Cache path     : C:\Users\amani\OneDrive\Desktop\pi - Copy - Copy - Copy (2)\data\interim\llm_arbiter_cache_v2.json
Action catalogue digest: 579c11c1925410373153


**Interpretation:** M7 uses the LLM as a constrained arbiter over a LinUCB shortlist, not as an unconstrained controller. The prompt exposes the diagnosed root cause, confidence, causal chain, KPI snapshot, and at most three safe actions. The model must return JSON with `chosen`, calibrated `action_scores`, `confidence`, and `reasoning`. The cache key includes the prompt hash, model name, prompt version, schema version, and action-catalog digest, so stale reasoning cannot be silently reused after the action catalogue or prompt changes.

In [20]:
def _shortlist_for_episode(bandit: LinUCBDispatcher, ep: Episode, stats: dict) -> tuple[np.ndarray, dict, list, str, str]:
    """Build the encoded state, UCB scores, safe shortlist, prompt, and cache key."""
    x = _encode_state({**ep.state, "__rc__": ep.primary_rc}, stats)
    ucb_scores = {action: bandit.ucb(ep.primary_rc, action, x) for action in ep.allowed_actions}
    shortlist = sorted(ep.allowed_actions, key=lambda action: -ucb_scores[action])[:min(3, len(ep.allowed_actions))]
    prompt = _build_prompt(ep.primary_rc, ep.rc_confidence, _state_summary(ep.state), shortlist, ucb_scores)
    key = _cache_key_for_prompt(ep, prompt)
    return x, ucb_scores, shortlist, prompt, key

def _load_llm_cache(cache_path=LLM_CACHE_PATH) -> dict:
    """Load the prompt-keyed LLM arbiter cache if it exists."""
    if not cache_path.exists():
        return {}
    try:
        raw = _js.loads(cache_path.read_text(encoding="utf-8"))
        return raw if isinstance(raw, dict) else {}
    except Exception:
        return {}

def _cache_entry_valid(entry: dict, prompt: str) -> bool:
    """Return whether a cache entry belongs to the current prompt/model contract."""
    return (
        entry.get("schema") == CACHE_SCHEMA_VERSION
        and entry.get("model") == LLM_MODEL
        and entry.get("prompt_version") == PROMPT_VERSION
        and entry.get("action_catalog") == ACTION_CATALOG_DIGEST
        and entry.get("prompt_hash") == _hl.sha256(prompt.encode("utf-8")).hexdigest()
    )

def precompute_llm_cache(bandit: LinUCBDispatcher, episodes: list[Episode],
                         stats: dict, cache_path=LLM_CACHE_PATH) -> dict:
    """Precompute or reuse prompt-keyed LLM/fallback entries for current test prompts."""
    cache = _load_llm_cache(cache_path)
    n_pre = n_new = n_ok = n_fail = 0
    backend_available: bool | None = None
    t0 = time.time()
    for ep in episodes:
        _, ucb_scores, shortlist, prompt, key = _shortlist_for_episode(bandit, ep, stats)
        entry = cache.get(key)
        if isinstance(entry, dict) and _cache_entry_valid(entry, prompt):
            n_pre += 1
            if entry.get("ok"):
                n_ok += 1
            else:
                n_fail += 1
            continue
        if n_ok >= LLM_TARGET_VALID:
            parsed = {"chosen": shortlist[0], "confidence": 0.0, "action_scores": {},
                      "reasoning": "LLM target coverage reached; UCB fallback used for non-queried cases.", "ok": False}
            raw = ""
        elif backend_available is False:
            parsed = {"chosen": shortlist[0], "confidence": 0.0, "action_scores": {},
                      "reasoning": "LLM backend unavailable during this reproducible run; UCB fallback used.", "ok": False}
            raw = ""
        else:
            raw = _call_ollama(prompt)
            parsed = _parse_response(raw, shortlist)
            if backend_available is None:
                backend_available = bool(parsed["ok"])
        entry = {
            **parsed,
            "schema": CACHE_SCHEMA_VERSION,
            "model": LLM_MODEL,
            "prompt_version": PROMPT_VERSION,
            "action_catalog": ACTION_CATALOG_DIGEST,
            "prompt_hash": _hl.sha256(prompt.encode("utf-8")).hexdigest(),
            "shortlist": shortlist,
            "ucb_scores": {action: round(float(ucb_scores[action]), 6) for action in ep.allowed_actions},
            "raw": raw or "",
            "created_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        }
        cache[key] = entry
        n_new += 1
        if entry.get("ok"):
            n_ok += 1
        else:
            n_fail += 1
        if n_new % 20 == 0:
            cache_path.write_text(_js.dumps(cache, indent=2), encoding="utf-8")
            print(f"  ... {n_new} new, valid={n_ok}, fallback={n_fail}, {time.time()-t0:.0f}s")
    cache_path.write_text(_js.dumps(cache, indent=2), encoding="utf-8")
    register_hash("llm_cache:v2", cache_path)
    print(f"LLM cache for current prompts: {n_pre} reused + {n_new} new; valid={n_ok}, fallback={n_fail}")
    return cache

def _choose_m7_action(shortlist: list, ucb_scores: dict, entry: dict,
                      llm_weight: float = LLM_BLEND_WEIGHT) -> tuple[str, dict]:
    """Blend normalized UCB scores with valid LLM scores or fall back to UCB."""
    ucb_argmax = max(shortlist, key=lambda action: ucb_scores[action])
    if not entry.get("ok"):
        return ucb_argmax, {"source": "ucb_fallback", "ucb_argmax": ucb_argmax,
                            "llm_chosen": None, "override": False,
                            "reasoning": entry.get("reasoning", "No valid LLM entry; UCB fallback used.")}
    vals = np.array([ucb_scores[action] for action in shortlist], dtype=float)
    denom = vals.max() - vals.min()
    ucb_norm = {action: (0.5 if denom < 1e-12 else float((ucb_scores[action] - vals.min()) / denom)) for action in shortlist}
    llm_scores = entry.get("action_scores", {}) or {}
    chosen_by_llm = entry.get("chosen") if entry.get("chosen") in shortlist else ucb_argmax
    blended = {}
    for action in shortlist:
        llm_score = float(np.clip(llm_scores.get(action, 1.0 if action == chosen_by_llm else 0.0), 0.0, 1.0))
        blended[action] = (1.0 - llm_weight) * ucb_norm[action] + llm_weight * llm_score
    action = max(shortlist, key=lambda candidate: blended[candidate])
    return action, {"source": "llm_blend", "ucb_argmax": ucb_argmax,
                    "llm_chosen": chosen_by_llm, "override": action != ucb_argmax,
                    "blended_scores": blended, "ucb_norm": ucb_norm,
                    "llm_scores": {a: float(np.clip(llm_scores.get(a, np.nan), 0.0, 1.0)) for a in shortlist if a in llm_scores},
                    "reasoning": str(entry.get("reasoning", "")).strip(),
                    "llm_confidence": float(entry.get("confidence", np.nan))}

def run_m7_decisions(bandit: LinUCBDispatcher, episodes: list[Episode], cache: dict,
                     stats: dict, update: bool = False) -> dict:
    """Run M7 and collect per-episode arbitration metadata and reasoning."""
    rewards, regrets, rows = [], [], []
    for step, ep in enumerate(episodes):
        x, ucb_scores, shortlist, prompt, key = _shortlist_for_episode(bandit, ep, stats)
        entry = cache.get(key, {})
        chosen, meta = _choose_m7_action(shortlist, ucb_scores, entry)
        reward, _ = oracle_reward(ep.primary_rc, chosen, ep.state)
        best = max(oracle_reward(ep.primary_rc, candidate, ep.state)[0] for candidate in ep.allowed_actions)
        regret = best - reward
        rewards.append(reward)
        regrets.append(regret)
        rows.append({
            "step": step,
            "episode_id": ep.episode_id,
            "primary_rc": ep.primary_rc,
            "rc_confidence": ep.rc_confidence,
            "shortlist": ", ".join(shortlist),
            "chosen_action": chosen,
            "ucb_argmax": meta.get("ucb_argmax"),
            "llm_chosen": meta.get("llm_chosen"),
            "source": meta.get("source"),
            "override": bool(meta.get("override", False)),
            "reward": reward,
            "best_reward": best,
            "regret": regret,
            "llm_ok": bool(entry.get("ok", False)),
            "llm_confidence": meta.get("llm_confidence", np.nan),
            "reasoning": meta.get("reasoning", ""),
            "prompt_key": key,
            "is_synthetic": ep.is_synthetic,
        })
        if update:
            bandit.update(ep.primary_rc, chosen, reward, x)
    decisions = pd.DataFrame(rows)
    return {"mean_reward": float(np.mean(rewards)) if rewards else float("nan"),
            "std_reward": float(np.std(rewards, ddof=1)) if len(rewards) > 1 else 0.0,
            "mean_regret": float(np.mean(regrets)) if regrets else float("nan"),
            "n_llm_used": int((decisions["source"] == "llm_blend").sum()) if not decisions.empty else 0,
            "n_ucb_fallback": int((decisions["source"] == "ucb_fallback").sum()) if not decisions.empty else 0,
            "n_llm_overrides": int(decisions["override"].sum()) if not decisions.empty else 0,
            "per_step_regret": regrets,
            "decisions": decisions}

def evaluate_m7_frozen(bandit, episodes, cache, stats) -> dict:
    """Evaluate frozen M7 on episodes without online updates."""
    result = run_m7_decisions(bandit, episodes, cache, stats, update=False)
    return {k: v for k, v in result.items() if k != "decisions"}

def run_m7_online_episode(bandit, episodes, cache, stats, update: bool = True) -> dict:
    """Evaluate M7 on a chronological stream and optionally update LinUCB arms."""
    result = run_m7_decisions(bandit, episodes, cache, stats, update=update)
    return {k: v for k, v in result.items() if k != "decisions"}

def build_m7_cache_audit(bandit: LinUCBDispatcher, episodes: list[Episode], cache: dict, stats: dict) -> pd.DataFrame:
    """Build one audit row per current test prompt/cache entry."""
    rows = []
    for ep in episodes:
        _, ucb_scores, shortlist, prompt, key = _shortlist_for_episode(bandit, ep, stats)
        entry = cache.get(key, {})
        rows.append({
            "episode_id": ep.episode_id,
            "primary_rc": ep.primary_rc,
            "shortlist_size": len(shortlist),
            "shortlist": ", ".join(shortlist),
            "cache_valid": isinstance(entry, dict) and _cache_entry_valid(entry, prompt),
            "llm_ok": bool(entry.get("ok", False)),
            "chosen": entry.get("chosen", ""),
            "reasoning_present": bool(str(entry.get("reasoning", "")).strip()),
            "reasoning": str(entry.get("reasoning", "")).strip(),
        })
    return pd.DataFrame(rows)

m7_prompt_contract = validate_m7_prompt_contract()
if not m7_prompt_contract["passed"].all():
    raise AssertionError(m7_prompt_contract.loc[~m7_prompt_contract["passed"]].to_string(index=False))

_arb_rng = np.random.default_rng(RANDOM_STATE + 100)
_m6_for_arbiter = LinUCBDispatcher(RC_TO_ACTIONS, D, alpha=0.4, lam=3.0, beta0=2.0, rng=_arb_rng)
run_linucb_episode(_m6_for_arbiter, _TRAINVAL, train_stats, update=True)
print("M7 prompt/cache contract:")
print(m7_prompt_contract.to_string(index=False))
print("\nArbiter bandit trained on train+val pool (frozen).")

print("\nPrecomputing LLM arbiter scores for real test episodes ...")
arbiter_cache = precompute_llm_cache(_m6_for_arbiter, test_eps, train_stats)
m7_cache_audit = build_m7_cache_audit(_m6_for_arbiter, test_eps, arbiter_cache, train_stats)
n_ok = int(m7_cache_audit["llm_ok"].sum())
n_fail = len(m7_cache_audit) - n_ok
m7_cache_by_rc = (
    m7_cache_audit.groupby("primary_rc", dropna=False)
    .agg(episodes=("episode_id", "size"),
         llm_valid=("llm_ok", "sum"),
         fallback=("llm_ok", lambda s: int((~s).sum())),
         reasoning_present=("reasoning_present", "sum"),
         mean_shortlist_size=("shortlist_size", "mean"))
    .reset_index()
)
print(f"Current test LLM coverage: {n_ok}/{len(test_eps)} valid ({n_ok/max(1,len(test_eps)):.1%}); fallback={n_fail}")
print("\nM7 cache coverage by root cause:")
print(m7_cache_by_rc.round(4).to_string(index=False))

fig_m7_cache = px.bar(
    m7_cache_by_rc, x="primary_rc", y=["llm_valid", "fallback"], barmode="stack",
    title="M7 LLM Cache Coverage by Root Cause",
    labels={"primary_rc": "Root cause", "value": "Episodes", "variable": "Cache source"},
)
fig_m7_cache.update_xaxes(tickangle=-35)
fig_m7_cache.update_layout(width=950, height=430)
save_fig(fig_m7_cache, "fig04n_m7_cache_coverage_by_rc"); fig_m7_cache.show()

fig_m7_shortlist = px.histogram(
    m7_cache_audit, x="shortlist_size", color="llm_ok", barmode="group",
    title="M7 Shortlist Size and Valid LLM Coverage",
    labels={"shortlist_size": "Shortlist size", "count": "Episodes", "llm_ok": "Valid LLM"},
)
fig_m7_shortlist.update_layout(width=760, height=420)
save_fig(fig_m7_shortlist, "fig04o_m7_shortlist_coverage"); fig_m7_shortlist.show()


M7 prompt/cache contract:
                           check  passed                                          detail
              llm_weight_bounded    True                                     weight=0.65
target_valid_not_above_test_size    True                             target=24, test=111
           cache_schema_positive    True                                        schema=2
   action_catalog_digest_present    True                            579c11c1925410373153
  prompt_requests_json_reasoning    True prompt includes JSON schema and reasoning field

Arbiter bandit trained on train+val pool (frozen).

Precomputing LLM arbiter scores for real test episodes ...
LLM cache for current prompts: 111 reused + 0 new; valid=24, fallback=87
Current test LLM coverage: 24/111 valid (21.6%); fallback=87

M7 cache coverage by root cause:
          primary_rc  episodes  llm_valid  fallback  reasoning_present  mean_shortlist_size
RC_CAPACITY_OVERLOAD         4          1         3                 

**Interpretation:** We audit the M7 prompt and cache before using the LLM scores. The contract checks all pass: the blend weight is 0.65, the target valid coverage is 24 out of 111 test episodes, the cache schema is version 2, the action-catalog digest is present, and the prompt explicitly requests JSON with a reasoning field. The cache is fully reproducible in this run: it reuses 111 current prompt entries, creates 0 new entries, and contains 24 valid LLM responses plus 87 explicit fallbacks.

The cache-coverage figure shows where the LLM actually has valid frozen-arbiter reasoning. `RC_NONE` receives 17 valid LLM entries and 17 fallbacks, `RC_TRANSPORT_DELAY` receives 5 valid entries and 51 fallbacks, `RC_CAPACITY_OVERLOAD` receives 1 valid entry and 3 fallbacks, and `RC_COVERAGE_HOLE` receives 1 valid entry and 8 fallbacks. `RC_SINR_DEGRADED` and `RC_WEAK_SIGNAL` have 0 valid LLM entries in this run. We interpret that coverage as partial by design, not as full LLM control. The shortlist figure also confirms that actionable RCs usually expose 3 shortlisted actions, while `RC_NONE` exposes only `ACT_NO_OP`. This keeps the LLM bounded inside the action mask.

In [21]:
def seed_sweep_m7(n_seeds: int = 10, alpha: float = 0.4, lam: float = 3.0, beta0: float = 2.0) -> pd.DataFrame:
    """Train M7 across seeds; LLM score blending is evaluated with prompt-keyed cache."""
    rows = []
    for seed in range(n_seeds):
        rng_frozen = np.random.default_rng(RANDOM_STATE + 100 + seed)
        frozen_bandit = LinUCBDispatcher(RC_TO_ACTIONS, D, alpha=alpha, lam=lam, beta0=beta0, rng=rng_frozen)
        train_result = run_linucb_episode(frozen_bandit, _TRAINVAL, train_stats, update=True)
        frozen_result = evaluate_m7_frozen(frozen_bandit, test_eps, arbiter_cache, train_stats)

        rng_online = np.random.default_rng(RANDOM_STATE + 100 + seed)
        online_bandit = LinUCBDispatcher(RC_TO_ACTIONS, D, alpha=alpha, lam=lam, beta0=beta0, rng=rng_online)
        run_linucb_episode(online_bandit, _TRAINVAL, train_stats, update=True)
        online_result = run_m7_online_episode(online_bandit, test_eps, arbiter_cache, train_stats, update=True)
        rows.append({
            "seed": seed,
            "train_reward": train_result["mean_reward"],
            "train_regret": train_result["mean_regret"],
            "test_frozen_reward": frozen_result["mean_reward"],
            "test_frozen_regret": frozen_result["mean_regret"],
            "test_online_reward": online_result["mean_reward"],
            "test_online_regret": online_result["mean_regret"],
            "n_llm_used": frozen_result["n_llm_used"],
            "n_ucb_fallback": frozen_result["n_ucb_fallback"],
            "n_llm_overrides": frozen_result["n_llm_overrides"],
        })
    return pd.DataFrame(rows)

def representative_m7_online_decisions(seed: int = RANDOM_STATE + 100) -> pd.DataFrame:
    """Return one representative online M7 decision trace with reasoning."""
    bandit = LinUCBDispatcher(RC_TO_ACTIONS, D, alpha=0.4, lam=3.0, beta0=2.0,
                              rng=np.random.default_rng(seed))
    run_linucb_episode(bandit, _TRAINVAL, train_stats, update=True)
    result = run_m7_decisions(bandit, test_eps, arbiter_cache, train_stats, update=True)
    decisions = result["decisions"].copy()
    decisions["cumulative_regret"] = decisions["regret"].cumsum()
    return decisions

print("Running M7 seed sweep (10 seeds) ...")
m7_sweep = seed_sweep_m7(n_seeds=10)
m7_online = m7_sweep["test_online_reward"]
m7_decisions = representative_m7_online_decisions()
m7_source_summary = (
    m7_decisions.groupby("source", dropna=False)
    .agg(decisions=("episode_id", "size"),
         mean_reward=("reward", "mean"),
         mean_regret=("regret", "mean"),
         overrides=("override", "sum"))
    .reset_index()
)
m7_rc_summary = (
    m7_decisions.groupby("primary_rc", dropna=False)
    .agg(episodes=("episode_id", "nunique"),
         llm_used=("source", lambda s: int((s == "llm_blend").sum())),
         fallbacks=("source", lambda s: int((s == "ucb_fallback").sum())),
         overrides=("override", "sum"),
         mean_reward=("reward", "mean"),
         mean_regret=("regret", "mean"))
    .reset_index()
)
m7_override_examples = m7_decisions[m7_decisions["override"]].copy()
m7_reasoning_examples = (
    m7_decisions[m7_decisions["llm_ok"]]
    .sort_values(["override", "regret", "primary_rc"], ascending=[False, False, True])
    .head(8)
    [["episode_id", "primary_rc", "shortlist", "ucb_argmax", "llm_chosen", "chosen_action", "reward", "regret", "llm_confidence", "reasoning"]]
)

print("M7 seed sweep summary:")
print(m7_sweep[["train_reward", "test_frozen_reward", "test_online_reward", "test_online_regret", "n_llm_used", "n_ucb_fallback", "n_llm_overrides"]].agg(["mean", "std", "min", "max"]).round(4).to_string())
print("\nM7 representative online source summary:")
print(m7_source_summary.round(4).to_string(index=False))
print("\nM7 representative online root-cause summary:")
print(m7_rc_summary.round(4).to_string(index=False))
print("\nM7 LLM reasoning examples:")
if m7_reasoning_examples.empty:
    print("No valid LLM reasoning examples available for current prompts.")
else:
    print(m7_reasoning_examples.round(4).to_string(index=False))
print(f"\nM7 online: mean={m7_online.mean():.4f}  std={m7_online.std():.4f}")
print(f"M7 frozen: mean={m7_sweep['test_frozen_reward'].mean():.4f}")
print(f"LLM score-blend coverage mean: {m7_sweep['n_llm_used'].mean():.1f}/{len(test_eps)}; overrides mean: {m7_sweep['n_llm_overrides'].mean():.1f}")

fig_m7_seed = px.box(
    m7_sweep.melt(id_vars="seed", value_vars=["test_frozen_reward", "test_online_reward"],
                  var_name="test_mode", value_name="reward"),
    x="test_mode", y="reward", color="test_mode",
    title="M7 Holdout Reward Stability Across 10 Seeds",
    labels={"test_mode": "Test mode", "reward": "Mean reward"},
)
fig_m7_seed.update_layout(width=760, height=430, showlegend=False)
save_fig(fig_m7_seed, "fig04p_m7_seed_reward_distribution"); fig_m7_seed.show()

fig_m7_source = px.bar(
    m7_source_summary, x="source", y="decisions", color="mean_regret", text="decisions",
    title="M7 Decision Source and Regret",
    labels={"source": "Decision source", "decisions": "Decisions", "mean_regret": "Mean regret"},
)
fig_m7_source.update_layout(width=760, height=430)
save_fig(fig_m7_source, "fig04q_m7_decision_source"); fig_m7_source.show()

fig_m7_rc = px.bar(
    m7_rc_summary, x="primary_rc", y=["llm_used", "fallbacks"], barmode="stack",
    title="M7 LLM Use and Fallbacks by Root Cause",
    labels={"primary_rc": "Root cause", "value": "Episodes", "variable": "Decision source"},
)
fig_m7_rc.update_xaxes(tickangle=-35)
fig_m7_rc.update_layout(width=950, height=430)
save_fig(fig_m7_rc, "fig04r_m7_source_by_rc"); fig_m7_rc.show()

fig_m7_regret = px.bar(
    m7_rc_summary, x="primary_rc", y="mean_regret", color="overrides", text="episodes",
    title="M7 Representative Online Holdout Regret by Root Cause",
    labels={"primary_rc": "Root cause", "mean_regret": "Mean regret", "overrides": "Overrides"},
)
fig_m7_regret.update_xaxes(tickangle=-35)
fig_m7_regret.update_layout(width=950, height=430)
save_fig(fig_m7_regret, "fig04s_m7_holdout_regret_by_rc"); fig_m7_regret.show()

if not m7_reasoning_examples.empty:
    table_df = m7_reasoning_examples.copy()
    table_df["reasoning"] = table_df["reasoning"].str.slice(0, 160)
    fig_m7_reasoning = go.Figure(data=[go.Table(
        header=dict(values=list(table_df.columns), fill_color="#f2f2f2", align="left"),
        cells=dict(values=[table_df[col] for col in table_df.columns], align="left", height=28),
    )])
    fig_m7_reasoning.update_layout(title_text="M7 LLM Reasoning Examples", width=1250, height=520)
    save_fig(fig_m7_reasoning, "fig04t_m7_reasoning_examples"); fig_m7_reasoning.show()


Running M7 seed sweep (10 seeds) ...
M7 seed sweep summary:
      train_reward  test_frozen_reward  test_online_reward  test_online_regret  n_llm_used  n_ucb_fallback  n_llm_overrides
mean        0.6831              0.5211              0.6135              0.0550     19.3000         91.7000           0.4000
std         0.0011              0.0234              0.0031              0.0031      2.4967          2.4967           0.8433
min         0.6807              0.4836              0.6081              0.0498     18.0000         87.0000           0.0000
max         0.6845              0.5546              0.6186              0.0604     24.0000         93.0000           2.0000

M7 representative online source summary:
      source  decisions  mean_reward  mean_regret  overrides
   llm_blend          4       0.4875       0.0750          1
ucb_fallback        107       0.6156       0.0568          0

M7 representative online root-cause summary:
          primary_rc  episodes  llm_used  fallbac

**Interpretation:** M7 slightly improves the frozen score over M6, but it does not improve the online score. Across 10 seeds, M7 has frozen reward 0.5211 versus M6's 0.5206, online reward 0.6135, online regret 0.0550, and online standard deviation 0.0031. The average frozen evaluation uses 19.3 LLM-blended decisions out of 111 and falls back to UCB 91.7 times, with only 0.4 overrides per run. That low override count is important: M7 is mostly LinUCB with occasional LLM arbitration, not an LLM-controlled optimizer.

The representative online source figure makes the same point more sharply. In that trace, only 4 decisions use valid LLM blending and 107 decisions fall back to UCB. The LLM-blended decisions average reward 0.4875 and regret 0.0750, while UCB fallback decisions average reward 0.6156 and regret 0.0568. The root-cause figure shows that the valid online LLM use is sparse: 1 decision each for `RC_CAPACITY_OVERLOAD`, `RC_COVERAGE_HOLE`, `RC_NONE`, and `RC_TRANSPORT_DELAY`, and 0 for `RC_SINR_DEGRADED` and `RC_WEAK_SIGNAL`. We therefore do not claim that M7 wins; we claim that it adds an auditable reasoning layer with limited coverage.

The reasoning table shows the LLM contribution directly. In one transport-delay case, the LLM explains that queue management addresses queue delay, but the blended policy still selects `ACT_PATH_REROUTE`, receiving reward 0.4000 and regret 0.1000. In a coverage-hole case, the LLM chooses `ACT_SWITCH_LINK_CELL` because RSSI is -101 dBm, but the final reward is only 0.2000 with regret 0.2000. The useful positive examples are also visible: for capacity overload, the LLM supports `ACT_LOAD_BALANCE` and gets 0 regret, and for `RC_NONE`, it correctly supports `ACT_NO_OP` with reward 0.9000. We read these examples as auditability evidence, not causal proof. The reasoning helps us inspect why an action was preferred, but the reward/regret columns decide whether that reasoning improved the benchmark.

<a id="sec-15"></a>

## 15 - M8: Offline Reward Ranker

M8 tests whether a supervised offline model can learn the reward-rule surface from historical decision episodes. We expand each train and validation episode into one row per allowed action, train only on those candidate rows, and evaluate the fitted ranker on the untouched chronological test split.

In [22]:
from sklearn.ensemble import HistGradientBoostingRegressor

ACTION_SCOPE_VOCAB = ["radio", "transport", "both"]

def action_feature_names() -> list[str]:
    """Return action-side feature names in the exact order emitted by `_action_features`."""
    return (
        [f"action::{action}" for action in ACTION_CODES]
        + [f"scope::{scope}" for scope in ACTION_SCOPE_VOCAB]
        + ["action::cost_weight", "action::reversible", "action::risky"]
    )

def ranker_feature_names() -> list[str]:
    """Return the full state-action feature names used by M8."""
    return encoder_feature_names() + action_feature_names() + ["diagnosis::canonical_prior"]

def _action_features(action: str) -> np.ndarray:
    """Encode one action into stable catalogue metadata used by the reward ranker."""
    if action not in ACTION_BY_CODE:
        raise KeyError(f"Unknown action code: {action}")
    meta = ACTION_BY_CODE[action]
    if meta["scope"] not in ACTION_SCOPE_VOCAB:
        raise ValueError(f"Action scope {meta['scope']} is not in ACTION_SCOPE_VOCAB")
    code_oh = [1.0 if action == candidate else 0.0 for candidate in ACTION_CODES]
    scope_oh = [1.0 if meta["scope"] == scope else 0.0 for scope in ACTION_SCOPE_VOCAB]
    flags = [
        float(meta["cost_weight"]),
        1.0 if meta["reversible"] else 0.0,
        1.0 if meta["risky"] else 0.0,
    ]
    return np.array(code_oh + scope_oh + flags, dtype=np.float64)

def _ranker_features(ep: Episode, action: str, stats: dict) -> np.ndarray:
    """Build one supervised feature vector for an episode-action candidate."""
    if action not in ep.allowed_actions:
        raise ValueError(f"{action} is not allowed for episode {ep.episode_id}")
    x_state = _encode_state({**ep.state, "__rc__": ep.primary_rc}, stats)
    prior = np.array([diagnosis_prior(ep.primary_rc, action)], dtype=np.float64)
    x = np.concatenate([x_state, _action_features(action), prior])
    expected = len(ranker_feature_names())
    if x.shape[0] != expected:
        raise AssertionError(f"Ranker feature dimension {x.shape[0]} != expected {expected}")
    return x

def make_ranker_frame(episodes: list[Episode], stats: dict) -> tuple[np.ndarray, np.ndarray, list[dict]]:
    """Expand episodes into one supervised row per allowed action."""
    X, y, rows = [], [], []
    for ep in episodes:
        if not ep.allowed_actions:
            raise ValueError(f"Episode {ep.episode_id} has no allowed actions")
        for action in ep.allowed_actions:
            reward, citation = oracle_reward(ep.primary_rc, action, ep.state)
            X.append(_ranker_features(ep, action, stats))
            y.append(reward)
            rows.append({
                "episode_id": ep.episode_id,
                "timestamp": ep.timestamp,
                "primary_rc": ep.primary_rc,
                "action": action,
                "oracle_reward": float(reward),
                "diagnosis_prior": diagnosis_prior(ep.primary_rc, action),
                "action_scope": ACTION_BY_CODE[action]["scope"],
                "action_cost": ACTION_BY_CODE[action]["cost_weight"],
                "action_risky": ACTION_BY_CODE[action]["risky"],
                "is_synthetic": ep.is_synthetic,
                "citation": citation,
            })
    if not X:
        return np.empty((0, len(ranker_feature_names())), dtype=np.float64), np.array([], dtype=np.float64), rows
    return np.vstack(X), np.asarray(y, dtype=np.float64), rows

def _best_action_for_episode(ep: Episode) -> tuple[str, float]:
    """Return the oracle-best allowed action and reward for one episode."""
    scored = [(action, oracle_reward(ep.primary_rc, action, ep.state)[0]) for action in ep.allowed_actions]
    return max(scored, key=lambda item: item[1])

class RewardRankerPolicy:
    """Offline supervised reward ranker that chooses the highest predicted action reward."""
    name = "M8 RewardRanker"

    def __init__(self, stats: dict, random_state: int = RANDOM_STATE):
        """Create a deterministic scikit-learn reward ranker over state-action rows."""
        self.stats = stats
        self.random_state = int(random_state)
        self.feature_names = ranker_feature_names()
        self.model = HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.05,
            max_iter=250,
            max_leaf_nodes=15,
            min_samples_leaf=12,
            l2_regularization=0.01,
            random_state=self.random_state,
        )
        self.is_fitted = False
        self.training_rows_ = 0
        self.training_reward_mean_ = np.nan

    def fit(self, episodes: list[Episode]) -> "RewardRankerPolicy":
        """Fit the ranker on expanded train-validation episode-action reward rows."""
        X, y, _ = make_ranker_frame(episodes, self.stats)
        if len(y) == 0:
            raise ValueError("Cannot fit RewardRankerPolicy on an empty episode list")
        self.model.fit(X, y)
        self.is_fitted = True
        self.training_rows_ = int(len(y))
        self.training_reward_mean_ = float(np.mean(y))
        return self

    def predict_action_rewards(self, ep: Episode) -> dict[str, float]:
        """Predict one reward per allowed action for an episode."""
        if not self.is_fitted:
            raise RuntimeError("RewardRankerPolicy must be fitted before prediction")
        X = np.vstack([_ranker_features(ep, action, self.stats) for action in ep.allowed_actions])
        preds = np.clip(self.model.predict(X), 0.0, 1.0)
        return {action: float(pred) for action, pred in zip(ep.allowed_actions, preds)}

    def select(self, ep: Episode) -> str:
        """Select the allowed action with the highest predicted reward."""
        preds = self.predict_action_rewards(ep)
        return max(ep.allowed_actions, key=lambda action: preds[action])

    def update(self, ep: Episode, action: str, reward: float) -> None:
        """Expose a no-op online update so M8 can share the policy interface."""
        return None

def validate_reward_ranker_contract(policy: RewardRankerPolicy, train_rows: pd.DataFrame) -> pd.DataFrame:
    """Validate the M8 feature contract, fitting state, action mask, and leakage boundary."""
    expected_pairs = {(rc, action) for rc, actions in RC_TO_ACTIONS.items() for action in actions}
    expanded_pairs = set(zip(train_rows["primary_rc"], train_rows["action"]))
    test_ids = {ep.episode_id for ep in test_eps}
    train_ids = set(train_rows["episode_id"])
    sample_ep = _TRAINVAL[0]
    sample_x = _ranker_features(sample_ep, sample_ep.allowed_actions[0], policy.stats)
    checks = [
        {"check": "feature_dimension_matches_names", "passed": sample_x.shape[0] == len(policy.feature_names),
         "detail": f"vector={sample_x.shape[0]}, names={len(policy.feature_names)}"},
        {"check": "state_encoder_dimension_reused", "passed": D == len(encoder_feature_names()),
         "detail": f"state_D={D}, state_names={len(encoder_feature_names())}"},
        {"check": "all_training_pairs_allowed_by_mask", "passed": expanded_pairs.issubset(expected_pairs),
         "detail": f"extra_pairs={len(expanded_pairs - expected_pairs)}"},
        {"check": "test_episode_ids_excluded_from_training", "passed": train_ids.isdisjoint(test_ids),
         "detail": f"overlap={len(train_ids & test_ids)}"},
        {"check": "reward_targets_bounded", "passed": train_rows["oracle_reward"].between(0.0, 1.0).all(),
         "detail": f"min={train_rows['oracle_reward'].min():.3f}, max={train_rows['oracle_reward'].max():.3f}"},
        {"check": "model_is_fitted", "passed": bool(policy.is_fitted),
         "detail": f"training_rows={policy.training_rows_}, n_iter={getattr(policy.model, 'n_iter_', 'NA')}"},
    ]
    return pd.DataFrame(checks)

def expanded_prediction_frame(policy: RewardRankerPolicy, episodes: list[Episode], split: str) -> pd.DataFrame:
    """Return expanded candidate rows with M8 predictions and absolute errors."""
    X, y, rows = make_ranker_frame(episodes, policy.stats)
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    frame["split"] = split
    frame["predicted_reward"] = np.clip(policy.model.predict(X), 0.0, 1.0)
    frame["prediction_error"] = frame["predicted_reward"] - frame["oracle_reward"]
    frame["abs_error"] = frame["prediction_error"].abs()
    return frame

def evaluate_reward_ranker(policy: RewardRankerPolicy, episodes: list[Episode], collect: bool = False) -> dict:
    """Evaluate the fitted M8 ranker on episodes without online updates."""
    rewards, regrets, chosen, decision_rows = [], [], [], []
    for step, ep in enumerate(episodes):
        preds = policy.predict_action_rewards(ep)
        action = max(ep.allowed_actions, key=lambda candidate: preds[candidate])
        reward, _ = oracle_reward(ep.primary_rc, action, ep.state)
        best_action, best_reward = _best_action_for_episode(ep)
        sorted_preds = sorted(preds.items(), key=lambda item: item[1], reverse=True)
        margin = sorted_preds[0][1] - sorted_preds[1][1] if len(sorted_preds) > 1 else sorted_preds[0][1]
        regret = float(best_reward - reward)
        rewards.append(float(reward))
        regrets.append(regret)
        chosen.append(action)
        if collect:
            decision_rows.append({
                "step": step,
                "episode_id": ep.episode_id,
                "timestamp": ep.timestamp,
                "primary_rc": ep.primary_rc,
                "chosen_action": action,
                "predicted_reward": float(preds[action]),
                "reward": float(reward),
                "best_action": best_action,
                "best_reward": float(best_reward),
                "regret": regret,
                "prediction_margin": float(margin),
                "allowed_action_count": len(ep.allowed_actions),
                "is_synthetic": ep.is_synthetic,
            })
    out = {
        "mean_reward": float(np.mean(rewards)) if rewards else float("nan"),
        "std_reward": float(np.std(rewards, ddof=1)) if len(rewards) > 1 else 0.0,
        "mean_regret": float(np.mean(regrets)) if regrets else float("nan"),
        "action_counts": dict(pd.Series(chosen).value_counts()),
        "_rewards": rewards,
        "per_step_regret": regrets,
    }
    if collect:
        out["decisions"] = pd.DataFrame(decision_rows)
    return out

def seed_sweep_m8(n_seeds: int = 20) -> pd.DataFrame:
    """Train and evaluate M8 across seeds for comparability with other sweeps."""
    rows = []
    for seed in range(n_seeds):
        policy = RewardRankerPolicy(train_stats, random_state=RANDOM_STATE + 800 + seed).fit(_TRAINVAL)
        frozen = evaluate_reward_ranker(policy, test_eps)
        rows.append({
            "seed": seed,
            "train_rows": policy.training_rows_,
            "train_reward_target_mean": policy.training_reward_mean_,
            "test_frozen_reward": frozen["mean_reward"],
            "test_frozen_regret": frozen["mean_regret"],
            "test_online_reward": frozen["mean_reward"],
            "test_online_regret": frozen["mean_regret"],
        })
    return pd.DataFrame(rows)

def ranker_metric_frame(frame: pd.DataFrame, split: str) -> pd.DataFrame:
    """Compute calibration metrics for expanded state-action predictions."""
    if frame.empty:
        return pd.DataFrame([{"split": split, "candidate_rows": 0, "mae": np.nan, "rmse": np.nan, "bias": np.nan}])
    err = frame["prediction_error"].to_numpy(dtype=float)
    return pd.DataFrame([{
        "split": split,
        "candidate_rows": int(len(frame)),
        "mae": float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "bias": float(np.mean(err)),
        "pred_min": float(frame["predicted_reward"].min()),
        "pred_max": float(frame["predicted_reward"].max()),
    }])

trainval_split_lookup = {ep.episode_id: "train" for ep in train_eps}
trainval_split_lookup.update({ep.episode_id: "validation" for ep in val_eps})
X_trainval, y_trainval, trainval_ranker_rows = make_ranker_frame(_TRAINVAL, train_stats)
m8_train_frame = pd.DataFrame(trainval_ranker_rows)
m8_train_frame["split"] = m8_train_frame["episode_id"].map(trainval_split_lookup).fillna("unknown")
m8_policy = RewardRankerPolicy(train_stats, random_state=RANDOM_STATE + 800).fit(_TRAINVAL)
m8_contract_audit = validate_reward_ranker_contract(m8_policy, m8_train_frame)
m8_train_pred_frame = expanded_prediction_frame(m8_policy, _TRAINVAL, "train_validation")
m8_test_pred_frame = expanded_prediction_frame(m8_policy, test_eps, "test")
m8_prediction_metrics = pd.concat([
    ranker_metric_frame(m8_train_pred_frame, "train_validation"),
    ranker_metric_frame(m8_test_pred_frame, "test"),
], ignore_index=True)

print("M8 RewardRanker contract audit:")
print(m8_contract_audit.to_string(index=False))
assert m8_contract_audit["passed"].all()

print("\nM8 expanded candidate-row audit:")
m8_frame_audit = pd.DataFrame([
    {"split": "train_validation", "episodes": len(_TRAINVAL), "candidate_rows": len(m8_train_frame),
     "root_causes": m8_train_frame["primary_rc"].nunique(), "actions": m8_train_frame["action"].nunique(),
     "synthetic_rows": int(m8_train_frame["is_synthetic"].sum())},
    {"split": "test", "episodes": len(test_eps), "candidate_rows": len(m8_test_pred_frame),
     "root_causes": m8_test_pred_frame["primary_rc"].nunique(), "actions": m8_test_pred_frame["action"].nunique(),
     "synthetic_rows": int(m8_test_pred_frame["is_synthetic"].sum())},
])
print(m8_frame_audit.to_string(index=False))

print("\nM8 candidate reward summary:")
m8_reward_summary = (
    pd.concat([m8_train_pred_frame, m8_test_pred_frame], ignore_index=True)
    .groupby("split")
    .agg(candidate_rows=("oracle_reward", "size"),
         mean_oracle_reward=("oracle_reward", "mean"),
         min_oracle_reward=("oracle_reward", "min"),
         max_oracle_reward=("oracle_reward", "max"),
         mean_predicted_reward=("predicted_reward", "mean"))
    .reset_index()
    .round(4)
)
print(m8_reward_summary.to_string(index=False))

print("\nM8 prediction calibration metrics:")
print(m8_prediction_metrics.round(4).to_string(index=False))

print("\nRunning M8 RewardRanker seed sweep (20 seeds) ...")
m8_sweep = seed_sweep_m8(n_seeds=20)
m8_online = m8_sweep["test_online_reward"]
m8_decisions = evaluate_reward_ranker(m8_policy, test_eps, collect=True)["decisions"]
m8_decisions["cumulative_regret"] = m8_decisions["regret"].cumsum()
m8_rc_summary = (
    m8_decisions.groupby("primary_rc", dropna=False)
    .agg(episodes=("episode_id", "nunique"),
         mean_reward=("reward", "mean"),
         mean_regret=("regret", "mean"),
         oracle_hits=("regret", lambda s: int((np.asarray(s) <= 1e-12).sum())),
         mean_predicted_reward=("predicted_reward", "mean"),
         mean_prediction_margin=("prediction_margin", "mean"))
    .reset_index()
    .sort_values(["mean_regret", "episodes"], ascending=[False, False])
)
m8_action_heat = pd.crosstab(m8_decisions["primary_rc"], m8_decisions["chosen_action"])
m8_error_by_rc = (
    m8_test_pred_frame.groupby("primary_rc", dropna=False)
    .agg(candidate_rows=("episode_id", "size"),
         mae=("abs_error", "mean"),
         bias=("prediction_error", "mean"))
    .reset_index()
    .sort_values("mae", ascending=False)
)
m8_trace_examples = (
    m8_decisions.sort_values(["regret", "prediction_margin", "primary_rc"], ascending=[False, False, True])
    .head(8)
    [["episode_id", "primary_rc", "chosen_action", "best_action", "predicted_reward", "reward", "best_reward", "regret", "prediction_margin"]]
)

print(f"M8 RewardRanker frozen: mean={m8_sweep['test_frozen_reward'].mean():.4f}  std={m8_sweep['test_frozen_reward'].std():.4f}")
print(f"M8 RewardRanker regret: mean={m8_sweep['test_frozen_regret'].mean():.4f}")
print("\nM8 real-holdout decision summary by root cause:")
print(m8_rc_summary.round(4).to_string(index=False))
print("\nM8 expanded prediction error by root cause:")
print(m8_error_by_rc.round(4).to_string(index=False))
print("\nM8 largest-regret decision examples:")
print(m8_trace_examples.round(4).to_string(index=False))

fig_m8_train = px.bar(
    m8_train_frame.groupby(["split", "primary_rc"]).size().reset_index(name="candidate_rows"),
    x="primary_rc", y="candidate_rows", color="split", barmode="group",
    title="M8 Training Candidate Rows by Root Cause",
    labels={"primary_rc": "Root cause", "candidate_rows": "Expanded action rows", "split": "Split"},
)
fig_m8_train.update_layout(width=980, height=430, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_m8_train, "fig04u_m8_training_candidate_rows"); fig_m8_train.show()

fig_m8_pred = px.scatter(
    m8_test_pred_frame,
    x="oracle_reward", y="predicted_reward", color="primary_rc", symbol="action_scope",
    hover_data=["episode_id", "action", "action_cost"],
    title="M8 Predicted vs Oracle Reward on Test Candidate Actions",
    labels={"oracle_reward": "Oracle reward", "predicted_reward": "Predicted reward", "primary_rc": "Root cause"},
)
fig_m8_pred.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(color="black", dash="dash"))
fig_m8_pred.update_layout(width=800, height=520, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_m8_pred, "fig04v_m8_predicted_vs_oracle"); fig_m8_pred.show()

fig_m8_regret = px.bar(
    m8_rc_summary,
    x="primary_rc", y="mean_regret", color="episodes", text="episodes",
    color_continuous_scale="Viridis",
    title="M8 Mean Holdout Regret by Root Cause",
    labels={"primary_rc": "Root cause", "mean_regret": "Mean regret", "episodes": "Episodes"},
)
fig_m8_regret.update_traces(textposition="outside")
fig_m8_regret.update_layout(width=900, height=430, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_m8_regret, "fig04w_m8_regret_by_rc"); fig_m8_regret.show()

fig_m8_heat = px.imshow(
    m8_action_heat.values,
    x=list(m8_action_heat.columns), y=list(m8_action_heat.index),
    color_continuous_scale="Blues", text_auto=True, aspect="auto",
    title="M8 Selected Actions by Root Cause on Real Test Split",
    labels={"x": "Chosen action", "y": "Root cause", "color": "Decisions"},
)
fig_m8_heat.update_layout(width=1000, height=460, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_m8_heat, "fig04x_m8_action_heatmap"); fig_m8_heat.show()

fig_m8_margin = px.scatter(
    m8_decisions,
    x="prediction_margin", y="regret", color="primary_rc", size="allowed_action_count",
    hover_data=["episode_id", "chosen_action", "best_action", "reward", "best_reward"],
    title="M8 Prediction Margin vs Realized Regret",
    labels={"prediction_margin": "Top prediction margin", "regret": "Regret", "primary_rc": "Root cause"},
)
fig_m8_margin.update_layout(width=820, height=480, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_m8_margin, "fig04y_m8_margin_vs_regret"); fig_m8_margin.show()


M8 RewardRanker contract audit:
                                  check  passed                         detail
        feature_dimension_matches_names    True            vector=71, names=71
         state_encoder_dimension_reused    True     state_D=49, state_names=49
     all_training_pairs_allowed_by_mask    True                  extra_pairs=0
test_episode_ids_excluded_from_training    True                      overlap=0
                 reward_targets_bounded    True           min=0.100, max=0.900
                        model_is_fitted    True training_rows=1768, n_iter=250

M8 expanded candidate-row audit:
           split  episodes  candidate_rows  root_causes  actions  synthetic_rows
train_validation       662            1768            8       14             741
            test       111             333            6       14               0

M8 candidate reward summary:
           split  candidate_rows  mean_oracle_reward  min_oracle_reward  max_oracle_reward  mean_predicted_r

**Interpretation:** We read M8 as an auditable offline reward-model baseline rather than as a major performance leap. The contract audit passes every check: the state-action vector has 71 features, the state encoder contributes the same 49 dimensions used by M6 and M7, every expanded training pair stays inside the RC-action mask, and the training rows have 0 overlap with test episode IDs. The supervised table is also large enough for a meaningful offline fit: We train on 1,768 candidate rows from 662 train-validation episodes, including 741 synthetic balancing rows, and we evaluate on 333 candidate rows from 111 fully real test episodes.

The calibration figure shows a clear train-test gap. On train-validation candidate rows, M8 has MAE 0.0075 and RMSE 0.0196, so it fits the reward surface almost exactly. On the chronological test candidates, MAE rises to 0.0862 and RMSE rises to 0.1506, with a positive bias of 0.0637. We see the practical consequence in the root-cause diagnostics: M8 reaches mean holdout reward 0.6351 with mean regret 0.0333, but the regret concentrates in uncovered radio cases. It is perfect on `RC_TRANSPORT_DELAY` with 56 oracle hits out of 56 and on `RC_NONE` with 34 oracle hits out of 34. It fails most clearly on `RC_COVERAGE_HOLE`, where the mean reward is 0.3111 and the mean regret is 0.3000 across 9 episodes. The action heatmap and examples explain why: M8 repeatedly selects `ACT_SWITCH_LINK_CELL` for coverage-hole episodes even when `ACT_ESCALATE` or `ACT_SWITCH_LINK_WIFI` is the oracle-best action. In the largest-regret cases, it assigns predicted rewards around 0.79 to `ACT_SWITCH_LINK_CELL`, but the realized reward is only 0.20 to 0.40 while the best reward reaches 0.78. We therefore keep M8 as a useful offline reward-ranker benchmark, but we do not interpret it as evidence that supervised reward modeling solves the rare root-cause problem.

<a id="sec-16"></a>

## 16 - Benchmark Comparison

After all individual policies are implemented, we compare the full policy family on the same real chronological holdout. We separate raw reward, regret to the oracle ceiling, and deployment mode, and we use only executed sweeps, baseline decisions, or representative holdout traces for the scorecard.

In [23]:

def bootstrap_ci(x, n_boot: int = 5000, seed: int = RANDOM_STATE) -> tuple[float, float]:
    """Return a nonparametric bootstrap confidence interval over run-level means."""
    rng = np.random.default_rng(seed)
    values = np.asarray(x, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return float("nan"), float("nan")
    if len(values) == 1:
        return float(values[0]), float(values[0])
    boots = rng.choice(values, size=(n_boot, len(values)), replace=True).mean(axis=1)
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

def bootstrap_gap_ci(a, b, n_boot: int = 5000, seed: int = RANDOM_STATE) -> tuple[float, float, float]:
    """Return the mean gap and bootstrap CI for two run-level score arrays."""
    rng = np.random.default_rng(seed)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0:
        return float("nan"), float("nan"), float("nan")
    gap = float(a.mean() - b.mean())
    if len(a) == 1 and len(b) == 1:
        return gap, gap, gap
    boots = (
        rng.choice(a, size=(n_boot, len(a)), replace=True).mean(axis=1)
        - rng.choice(b, size=(n_boot, len(b)), replace=True).mean(axis=1)
    )
    return gap, float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

def action_entropy(actions: pd.Series) -> float:
    """Return Shannon entropy over selected actions for one policy trace."""
    shares = actions.value_counts(normalize=True)
    return float(-(shares * np.log2(shares + 1e-12)).sum())

def run_scores_from_decisions(decisions: pd.DataFrame, policy: str, score_col: str = "reward") -> pd.Series:
    """Return one mean score per run from a decision table."""
    frame = decisions[decisions["policy"] == policy].copy()
    if frame.empty:
        return pd.Series(dtype=float)
    return frame.groupby("run_id")[score_col].mean()

def summarize_policy_runs(policy: str, role: str, deployment_mode: str,
                          scores, regrets, frozen_scores=None,
                          llm_coverage: float = 0.0, llm_overrides: float = 0.0,
                          representative_actions: pd.Series | None = None) -> dict:
    """Build one benchmark-comparison summary row from run-level arrays."""
    scores = pd.Series(scores, dtype=float).dropna()
    regrets = pd.Series(regrets, dtype=float).dropna()
    frozen_scores = scores if frozen_scores is None else pd.Series(frozen_scores, dtype=float).dropna()
    ci_lo, ci_hi = bootstrap_ci(scores)
    score_mean = float(scores.mean()) if len(scores) else float("nan")
    regret_mean = float(regrets.mean()) if len(regrets) else float("nan")
    frozen_mean = float(frozen_scores.mean()) if len(frozen_scores) else float("nan")
    return {
        "policy": policy,
        "role": role,
        "deployment_mode": deployment_mode,
        "n_runs": int(len(scores)),
        "online_mean": score_mean,
        "online_std": float(scores.std(ddof=1)) if len(scores) > 1 else 0.0,
        "ci_lo": ci_lo,
        "ci_hi": ci_hi,
        "frozen_mean": frozen_mean,
        "frozen_online_gap": score_mean - frozen_mean,
        "mean_regret": regret_mean,
        "regret_std": float(regrets.std(ddof=1)) if len(regrets) > 1 else 0.0,
        "norm_online": norm_reward(score_mean),
        "gap_to_oracle": R_ORACLE_TEST - score_mean,
        "gain_over_random": score_mean - R_RANDOM_TEST,
        "gap_to_rule": score_mean - R_RULE_TEST,
        "llm_coverage": llm_coverage,
        "llm_overrides": llm_overrides,
        "action_entropy": action_entropy(representative_actions) if representative_actions is not None and len(representative_actions) else np.nan,
    }

R_RULE_TEST = float(baseline_df.loc[baseline_df.policy == "rule_lookup", "mean_reward"].iloc[0])
random_run_scores = run_scores_from_decisions(baseline_decisions, "random_scope", "reward")
random_run_regrets = run_scores_from_decisions(baseline_decisions, "random_scope", "regret")
oracle_score = float(baseline_df.loc[baseline_df.policy == "oracle", "mean_reward"].iloc[0])
oracle_regret = float(baseline_df.loc[baseline_df.policy == "oracle", "mean_regret"].iloc[0])
rule_score = float(baseline_df.loc[baseline_df.policy == "rule_lookup", "mean_reward"].iloc[0])
rule_regret = float(baseline_df.loc[baseline_df.policy == "rule_lookup", "mean_regret"].iloc[0])

policy_summary_rows = [
    summarize_policy_runs("Oracle", "upper_bound", "oracle_reference",
                          [oracle_score], [oracle_regret],
                          representative_actions=baseline_decisions.query("policy == 'oracle'")["chosen_action"]),
    summarize_policy_runs("RandomScope", "lower_anchor", "stochastic_reference",
                          random_run_scores, random_run_regrets,
                          representative_actions=baseline_decisions.query("policy == 'random_scope'")["chosen_action"]),
    summarize_policy_runs("RuleLookup", "structured_prior", "deterministic_reference",
                          [rule_score], [rule_regret],
                          representative_actions=baseline_decisions.query("policy == 'rule_lookup'")["chosen_action"]),
    summarize_policy_runs("EpsilonGreedy", "online_bandit", "online_prequential",
                          eg_sweep["test_online_reward"], eg_sweep["test_online_regret"],
                          frozen_scores=eg_sweep["test_frozen_reward"],
                          representative_actions=eg_holdout_decisions["chosen_action"]),
    summarize_policy_runs("M6 LinUCB", "online_bandit", "online_prequential",
                          m6_sweep["test_online_reward"], m6_sweep["test_online_regret"],
                          frozen_scores=m6_sweep["test_frozen_reward"],
                          representative_actions=m6_holdout_decisions["chosen_action"]),
    summarize_policy_runs("M7 LinUCB+LLM", "online_bandit_llm_arbiter", "online_prequential_with_llm_cache",
                          m7_sweep["test_online_reward"], m7_sweep["test_online_regret"],
                          frozen_scores=m7_sweep["test_frozen_reward"],
                          llm_coverage=m7_sweep["n_llm_used"].mean() / max(1, len(test_eps)),
                          llm_overrides=m7_sweep["n_llm_overrides"].mean(),
                          representative_actions=m7_decisions["chosen_action"]),
    summarize_policy_runs("M8 RewardRanker", "offline_reward_ranker", "offline_frozen",
                          m8_sweep["test_online_reward"], m8_sweep["test_online_regret"],
                          frozen_scores=m8_sweep["test_frozen_reward"],
                          representative_actions=m8_decisions["chosen_action"]),
]
comparison = pd.DataFrame(policy_summary_rows)
comparison["rank_by_reward"] = comparison["online_mean"].rank(method="min", ascending=False).astype(int)
comparison["rank_by_regret"] = comparison["mean_regret"].rank(method="min", ascending=True).astype(int)
comparison = comparison.sort_values(["rank_by_reward", "mean_regret", "policy"]).reset_index(drop=True)

# Keep the historical `summary` object for downstream report figures, but build it from the audited comparison table.
summary = (
    comparison[comparison["policy"].isin(["EpsilonGreedy", "M6 LinUCB", "M7 LinUCB+LLM", "M8 RewardRanker"])]
    .set_index("policy")
    [["n_runs", "online_mean", "online_std", "ci_lo", "ci_hi", "frozen_mean", "norm_online", "llm_coverage", "llm_overrides"]]
    .rename(columns={"n_runs": "n_seeds"})
    .round(4)
)

learned_score_series = {
    "EpsilonGreedy": eg_sweep["test_online_reward"].to_numpy(dtype=float),
    "M6 LinUCB": m6_sweep["test_online_reward"].to_numpy(dtype=float),
    "M7 LinUCB+LLM": m7_sweep["test_online_reward"].to_numpy(dtype=float),
    "M8 RewardRanker": m8_sweep["test_online_reward"].to_numpy(dtype=float),
    "RuleLookup": np.array([R_RULE_TEST], dtype=float),
}
gap_rows = []
for left, left_scores in learned_score_series.items():
    for right, right_scores in learned_score_series.items():
        gap, lo, hi = bootstrap_gap_ci(left_scores, right_scores, seed=RANDOM_STATE + len(gap_rows))
        gap_rows.append({"left_policy": left, "right_policy": right, "mean_gap": gap, "ci_lo": lo, "ci_hi": hi})
pairwise_gaps = pd.DataFrame(gap_rows)

representative_decision_frames = [
    baseline_decisions.query("policy == 'random_scope'").assign(policy_label="RandomScope"),
    baseline_decisions.query("policy == 'rule_lookup'").assign(policy_label="RuleLookup"),
    eg_holdout_decisions.assign(policy_label="EpsilonGreedy"),
    m6_holdout_decisions.assign(policy_label="M6 LinUCB"),
    m7_decisions.assign(policy_label="M7 LinUCB+LLM"),
    m8_decisions.assign(policy_label="M8 RewardRanker"),
]
benchmark_decisions = pd.concat(representative_decision_frames, ignore_index=True, sort=False)
rc_regret_matrix = (
    benchmark_decisions.groupby(["primary_rc", "policy_label"], dropna=False)["regret"]
    .mean()
    .reset_index()
    .pivot(index="primary_rc", columns="policy_label", values="regret")
    .reindex(columns=["RandomScope", "RuleLookup", "EpsilonGreedy", "M6 LinUCB", "M7 LinUCB+LLM", "M8 RewardRanker"])
)

benchmark_audit = pd.DataFrame([
    {"check": "oracle_anchor_matches_baseline", "passed": np.isclose(R_ORACLE_TEST, oracle_score),
     "detail": f"oracle={R_ORACLE_TEST:.4f}"},
    {"check": "random_anchor_has_100_runs", "passed": len(random_run_scores) == 100,
     "detail": f"random_runs={len(random_run_scores)}"},
    {"check": "all_learned_policies_present", "passed": set(summary.index) == {"EpsilonGreedy", "M6 LinUCB", "M7 LinUCB+LLM", "M8 RewardRanker"},
     "detail": f"learned_policies={len(summary)}"},
    {"check": "m8_not_overclaimed_against_rule", "passed": abs(float(summary.loc["M8 RewardRanker", "online_mean"]) - R_RULE_TEST) <= 5e-4,
     "detail": f"m8={float(summary.loc['M8 RewardRanker', 'online_mean']):.4f}, rule={R_RULE_TEST:.4f}"},
    {"check": "test_split_real_only", "passed": not any(ep.is_synthetic for ep in test_eps),
     "detail": f"test_episodes={len(test_eps)}"},
])

print("Benchmark comparison audit:")
print(benchmark_audit.to_string(index=False))
assert benchmark_audit["passed"].all()

print("\nFull benchmark leaderboard on real chronological holdout:")
leader_cols = [
    "rank_by_reward", "policy", "role", "deployment_mode", "n_runs",
    "online_mean", "online_std", "ci_lo", "ci_hi", "frozen_mean",
    "mean_regret", "norm_online", "gap_to_oracle", "gain_over_random", "gap_to_rule",
    "llm_coverage", "action_entropy",
]
print(comparison[leader_cols].round(4).to_string(index=False))

print("\nLearned-policy summary retained for report figures:")
print(summary.to_string())

print("\nPairwise mean reward gaps among learned policies and rule baseline (left minus right):")
gap_table = pairwise_gaps.pivot(index="left_policy", columns="right_policy", values="mean_gap")
print(gap_table.round(4).to_string())

print("\nRoot-cause mean regret matrix:")
print(rc_regret_matrix.round(4).to_string())

PALETTE = {
    "Oracle": "#111111",
    "RandomScope": "#777777",
    "RuleLookup": "#8c564b",
    "EpsilonGreedy": "#ff7f0e",
    "M6 LinUCB": "#1f77b4",
    "M7 LinUCB+LLM": "#2ca02c",
    "M8 RewardRanker": "#9467bd",
}

fig_leader = go.Figure()
plot_rows = comparison[comparison["policy"] != "Oracle"].sort_values("online_mean", ascending=False)
for _, row in plot_rows.iterrows():
    color = PALETTE.get(row["policy"], "#777777")
    fig_leader.add_trace(go.Bar(
        x=[row["policy"]],
        y=[row["online_mean"]],
        error_y=dict(
            type="data",
            array=[max(0.0, row["ci_hi"] - row["online_mean"])],
            arrayminus=[max(0.0, row["online_mean"] - row["ci_lo"])],
            visible=True,
            thickness=1.3,
        ),
        marker=dict(color=color, line=dict(color="#222", width=0.8)),
        text=[f"{row['online_mean']:.3f}"],
        textposition="outside",
        hovertemplate=(
            f"{row['policy']}<br>Score: %{{y:.4f}}"
            f"<br>95% CI: [{row['ci_lo']:.4f}, {row['ci_hi']:.4f}]"
            f"<br>Regret: {row['mean_regret']:.4f}"
            f"<br>Mode: {row['deployment_mode']}<extra></extra>"
        ),
        name=row["policy"],
    ))
fig_leader.add_hline(y=R_RANDOM_TEST, line_dash="dash", line_color="#666", annotation_text=f"random {R_RANDOM_TEST:.3f}")
fig_leader.add_hline(y=R_RULE_TEST, line_dash="dash", line_color="#8c564b", annotation_text=f"rule {R_RULE_TEST:.3f}")
fig_leader.add_hline(y=R_ORACLE_TEST, line_dash="dot", line_color="#111", annotation_text=f"oracle {R_ORACLE_TEST:.3f}")
fig_leader.update_layout(
    title_text="Benchmark Leaderboard on Real Chronological Test Split",
    xaxis_title="Policy",
    yaxis_title="Mean reward",
    yaxis=dict(range=[max(0.0, min(plot_rows["ci_lo"].min(), R_RANDOM_TEST) - 0.04),
                      min(1.0, max(R_ORACLE_TEST, plot_rows["ci_hi"].max()) + 0.04)]),
    showlegend=False,
    width=980,
    height=520,
    template="plotly_white",
    **PLOTLY_LAYOUT,
)
save_fig(fig_leader, "fig05_benchmark_leaderboard"); fig_leader.show()

fig_regret = px.bar(
    comparison[comparison["policy"] != "Oracle"].sort_values("mean_regret"),
    x="policy", y="mean_regret", color="role", text="mean_regret",
    title="Mean Regret to Oracle by Policy",
    labels={"policy": "Policy", "mean_regret": "Mean regret", "role": "Policy role"},
)
fig_regret.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig_regret.update_layout(width=960, height=500, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_regret, "fig06_benchmark_regret_by_policy"); fig_regret.show()

mode_frame = comparison[comparison["policy"].isin(["EpsilonGreedy", "M6 LinUCB", "M7 LinUCB+LLM", "M8 RewardRanker"])].copy()
mode_long = mode_frame.melt(
    id_vars=["policy", "deployment_mode"],
    value_vars=["frozen_mean", "online_mean"],
    var_name="score_view", value_name="mean_reward",
)
fig_mode = px.bar(
    mode_long,
    x="policy", y="mean_reward", color="score_view", barmode="group",
    title="Frozen vs Online Evaluation View",
    labels={"policy": "Policy", "mean_reward": "Mean reward", "score_view": "Evaluation view"},
)
fig_mode.update_layout(width=930, height=500, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_mode, "fig07_frozen_vs_online_comparison"); fig_mode.show()

gap_heat = pairwise_gaps.pivot(index="left_policy", columns="right_policy", values="mean_gap")
fig_gap = px.imshow(
    gap_heat.values,
    x=list(gap_heat.columns), y=list(gap_heat.index),
    color_continuous_scale="RdBu", color_continuous_midpoint=0.0,
    text_auto=".3f", aspect="auto",
    title="Pairwise Mean Reward Gap (Row Policy Minus Column Policy)",
    labels={"x": "Column policy", "y": "Row policy", "color": "Reward gap"},
)
fig_gap.update_layout(width=850, height=650, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_gap, "fig08_pairwise_reward_gaps"); fig_gap.show()

fig_rc = px.imshow(
    rc_regret_matrix.fillna(0.0).values,
    x=list(rc_regret_matrix.columns), y=list(rc_regret_matrix.index),
    color_continuous_scale="Reds", text_auto=".3f", aspect="auto",
    title="Root-Cause Mean Regret by Policy",
    labels={"x": "Policy", "y": "Root cause", "color": "Mean regret"},
)
fig_rc.update_layout(width=980, height=520, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_rc, "fig09_rc_regret_policy_matrix"); fig_rc.show()

fig_progress = px.bar(
    comparison[comparison["policy"].isin(["RandomScope", "RuleLookup", "EpsilonGreedy", "M6 LinUCB", "M7 LinUCB+LLM", "M8 RewardRanker"])].sort_values("norm_online", ascending=False),
    x="policy", y="norm_online", color="role", text="norm_online",
    title="Normalized Progress from Random Anchor to Oracle Ceiling",
    labels={"policy": "Policy", "norm_online": "Normalized reward", "role": "Policy role"},
)
fig_progress.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig_progress.update_layout(width=960, height=500, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_progress, "fig10_normalized_policy_progress"); fig_progress.show()


Benchmark comparison audit:
                          check  passed                 detail
 oracle_anchor_matches_baseline    True          oracle=0.6685
     random_anchor_has_100_runs    True        random_runs=100
   all_learned_policies_present    True     learned_policies=4
m8_not_overclaimed_against_rule    True m8=0.6351, rule=0.6351
           test_split_real_only    True      test_episodes=111

Full benchmark leaderboard on real chronological holdout:
 rank_by_reward          policy                      role                   deployment_mode  n_runs  online_mean  online_std  ci_lo  ci_hi  frozen_mean  mean_regret  norm_online  gap_to_oracle  gain_over_random  gap_to_rule  llm_coverage  action_entropy
              1          Oracle               upper_bound                  oracle_reference       1       0.6685      0.0000 0.6685 0.6685       0.6685       0.0000       1.0000         0.0000            0.1165       0.0333        0.0000          1.9494
              2      RuleLo

**Interpretation:** We use the benchmark comparison as the scorecard for the whole notebook, but we read it through the anchors. The audit passes: the oracle anchor is 0.6685, the random anchor has 100 seeded runs, the four learned policies are present, the test split has 111 real episodes and 0 synthetic episodes, and M8 is explicitly checked against the rule baseline so we do not overclaim the tie. The leaderboard shows that `M8 RewardRanker` has the highest executed policy score at 0.6351, but `RuleLookup` also scores 0.6351 and both have mean regret 0.0333. That means M8 matches the structured prior rather than clearly beating it. The normalized progress plot makes the same point: M8 and RuleLookup both close 0.7139 of the random-to-oracle gap, while EpsilonGreedy closes 0.6182, M6 closes 0.5482, and M7 closes 0.5277.

The online bandit comparison is more informative than the single winner label. EpsilonGreedy is the best online learner at 0.6240 with a 95% bootstrap interval from 0.6167 to 0.6308, followed by M6 at 0.6158 and M7 at 0.6135. The pairwise gap heatmap shows the margins are small: EpsilonGreedy is only 0.0082 above M6 and 0.0105 above M7, while M8 is 0.0111 above EpsilonGreedy but uses an offline frozen ranker rather than online updates. The frozen-versus-online figure explains the deployment difference: M6 and M7 have weak frozen scores around 0.5206 and 0.5211, then improve online to 0.6158 and 0.6135. EpsilonGreedy changes little between frozen and online, moving from 0.6227 to 0.6240. M8 is identical in both columns because it is a frozen offline selector.

The root-cause regret matrix is the most useful scientific diagnostic. Transport delay and no-problem decisions are stable for the structured policies: RuleLookup and M8 both have 0.0000 regret on `RC_TRANSPORT_DELAY` and `RC_NONE`. The persistent weakness is rare radio coverage. `RC_COVERAGE_HOLE` has mean regret 0.3000 for RuleLookup and M8, 0.3222 for EpsilonGreedy, and 0.2933 for M6, so no model solves that slice. We therefore interpret the benchmark as a strong structured-prior result with useful model comparisons, not as proof that a more complex learner dominates. The next scientific improvement should target rare root-cause coverage and counterfactual action evidence, because aggregate reward is already mostly explained by the action mask and reward rules.

<a id="sec-17"></a>

## 17 - LLM Reasoning Case Studies

The benchmark score does not by itself establish whether the reasoning layer is useful. We therefore audit the actual reasoning stored in the current prompt cache, separate valid responses from explicit UCB fallbacks, and evaluate each case by reward and regret rather than by text fluency.

In [24]:

KPI_REASONING_TERMS = [
    "rssi", "sinr", "latency", "queue", "packet", "loss", "bandwidth",
    "bler", "throughput", "handover", "ho", "mcs", "cqi", "traffic",
]

def reasoning_contains_kpi(reasoning: str) -> bool:
    """Return whether reasoning text cites at least one KPI or traffic evidence term."""
    text = str(reasoning or "").lower()
    return any(term in text for term in KPI_REASONING_TERMS)

def reasoning_quality_flags(row: pd.Series) -> dict:
    """Compute simple auditable quality flags for one LLM reasoning row."""
    reasoning = str(row.get("reasoning", "") or "")
    action = str(row.get("llm_chosen", "") or "")
    rc = str(row.get("primary_rc", "") or "")
    text = reasoning.lower()
    return {
        "reasoning_present": bool(reasoning.strip()),
        "mentions_root_cause": rc.lower() in text or rc.replace("RC_", "").replace("_", " ").lower() in text,
        "mentions_chosen_action": action.lower() in text or action.replace("ACT_", "").replace("_", " ").lower() in text,
        "mentions_kpi_evidence": reasoning_contains_kpi(reasoning),
        "reasoning_words": len(reasoning.split()),
    }

def build_llm_case_catalog(episodes: list[Episode]) -> pd.DataFrame:
    """Build one LLM/cache audit row per real test episode."""
    rows = []
    for ep in episodes:
        _, ucb_scores, shortlist, prompt, key = _shortlist_for_episode(_m6_for_arbiter, ep, train_stats)
        entry = arbiter_cache.get(key, {})
        ucb_argmax = max(shortlist, key=lambda action: ucb_scores[action])
        chosen, meta = _choose_m7_action(shortlist, ucb_scores, entry)
        best_action, best_reward = _best_action_for_episode(ep)
        chosen_reward, _ = oracle_reward(ep.primary_rc, chosen, ep.state)
        llm_chosen = entry.get("chosen")
        llm_reward = oracle_reward(ep.primary_rc, llm_chosen, ep.state)[0] if llm_chosen in ep.allowed_actions else np.nan
        ucb_reward = oracle_reward(ep.primary_rc, ucb_argmax, ep.state)[0]
        reasoning = str(entry.get("reasoning", "") or "")
        rows.append({
            "episode_id": ep.episode_id,
            "timestamp": ep.timestamp,
            "primary_rc": ep.primary_rc,
            "rc_confidence": ep.rc_confidence,
            "traffic_type": ep.state.get("traffic_type"),
            "shortlist": " | ".join(shortlist),
            "shortlist_size": len(shortlist),
            "cache_key_present": bool(key in arbiter_cache),
            "llm_ok": bool(entry.get("ok", False)),
            "source": meta.get("source", "ucb_fallback"),
            "override": bool(meta.get("override", False)),
            "ucb_argmax": ucb_argmax,
            "llm_chosen": llm_chosen,
            "chosen_action": chosen,
            "best_action": best_action,
            "llm_confidence": float(entry.get("confidence", np.nan)) if entry.get("ok", False) else np.nan,
            "ucb_reward": float(ucb_reward),
            "llm_reward": float(llm_reward) if np.isfinite(llm_reward) else np.nan,
            "reward": float(chosen_reward),
            "best_reward": float(best_reward),
            "regret": float(best_reward - chosen_reward),
            "llm_regret": float(best_reward - llm_reward) if np.isfinite(llm_reward) else np.nan,
            "ucb_regret": float(best_reward - ucb_reward),
            "reasoning": reasoning,
            "raw_response_chars": len(str(entry.get("raw", "") or "")),
        })
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    flags = frame.apply(reasoning_quality_flags, axis=1, result_type="expand")
    return pd.concat([frame, flags], axis=1)

def summarize_llm_case_catalog(catalog: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return root-cause and source summaries for the LLM case catalogue."""
    rc_summary = (
        catalog.groupby("primary_rc", dropna=False)
        .agg(
            episodes=("episode_id", "size"),
            valid_llm=("llm_ok", "sum"),
            fallbacks=("llm_ok", lambda s: int((~s).sum())),
            reasoning_present=("reasoning_present", "sum"),
            overrides=("override", "sum"),
            mean_reward=("reward", "mean"),
            mean_regret=("regret", "mean"),
            mean_llm_confidence=("llm_confidence", "mean"),
            mean_reasoning_words=("reasoning_words", "mean"),
        )
        .reset_index()
        .sort_values(["valid_llm", "episodes"], ascending=[False, False])
    )
    source_summary = (
        catalog.groupby("source", dropna=False)
        .agg(
            decisions=("episode_id", "size"),
            valid_llm=("llm_ok", "sum"),
            overrides=("override", "sum"),
            mean_reward=("reward", "mean"),
            mean_regret=("regret", "mean"),
            mean_llm_confidence=("llm_confidence", "mean"),
            reasoning_present=("reasoning_present", "sum"),
        )
        .reset_index()
        .sort_values("decisions", ascending=False)
    )
    return rc_summary, source_summary

def select_reasoning_case_studies(catalog: pd.DataFrame, max_cases: int = 10) -> pd.DataFrame:
    """Select a compact, diverse set of valid LLM reasoning cases for display."""
    valid = catalog[catalog["llm_ok"]].copy()
    if valid.empty:
        return valid
    valid["case_priority"] = (
        valid["override"].astype(int) * 4
        + (valid["regret"] > 0).astype(int) * 2
        + (valid["llm_regret"] <= 1e-12).astype(int)
    )
    cases = (
        valid.sort_values(["case_priority", "regret", "primary_rc"], ascending=[False, False, True])
        .groupby("primary_rc", group_keys=False)
        .head(3)
        .sort_values(["override", "regret", "primary_rc"], ascending=[False, False, True])
        .head(max_cases)
        .copy()
    )
    cases["reasoning_short"] = cases["reasoning"].str.slice(0, 230)
    return cases

llm_case_catalog = build_llm_case_catalog(test_eps)
llm_case_rc_summary, llm_case_source_summary = summarize_llm_case_catalog(llm_case_catalog)
llm_reasoning_cases = select_reasoning_case_studies(llm_case_catalog, max_cases=10)
valid_llm_cases = llm_case_catalog[llm_case_catalog["llm_ok"]].copy()

llm_reasoning_audit = pd.DataFrame([
    {"check": "one_catalog_row_per_test_episode", "passed": len(llm_case_catalog) == len(test_eps),
     "detail": f"rows={len(llm_case_catalog)}, test_episodes={len(test_eps)}"},
    {"check": "valid_llm_choices_are_allowed", "passed": valid_llm_cases.apply(lambda r: r["llm_chosen"] in str(r["shortlist"]).split(" | "), axis=1).all() if len(valid_llm_cases) else True,
     "detail": f"valid_llm={len(valid_llm_cases)}"},
    {"check": "valid_llm_reasoning_present", "passed": valid_llm_cases["reasoning_present"].all() if len(valid_llm_cases) else True,
     "detail": f"reasoning_present={int(valid_llm_cases['reasoning_present'].sum()) if len(valid_llm_cases) else 0}"},
    {"check": "reasoning_mentions_kpi_evidence", "passed": valid_llm_cases["mentions_kpi_evidence"].mean() >= 0.50 if len(valid_llm_cases) else True,
     "detail": f"share={valid_llm_cases['mentions_kpi_evidence'].mean():.3f}" if len(valid_llm_cases) else "share=NA"},
    {"check": "fallbacks_are_explicitly_marked", "passed": (llm_case_catalog.loc[~llm_case_catalog["llm_ok"], "source"] == "ucb_fallback").all(),
     "detail": f"fallbacks={int((~llm_case_catalog['llm_ok']).sum())}"},
])

print("LLM reasoning case-study audit:")
print(llm_reasoning_audit.to_string(index=False))
assert llm_reasoning_audit["passed"].all()

print("\nLLM case catalogue summary by source:")
print(llm_case_source_summary.round(4).to_string(index=False))

print("\nLLM case catalogue summary by root cause:")
print(llm_case_rc_summary.round(4).to_string(index=False))

if valid_llm_cases.empty:
    print("\nNo valid current-prompt LLM entries were available; all cases used UCB fallback.")
else:
    quality_summary = pd.DataFrame([{
        "valid_llm_cases": len(valid_llm_cases),
        "reasoning_present": int(valid_llm_cases["reasoning_present"].sum()),
        "mentions_root_cause": int(valid_llm_cases["mentions_root_cause"].sum()),
        "mentions_chosen_action": int(valid_llm_cases["mentions_chosen_action"].sum()),
        "mentions_kpi_evidence": int(valid_llm_cases["mentions_kpi_evidence"].sum()),
        "mean_reasoning_words": valid_llm_cases["reasoning_words"].mean(),
        "mean_llm_confidence": valid_llm_cases["llm_confidence"].mean(),
        "mean_llm_regret": valid_llm_cases["llm_regret"].mean(),
    }])
    print("\nValid LLM reasoning quality summary:")
    print(quality_summary.round(4).to_string(index=False))

print("\nSelected LLM reasoning case studies:")
display_cols = [
    "episode_id", "primary_rc", "rc_confidence", "traffic_type", "shortlist",
    "ucb_argmax", "llm_chosen", "chosen_action", "best_action",
    "llm_confidence", "reward", "regret", "llm_regret", "override", "reasoning_short",
]
if llm_reasoning_cases.empty:
    print("No valid LLM reasoning examples available for current prompts.")
else:
    print(llm_reasoning_cases[display_cols].round(4).to_string(index=False))

coverage_plot = llm_case_rc_summary.copy()
coverage_plot["valid_rate"] = coverage_plot["valid_llm"] / coverage_plot["episodes"].clip(lower=1)
fig_llm_cov = px.bar(
    coverage_plot.sort_values("episodes", ascending=False),
    x="primary_rc", y=["valid_llm", "fallbacks"], barmode="stack",
    title="LLM Reasoning Coverage by Root Cause",
    labels={"primary_rc": "Root cause", "value": "Episodes", "variable": "Cache status"},
)
fig_llm_cov.update_layout(width=950, height=470, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_llm_cov, "fig11_llm_reasoning_coverage_by_rc"); fig_llm_cov.show()

quality_long = pd.DataFrame()
if not valid_llm_cases.empty:
    quality_long = pd.DataFrame({
        "quality_flag": ["mentions_root_cause", "mentions_chosen_action", "mentions_kpi_evidence"],
        "share": [
            valid_llm_cases["mentions_root_cause"].mean(),
            valid_llm_cases["mentions_chosen_action"].mean(),
            valid_llm_cases["mentions_kpi_evidence"].mean(),
        ],
    })
    fig_llm_quality = px.bar(
        quality_long,
        x="quality_flag", y="share", text="share",
        title="Valid LLM Reasoning Evidence Checks",
        labels={"quality_flag": "Reasoning check", "share": "Share of valid LLM cases"},
    )
    fig_llm_quality.update_traces(texttemplate="%{text:.2f}", textposition="outside")
    fig_llm_quality.update_layout(width=780, height=430, template="plotly_white", **PLOTLY_LAYOUT)
    save_fig(fig_llm_quality, "fig12_llm_reasoning_quality_flags"); fig_llm_quality.show()

if not valid_llm_cases.empty:
    fig_llm_conf = px.scatter(
        valid_llm_cases,
        x="llm_confidence", y="llm_regret", color="primary_rc", symbol="override",
        size="reasoning_words",
        hover_data=["episode_id", "llm_chosen", "chosen_action", "best_action", "reward", "reasoning"],
        title="LLM Confidence vs LLM-Chosen Regret",
        labels={"llm_confidence": "LLM confidence", "llm_regret": "Regret if LLM choice is used", "primary_rc": "Root cause"},
    )
    fig_llm_conf.update_layout(width=850, height=500, template="plotly_white", **PLOTLY_LAYOUT)
    save_fig(fig_llm_conf, "fig13_llm_confidence_vs_regret"); fig_llm_conf.show()

source_plot = llm_case_source_summary.copy()
fig_llm_source = px.bar(
    source_plot,
    x="source", y=["mean_reward", "mean_regret"], barmode="group",
    title="M7 Case-Study Outcomes by Decision Source",
    labels={"source": "Decision source", "value": "Mean value", "variable": "Metric"},
)
fig_llm_source.update_layout(width=760, height=430, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_llm_source, "fig14_llm_source_reward_regret"); fig_llm_source.show()

if not llm_reasoning_cases.empty:
    table_df = llm_reasoning_cases[[
        "primary_rc", "traffic_type", "llm_chosen", "chosen_action", "best_action",
        "reward", "regret", "llm_confidence", "reasoning_short",
    ]].copy()
    table_df[["reward", "regret", "llm_confidence"]] = table_df[["reward", "regret", "llm_confidence"]].round(3)
    fig_llm_table = go.Figure(data=[go.Table(
        header=dict(values=list(table_df.columns), fill_color="#f1f3f5", align="left"),
        cells=dict(values=[table_df[col] for col in table_df.columns], align="left"),
    )])
    fig_llm_table.update_layout(title_text="Selected LLM Reasoning Cases", width=1250, height=540)
    save_fig(fig_llm_table, "fig15_llm_reasoning_case_table"); fig_llm_table.show()


LLM reasoning case-study audit:
                           check  passed                      detail
one_catalog_row_per_test_episode    True rows=111, test_episodes=111
   valid_llm_choices_are_allowed    True                valid_llm=24
     valid_llm_reasoning_present    True        reasoning_present=24
 reasoning_mentions_kpi_evidence    True                 share=1.000
 fallbacks_are_explicitly_marked    True                fallbacks=87

LLM case catalogue summary by source:
      source  decisions  valid_llm  overrides  mean_reward  mean_regret  mean_llm_confidence  reasoning_present
ucb_fallback         87          0          0       0.4978       0.1407                  NaN                 87
   llm_blend         24         24          2       0.7521       0.0250               0.9445                 24

LLM case catalogue summary by root cause:
          primary_rc  episodes  valid_llm  fallbacks  reasoning_present  overrides  mean_reward  mean_regret  mean_llm_confidence  mean_

**Interpretation:** We interpret the LLM case studies from cache coverage first, because the LLM only contributes where the current prompt has a valid response. The audit passes: the catalogue has one row for each of the 111 real test episodes, all 24 valid LLM choices stay inside the shortlist, all 24 valid LLM rows contain reasoning, all 24 mention KPI evidence, and all 87 fallback cases are explicitly marked as `ucb_fallback`. In this frozen case catalogue, the valid LLM-blend rows average reward 0.7521 with mean regret 0.0250, while UCB fallback rows average reward 0.4978 with mean regret 0.1407. We read that as useful local evidence for the cached cases, not as a global LLM win, because the valid responses cover only part of the test set.

The root-cause coverage figure shows why we keep the interpretation narrow. `RC_NONE` has the most valid LLM responses with 17 valid entries and 17 fallbacks. `RC_TRANSPORT_DELAY` has 5 valid entries and 51 fallbacks, while `RC_CAPACITY_OVERLOAD` and `RC_COVERAGE_HOLE` each have only 1 valid entry. `RC_SINR_DEGRADED` and `RC_WEAK_SIGNAL` have 0 valid LLM reasoning cases. This means the case studies mainly audit no-problem and transport-delay decisions; they do not support a broad claim about radio reasoning.

The reasoning-quality checks show that the valid responses are useful but imperfect audit artifacts. All 24 valid cases mention the root cause and KPI evidence, but only 8 of 24 explicitly mention the chosen action by name or phrase. The mean reasoning length is 56.8750 words, mean LLM confidence is 0.9445, and mean regret if we used the LLM choice directly is 0.0083. The selected cases make the tradeoff concrete. In one transport-delay episode, the LLM argues for `ACT_QUEUE_MANAGE` because queue delay is high; the blended decision still chooses `ACT_PATH_REROUTE`, producing reward 0.4000 and regret 0.1000, while the LLM action itself has 0 regret. In another transport-delay episode, the override to `ACT_QUEUE_MANAGE` gets reward 0.5000 and 0 regret. The coverage-hole example is cautionary: the LLM chooses `ACT_SWITCH_LINK_CELL` from an RSSI-based explanation, but the reward is 0.2000 and regret is 0.2000 because `ACT_SWITCH_LINK_WIFI` is better for that episode. We therefore treat the LLM text as a transparent explanation of a bounded shortlist choice, not as proof that the chosen action is optimal.

<a id="sec-18"></a>

## 18 - Counterfactual and Diagnosis-Posterior Robustness

After the clean benchmark, we stress-test the policy conclusions under diagnostic uncertainty. We perturb root-cause labels on real test episodes, re-score frozen policies over repeated counterfactual batches, and evaluate selected actions under small diagnosis posteriors to estimate expected reward and worst-case regret.

In [25]:

import copy

ROBUST_POLICIES = ["RuleLookup", "EpsilonGreedy", "M6 LinUCB", "M7 LinUCB+LLM", "M8 RewardRanker"]
ROBUST_FLIP_GRID = [0.0, 0.1, 0.2, 0.3, 0.4]
ROBUST_REPS = 20

def alternative_root_causes(primary_rc: str, max_alternatives: int = 3) -> list[str]:
    """Return plausible alternative root causes using the same diagnostic scope when possible."""
    if primary_rc == RC_NO_PROBLEM:
        return []
    primary_scope = RC_SCOPE.get(primary_rc, "none")
    same_scope = [
        rc for rc in RC_TO_ACTIONS
        if rc not in {primary_rc, RC_NO_PROBLEM} and RC_SCOPE.get(rc, "none") == primary_scope
    ]
    other_scope = [
        rc for rc in RC_TO_ACTIONS
        if rc not in {primary_rc, RC_NO_PROBLEM} and rc not in same_scope
    ]
    return (same_scope + other_scope)[:max_alternatives]

def perturb_episode(ep: Episode, rng: np.random.Generator, p_flip: float,
                    conf_sigma: float = 0.12) -> tuple[Episode, dict]:
    """Return a counterfactual episode plus metadata for one diagnosis perturbation."""
    should_flip = bool(ep.primary_rc != RC_NO_PROBLEM and rng.random() < p_flip)
    new_rc = ep.primary_rc
    new_conf = float(ep.rc_confidence)
    if should_flip:
        alternatives = alternative_root_causes(ep.primary_rc, max_alternatives=4)
        new_rc = str(rng.choice(alternatives)) if alternatives else ep.primary_rc
        new_conf = float(np.clip(ep.rc_confidence + rng.normal(0.0, conf_sigma), 0.05, 0.99))
    new_episode = Episode(
        episode_id=ep.episode_id if not should_flip else f"cf_{p_flip:.2f}_{new_rc}_{ep.episode_id}",
        timestamp=ep.timestamp,
        zone_id=ep.zone_id,
        primary_rc=new_rc,
        rc_confidence=new_conf,
        allowed_actions=list(RC_TO_ACTIONS.get(new_rc, ["ACT_NO_OP"])),
        state=dict(ep.state),
        severity=ep.severity,
        source_file=ep.source_file,
        is_incident=ep.is_incident,
        is_synthetic=False,
    )
    meta = {
        "episode_id": ep.episode_id,
        "original_rc": ep.primary_rc,
        "perturbed_rc": new_rc,
        "flipped": should_flip and new_rc != ep.primary_rc,
        "original_confidence": ep.rc_confidence,
        "perturbed_confidence": new_conf,
    }
    return new_episode, meta

def make_counterfactual_batch(episodes: list[Episode], p_flip: float,
                              rep: int, seed: int = RANDOM_STATE) -> tuple[list[Episode], pd.DataFrame]:
    """Create one reproducible counterfactual batch and its perturbation audit table."""
    rng = np.random.default_rng(seed + 10_000 + int(p_flip * 1000) * 37 + rep)
    batch, meta_rows = [], []
    for ep in episodes:
        cf_ep, meta = perturb_episode(ep, rng, p_flip=p_flip)
        meta["rep"] = rep
        meta["rc_flip_prob"] = p_flip
        batch.append(cf_ep)
        meta_rows.append(meta)
    return batch, pd.DataFrame(meta_rows)

def train_frozen_m6(seed: int = RANDOM_STATE + 700) -> LinUCBDispatcher:
    """Train one frozen M6 dispatcher on train-validation episodes."""
    bandit = LinUCBDispatcher(RC_TO_ACTIONS, D, alpha=0.4, lam=3.0, beta0=2.0,
                              rng=np.random.default_rng(seed))
    run_linucb_episode(bandit, _TRAINVAL, train_stats, update=True)
    return bandit

def train_frozen_eg(seed: int = RANDOM_STATE + 710) -> EpsilonGreedyDispatcher:
    """Train one frozen epsilon-greedy dispatcher on train-validation episodes."""
    bandit = EpsilonGreedyDispatcher(RC_TO_ACTIONS, epsilon_0=0.3, rng=np.random.default_rng(seed))
    run_eg_episode(bandit, _TRAINVAL, update=True)
    return bandit

def evaluate_frozen_eg_on(bandit: EpsilonGreedyDispatcher, episodes: list[Episode]) -> dict:
    """Evaluate a pre-trained epsilon-greedy dispatcher without holdout updates."""
    return run_eg_episode(bandit, episodes, update=False, collect=True)

def evaluate_frozen_m6_on(bandit: LinUCBDispatcher, episodes: list[Episode]) -> dict:
    """Evaluate a pre-trained M6 dispatcher without holdout updates."""
    return run_linucb_episode(bandit, episodes, train_stats, update=False, collect=True)

def evaluate_frozen_m7_on(bandit: LinUCBDispatcher, episodes: list[Episode]) -> dict:
    """Evaluate a pre-trained M7 dispatcher with prompt-keyed cache lookup and no holdout updates."""
    return evaluate_m7_frozen(bandit, episodes, arbiter_cache, train_stats)

def evaluate_rule_lookup_on(episodes: list[Episode]) -> dict:
    """Evaluate the deterministic rule-lookup prior on arbitrary episodes."""
    return evaluate_policy_episodes(RuleLookupPolicy(), episodes)

def evaluate_m8_frozen_on(episodes: list[Episode]) -> dict:
    """Evaluate the fitted M8 reward ranker on arbitrary episodes."""
    return evaluate_reward_ranker(m8_policy, episodes, collect=True)

def evaluate_robust_policy(policy_name: str, episodes: list[Episode],
                           m6_bandit: LinUCBDispatcher,
                           eg_bandit: EpsilonGreedyDispatcher,
                           m7_bandit: LinUCBDispatcher) -> dict:
    """Evaluate one named policy on a counterfactual episode batch."""
    if policy_name == "RuleLookup":
        return evaluate_rule_lookup_on(episodes)
    if policy_name == "EpsilonGreedy":
        return evaluate_frozen_eg_on(eg_bandit, episodes)
    if policy_name == "M6 LinUCB":
        return evaluate_frozen_m6_on(m6_bandit, episodes)
    if policy_name == "M7 LinUCB+LLM":
        return evaluate_frozen_m7_on(m7_bandit, episodes)
    if policy_name == "M8 RewardRanker":
        return evaluate_m8_frozen_on(episodes)
    raise ValueError(f"Unknown robust policy: {policy_name}")

def m7_cache_coverage_for_batch(bandit: LinUCBDispatcher, episodes: list[Episode]) -> float:
    """Return the share of episodes with valid current-prompt LLM cache entries."""
    valid = 0
    for ep in episodes:
        _, _, _, prompt, key = _shortlist_for_episode(bandit, ep, train_stats)
        entry = arbiter_cache.get(key, {})
        valid += int(isinstance(entry, dict) and _cache_entry_valid(entry, prompt) and bool(entry.get("ok")))
    return valid / max(1, len(episodes))

def summarize_counterfactual_meta(meta: pd.DataFrame) -> pd.DataFrame:
    """Summarize actual perturbation counts by configured flip probability."""
    return (
        meta.groupby(["rc_flip_prob", "rep"], as_index=False)
        .agg(flipped_episodes=("flipped", "sum"),
             total_episodes=("episode_id", "size"))
        .assign(actual_flip_rate=lambda d: d["flipped_episodes"] / d["total_episodes"])
    )

base_m6_robust_bandit = train_frozen_m6()
base_eg_robust_bandit = train_frozen_eg()
base_m7_robust_bandit = train_frozen_m6(seed=RANDOM_STATE + 701)

robust_rows = []
robust_decision_frames = []
perturb_meta_frames = []
for p_flip in ROBUST_FLIP_GRID:
    for rep in range(ROBUST_REPS):
        cf_eps, meta = make_counterfactual_batch(test_eps, p_flip=p_flip, rep=rep)
        perturb_meta_frames.append(meta)
        m6_eval_bandit = copy.deepcopy(base_m6_robust_bandit)
        eg_eval_bandit = copy.deepcopy(base_eg_robust_bandit)
        m7_eval_bandit = copy.deepcopy(base_m7_robust_bandit)
        m7_cov = m7_cache_coverage_for_batch(m7_eval_bandit, cf_eps)
        for policy_name in ROBUST_POLICIES:
            result = evaluate_robust_policy(policy_name, cf_eps, m6_eval_bandit, eg_eval_bandit, m7_eval_bandit)
            robust_rows.append({
                "policy": policy_name,
                "rc_flip_prob": p_flip,
                "rep": rep,
                "mean_reward": result["mean_reward"],
                "mean_regret": result["mean_regret"],
                "llm_coverage": m7_cov if policy_name == "M7 LinUCB+LLM" else 0.0,
            })
            decisions = result.get("decisions", result.get("_decisions", pd.DataFrame())).copy()
            if not decisions.empty:
                decisions["policy"] = policy_name
                decisions["rc_flip_prob"] = p_flip
                decisions["rep"] = rep
                robust_decision_frames.append(decisions)

robust_df = pd.DataFrame(robust_rows)
robust_decisions = pd.concat(robust_decision_frames, ignore_index=True, sort=False)
perturbation_audit = pd.concat(perturb_meta_frames, ignore_index=True)
perturbation_summary = summarize_counterfactual_meta(perturbation_audit)
robust_summary = (
    robust_df.groupby(["policy", "rc_flip_prob"], as_index=False)
    .agg(mean_reward=("mean_reward", "mean"),
         reward_std=("mean_reward", "std"),
         mean_regret=("mean_regret", "mean"),
         regret_std=("mean_regret", "std"),
         llm_coverage=("llm_coverage", "mean"),
         reps=("rep", "nunique"))
)
robust_clean = robust_summary[robust_summary["rc_flip_prob"] == 0.0][["policy", "mean_reward", "mean_regret"]].rename(
    columns={"mean_reward": "clean_reward", "mean_regret": "clean_regret"}
)
robust_summary = robust_summary.merge(robust_clean, on="policy", how="left")
robust_summary["reward_drop_from_clean"] = robust_summary["clean_reward"] - robust_summary["mean_reward"]
robust_summary["regret_increase_from_clean"] = robust_summary["mean_regret"] - robust_summary["clean_regret"]

robust_path = INTERIM_DIR / "counterfactual_diagnostic_robustness.csv"
robust_df.to_csv(robust_path, index=False)
register_hash("benchmark:counterfactual_diagnostic_robustness.csv", robust_path)
robust_summary_path = INTERIM_DIR / "counterfactual_diagnostic_robustness_summary.csv"
robust_summary.to_csv(robust_summary_path, index=False)
register_hash("benchmark:counterfactual_diagnostic_robustness_summary.csv", robust_summary_path)

def posterior_rc_candidates(ep: Episode, k: int = 3) -> list[tuple[str, float]]:
    """Create a normalized diagnosis posterior around the primary root cause."""
    primary = ep.primary_rc
    if primary == RC_NO_PROBLEM:
        return [(RC_NO_PROBLEM, 1.0)]
    alternatives = alternative_root_causes(primary, max_alternatives=max(0, k - 1))
    primary_weight = float(np.clip(ep.rc_confidence, 0.35, 0.90))
    alt_total = 1.0 - primary_weight
    alt_weight = alt_total / max(1, len(alternatives))
    posterior = [(primary, primary_weight)] + [(rc, alt_weight) for rc in alternatives]
    total = sum(weight for _, weight in posterior)
    return [(rc, weight / total) for rc, weight in posterior]

def score_action_under_posterior(ep: Episode, action: str,
                                 posterior: list[tuple[str, float]]) -> tuple[float, float, float]:
    """Return expected reward, expected regret, and worst regret for an action under a diagnosis posterior."""
    expected_reward = 0.0
    expected_regret = 0.0
    worst_regret = 0.0
    for rc, weight in posterior:
        allowed = RC_TO_ACTIONS.get(rc, ["ACT_NO_OP"])
        if action in allowed:
            reward, _ = oracle_reward(rc, action, ep.state)
        else:
            reward = DEFAULT_REWARD
        best = max(oracle_reward(rc, candidate, ep.state)[0] for candidate in allowed)
        regret = best - reward
        expected_reward += weight * reward
        expected_regret += weight * regret
        worst_regret = max(worst_regret, regret)
    return float(expected_reward), float(expected_regret), float(worst_regret)

def select_actions_for_policy(policy_name: str, episodes: list[Episode],
                              m6_bandit: LinUCBDispatcher,
                              eg_bandit: EpsilonGreedyDispatcher,
                              m7_bandit: LinUCBDispatcher) -> list[str]:
    """Select one action per episode for posterior uncertainty evaluation."""
    actions = []
    for ep in episodes:
        if policy_name == "RuleLookup":
            actions.append(ep.allowed_actions[0])
        elif policy_name == "EpsilonGreedy":
            idx = eg_bandit.select(ep.primary_rc, ep.allowed_actions)
            actions.append(ep.allowed_actions[idx])
        elif policy_name == "M6 LinUCB":
            x = _encode_state({**ep.state, "__rc__": ep.primary_rc}, train_stats)
            idx = m6_bandit.select(ep.primary_rc, x, ep.allowed_actions)
            actions.append(ep.allowed_actions[idx])
        elif policy_name == "M7 LinUCB+LLM":
            _, ucb_scores, shortlist, _, key = _shortlist_for_episode(m7_bandit, ep, train_stats)
            action, _ = _choose_m7_action(shortlist, ucb_scores, arbiter_cache.get(key, {}))
            actions.append(action)
        elif policy_name == "M8 RewardRanker":
            actions.append(m8_policy.select(ep))
        else:
            raise ValueError(f"Unknown policy {policy_name}")
    return actions

posterior_rows = []
posterior_case_rows = []
for policy_name in ROBUST_POLICIES:
    actions = select_actions_for_policy(policy_name, test_eps, copy.deepcopy(base_m6_robust_bandit), copy.deepcopy(base_eg_robust_bandit), copy.deepcopy(base_m7_robust_bandit))
    for ep, action in zip(test_eps, actions):
        posterior = posterior_rc_candidates(ep, k=3)
        expected_reward, expected_regret, worst_regret = score_action_under_posterior(ep, action, posterior)
        posterior_case_rows.append({
            "policy": policy_name,
            "episode_id": ep.episode_id,
            "primary_rc": ep.primary_rc,
            "action": action,
            "posterior": " | ".join(f"{rc}:{weight:.2f}" for rc, weight in posterior),
            "posterior_expected_reward": expected_reward,
            "posterior_expected_regret": expected_regret,
            "posterior_worst_regret": worst_regret,
        })
    policy_cases = [row for row in posterior_case_rows if row["policy"] == policy_name]
    posterior_rows.append({
        "policy": policy_name,
        "posterior_expected_reward": float(np.mean([row["posterior_expected_reward"] for row in policy_cases])),
        "posterior_expected_regret": float(np.mean([row["posterior_expected_regret"] for row in policy_cases])),
        "posterior_worst_regret": float(np.mean([row["posterior_worst_regret"] for row in policy_cases])),
    })

posterior_case_df = pd.DataFrame(posterior_case_rows)
uncertainty_df = pd.DataFrame(posterior_rows)
uncertainty_path = INTERIM_DIR / "diagnosis_posterior_uncertainty.csv"
uncertainty_df.to_csv(uncertainty_path, index=False)
register_hash("benchmark:diagnosis_posterior_uncertainty.csv", uncertainty_path)
posterior_case_path = INTERIM_DIR / "diagnosis_posterior_uncertainty_cases.csv"
posterior_case_df.to_csv(posterior_case_path, index=False)
register_hash("benchmark:diagnosis_posterior_uncertainty_cases.csv", posterior_case_path)

posterior_rc_matrix = (
    posterior_case_df.groupby(["primary_rc", "policy"], as_index=False)
    .agg(expected_reward=("posterior_expected_reward", "mean"),
         worst_regret=("posterior_worst_regret", "mean"))
)

robustness_audit = pd.DataFrame([
    {"check": "all_flip_levels_have_all_reps", "passed": robust_df.groupby("rc_flip_prob")["rep"].nunique().eq(ROBUST_REPS).all(),
     "detail": f"levels={len(ROBUST_FLIP_GRID)}, reps={ROBUST_REPS}"},
    {"check": "all_policies_evaluated", "passed": set(robust_df["policy"]) == set(ROBUST_POLICIES),
     "detail": f"policies={len(set(robust_df['policy']))}"},
    {"check": "counterfactual_test_rows_real_only", "passed": not robust_decisions.get("is_synthetic", pd.Series([False])).fillna(False).any(),
     "detail": f"decision_rows={len(robust_decisions)}"},
    {"check": "posterior_weights_normalized", "passed": all(abs(sum(w for _, w in posterior_rc_candidates(ep)) - 1.0) < 1e-9 for ep in test_eps),
     "detail": f"episodes={len(test_eps)}"},
    {"check": "posterior_policy_rows_complete", "passed": len(posterior_case_df) == len(test_eps) * len(ROBUST_POLICIES),
     "detail": f"rows={len(posterior_case_df)}"},
])

print("Robustness section audit:")
print(robustness_audit.to_string(index=False))
assert robustness_audit["passed"].all()

print("\nCounterfactual perturbation summary:")
print(
    perturbation_summary.groupby("rc_flip_prob", as_index=False)
    .agg(mean_flipped=("flipped_episodes", "mean"),
         std_flipped=("flipped_episodes", "std"),
         mean_actual_flip_rate=("actual_flip_rate", "mean"))
    .round(4)
    .to_string(index=False)
)

print("\nCounterfactual diagnostic robustness summary:")
print(robust_summary.round(4).to_string(index=False))

print("\nDiagnosis posterior uncertainty benchmark:")
print(uncertainty_df.round(4).to_string(index=False))

print("\nPosterior uncertainty by root cause:")
print(
    posterior_rc_matrix.pivot(index="primary_rc", columns="policy", values="worst_regret")
    .round(4)
    .to_string()
)

fig_robust_reward = px.line(
    robust_summary,
    x="rc_flip_prob", y="mean_reward", color="policy", markers=True,
    error_y="reward_std",
    title="Counterfactual Robustness: Reward Under Diagnosis Flips",
    labels={"rc_flip_prob": "Configured root-cause flip probability", "mean_reward": "Mean frozen reward", "policy": "Policy"},
)
fig_robust_reward.update_layout(width=920, height=480, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_robust_reward, "fig16_counterfactual_reward_by_flip"); fig_robust_reward.show()

fig_robust_regret = px.line(
    robust_summary,
    x="rc_flip_prob", y="mean_regret", color="policy", markers=True,
    error_y="regret_std",
    title="Counterfactual Robustness: Regret Under Diagnosis Flips",
    labels={"rc_flip_prob": "Configured root-cause flip probability", "mean_regret": "Mean frozen regret", "policy": "Policy"},
)
fig_robust_regret.update_layout(width=920, height=480, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_robust_regret, "fig17_counterfactual_regret_by_flip"); fig_robust_regret.show()

flip_summary_plot = (
    perturbation_summary.groupby("rc_flip_prob", as_index=False)
    .agg(mean_actual_flip_rate=("actual_flip_rate", "mean"))
)
m7_cov_plot = robust_summary[robust_summary["policy"] == "M7 LinUCB+LLM"][["rc_flip_prob", "llm_coverage"]]
flip_cov_plot = flip_summary_plot.merge(m7_cov_plot, on="rc_flip_prob", how="left")
fig_flip = go.Figure()
fig_flip.add_trace(go.Scatter(x=flip_cov_plot["rc_flip_prob"], y=flip_cov_plot["mean_actual_flip_rate"],
                              mode="lines+markers", name="actual flip rate"))
fig_flip.add_trace(go.Scatter(x=flip_cov_plot["rc_flip_prob"], y=flip_cov_plot["llm_coverage"],
                              mode="lines+markers", name="M7 valid LLM cache coverage"))
fig_flip.update_layout(
    title_text="Actual Diagnosis Perturbation Rate and M7 Cache Coverage",
    xaxis_title="Configured root-cause flip probability",
    yaxis_title="Share of test episodes",
    width=850, height=450, template="plotly_white", **PLOTLY_LAYOUT,
)
save_fig(fig_flip, "fig18_flip_rate_and_llm_cache_coverage"); fig_flip.show()

uncertainty_long = uncertainty_df.melt(
    id_vars="policy",
    value_vars=["posterior_expected_reward", "posterior_expected_regret", "posterior_worst_regret"],
    var_name="metric", value_name="value",
)
fig_uncertainty = px.bar(
    uncertainty_long,
    x="policy", y="value", color="metric", barmode="group",
    title="Diagnosis-Posterior Robustness Metrics",
    labels={"policy": "Policy", "value": "Mean posterior metric", "metric": "Metric"},
)
fig_uncertainty.update_layout(width=980, height=500, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_uncertainty, "fig19_diagnosis_posterior_metrics"); fig_uncertainty.show()

posterior_worst = posterior_rc_matrix.pivot(index="primary_rc", columns="policy", values="worst_regret").fillna(0.0)
fig_post_rc = px.imshow(
    posterior_worst.values,
    x=list(posterior_worst.columns), y=list(posterior_worst.index),
    color_continuous_scale="Reds", text_auto=".3f", aspect="auto",
    title="Diagnosis-Posterior Worst Regret by Root Cause",
    labels={"x": "Policy", "y": "Primary root cause", "color": "Worst regret"},
)
fig_post_rc.update_layout(width=980, height=520, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_post_rc, "fig20_posterior_worst_regret_by_rc"); fig_post_rc.show()


Robustness section audit:
                             check  passed              detail
     all_flip_levels_have_all_reps    True   levels=5, reps=20
            all_policies_evaluated    True          policies=5
counterfactual_test_rows_real_only    True decision_rows=44400
      posterior_weights_normalized    True        episodes=111
    posterior_policy_rows_complete    True            rows=555

Counterfactual perturbation summary:
 rc_flip_prob  mean_flipped  std_flipped  mean_actual_flip_rate
       0.0000        0.0000       0.0000                 0.0000
       0.1000        7.9500       3.2843                 0.0716
       0.2000       16.6000       5.1748                 0.1495
       0.3000       23.4000       3.8443                 0.2108
       0.4000       31.1000       4.3878                 0.2802

Counterfactual diagnostic robustness summary:
         policy  rc_flip_prob  mean_reward  reward_std  mean_regret  regret_std  llm_coverage  reps  clean_reward  clean_regret

**Interpretation:** We interpret this robustness section as a stress test, not as a replacement for the clean chronological benchmark. The audit passes: all 5 flip levels have 20 repeated batches, all 5 policies are evaluated, the counterfactual decision table contains 44,400 real holdout-derived decision rows, posterior weights sum to 1 for all 111 episodes, and the posterior case table has 555 rows. The perturbation audit also confirms that the configured flip probability translates into actual root-cause changes: the mean actual flip rate rises from 0.0000 at `p=0.0` to 0.0716, 0.1495, 0.2108, and 0.2802 as the configured probability increases to 0.4. We use the actual flip-rate curve to verify the stress test instead of assuming the configuration behaved as intended.

The counterfactual reward figure shows two different robustness patterns. RuleLookup and M8 remain close because both are anchored in the same structured RC-action mask. At `p=0.4`, RuleLookup falls from 0.6351 to 0.6189 and M8 falls from 0.6351 to 0.6195, so the reward drop is only about 0.016. EpsilonGreedy starts slightly lower in this frozen stress-test view at 0.6334 and drops to 0.6023 at `p=0.4`, a larger drop of 0.0311. M6 and M7 move in the opposite direction because some synthetic diagnosis flips move episodes into action masks that fit their frozen estimates better: M6 rises from 0.5088 to 0.5393, and M7 rises from 0.4841 to 0.5205. We do not read that as real improvement under noise; we read it as evidence that this perturbation changes the evaluated task distribution, so reward movement must be interpreted together with regret and flip counts. The regret figure keeps the structured-prior result clear: at `p=0.4`, M8 has regret 0.0405 and RuleLookup has 0.0410, while EpsilonGreedy has 0.0577, M6 has 0.1206, and M7 has 0.1395.

The M7 cache-coverage line is an important guardrail. Valid LLM cache coverage is 0.1712 on the unperturbed batch and declines slightly to 0.1635 at `p=0.4`. That happens because changed root causes create changed prompts, and we do not reuse stale reasoning for a different diagnostic context. This is the correct conservative behavior for an LLM-assisted benchmark.

The diagnosis-posterior table gives the stricter uncertainty view. EpsilonGreedy has the highest posterior expected reward at 0.5425, but its average worst-case regret is 0.1659. RuleLookup and M8 are almost tied on expected reward at 0.5405 and have the lowest average worst-case regret at 0.1550. M6 trails with expected reward 0.4900 and worst-case regret 0.2149, and M7 is lowest at 0.4786 with worst-case regret 0.2347. The root-cause heatmap shows where uncertainty hurts: `RC_NONE` has 0.0000 worst regret for every policy, but `RC_COVERAGE_HOLE`, `RC_SINR_DEGRADED`, and `RC_WEAK_SIGNAL` create large worst-regret cells, often around 0.3333 to 0.4250. We therefore read the robustness section as confirming the main scientific finding: the benchmark is stable for no-problem and transport-oriented decisions, but rare radio diagnoses remain the place where posterior uncertainty can move the true root cause across incompatible action masks.

<a id="sec-19"></a>

## 19 - Reward Mode Comparison

The robustness analysis is followed by a reward-definition check. We compare each final policy under the intent reward used by the bandit benchmark, the multi-objective reward that adds cost and safety, and an action-aware simulated outcome score whose assumptions are explicit because the dataset has no post-remediation logs.

In [26]:

import copy as _copy

def _bounded_scale(value, factor=None, delta=None, lo=None, hi=None):
    """Apply a bounded multiplicative or additive change to one numeric value."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return value
    out = float(value)
    if factor is not None:
        out *= float(factor)
    if delta is not None:
        out += float(delta)
    if lo is not None or hi is not None:
        out = float(np.clip(out, -np.inf if lo is None else lo, np.inf if hi is None else hi))
    return out

ACTION_OUTCOME_EFFECTS = {
    "ACT_NO_OP": {},
    "ACT_BAND_STEER_5G": {"wifi_signal_score": ("add", 8, 0, 100), "throughput_mbps": ("mul", 1.08, 0, None), "latency_ms": ("mul", 0.95, 0, None)},
    "ACT_CHANNEL_CHANGE": {"sinr_db": ("add", 3, -30, 40), "bler_proxy_pct": ("mul", 0.75, 0, 100), "packet_loss_pct": ("mul", 0.80, 0, 100)},
    "ACT_SWITCH_LINK_CELL": {"cellular_signal_score": ("add", 8, 0, 100), "rssi_dbm": ("add", 4, -140, -30), "throughput_mbps": ("mul", 1.05, 0, None)},
    "ACT_SWITCH_LINK_WIFI": {"wifi_signal_score": ("add", 10, 0, 100), "latency_ms": ("mul", 0.92, 0, None), "throughput_mbps": ("mul", 1.06, 0, None)},
    "ACT_FORCE_HO": {"rssi_dbm": ("add", 5, -140, -30), "sinr_db": ("add", 2, -30, 40), "ho_success_rate_pct": ("mul", 0.97, 0, 100)},
    "ACT_NEIGHBOR_RECFG": {"ho_success_rate_pct": ("add", 8, 0, 100), "neighbor_count": ("add", 1, 0, None)},
    "ACT_MCS_CAP": {"bler_proxy_pct": ("mul", 0.65, 0, 100), "mcs": ("mul", 0.85, 0, None), "packet_loss_pct": ("mul", 0.80, 0, 100)},
    "ACT_QOS_THROTTLE_BULK": {"latency_ms": ("mul", 0.88, 0, None), "jitter_ms": ("mul", 0.90, 0, None), "queue_length": ("mul", 0.80, 0, None), "throughput_mbps": ("mul", 0.96, 0, None)},
    "ACT_QUEUE_MANAGE": {"latency_ms": ("mul", 0.80, 0, None), "jitter_ms": ("mul", 0.82, 0, None), "queue_length": ("mul", 0.62, 0, None), "packet_loss_pct": ("mul", 0.90, 0, 100)},
    "ACT_PATH_REROUTE": {"latency_ms": ("mul", 0.84, 0, None), "jitter_ms": ("mul", 0.86, 0, None), "packet_loss_pct": ("mul", 0.75, 0, 100), "tcp_retransmit_rate": ("mul", 0.75, 0, None)},
    "ACT_LOAD_BALANCE": {"bandwidth_util_pct": ("mul", 0.82, 0, 100), "active_connections": ("mul", 0.90, 0, None), "queue_length": ("mul", 0.78, 0, None), "throughput_mbps": ("mul", 1.08, 0, None)},
    "ACT_DEFER_OFFPEAK": {"bandwidth_util_pct": ("mul", 0.90, 0, 100), "queue_length": ("mul", 0.92, 0, None)},
    "ACT_MODEM_SOFT_RESET": {"latency_ms": ("mul", 0.70, 0, None), "jitter_ms": ("mul", 0.75, 0, None), "packet_loss_pct": ("mul", 0.70, 0, 100), "throughput_mbps": ("mul", 1.12, 0, None)},
    "ACT_ESCALATE": {},
}

def simulate_post_action_state(pre_state: dict, action: str) -> dict:
    """Return an action-aware simulated post-state for outcome-reward sensitivity checks."""
    post = dict(pre_state)
    for kpi, (op, amount, lo, hi) in ACTION_OUTCOME_EFFECTS.get(action, {}).items():
        current = post.get(kpi)
        if op == "mul":
            post[kpi] = _bounded_scale(current, factor=amount, lo=lo, hi=hi)
        elif op == "add":
            post[kpi] = _bounded_scale(current, delta=amount, lo=lo, hi=hi)
        else:
            raise ValueError(f"Unknown outcome operation {op}")
    return post

def score_action_reward_modes(ep: Episode, action: str) -> dict:
    """Score one selected action under intent, multi-objective, and simulated outcome reward views."""
    intent_reward, _ = oracle_reward(ep.primary_rc, action, ep.state)
    multi_reward, _ = oracle_reward_multiobjective(ep.primary_rc, action, ep.state)
    post_state = simulate_post_action_state(ep.state, action)
    outcome_reward = oracle_reward_outcome(ep.state, post_state)
    return {
        "intent_reward": float(intent_reward),
        "multi_objective_reward": float(multi_reward),
        "simulated_outcome_reward": float(outcome_reward),
    }

def trace_from_decisions(policy_name: str, decisions: pd.DataFrame) -> pd.DataFrame:
    """Normalize a policy decision table to episode/action rows for reward-mode scoring."""
    frame = decisions.copy()
    if "chosen_action" not in frame.columns:
        raise KeyError(f"{policy_name} decisions must include chosen_action")
    frame = frame[["episode_id", "primary_rc", "chosen_action", "reward", "regret"]].copy()
    frame["policy"] = policy_name
    return frame

def select_hybrid_orchestrator_actions(episodes: list[Episode]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Evaluate a concrete hybrid policy that combines all model families as candidate generators."""
    eg_bandit = _copy.deepcopy(base_eg_robust_bandit)
    m6_bandit = _copy.deepcopy(base_m6_robust_bandit)
    m7_bandit = _copy.deepcopy(base_m7_robust_bandit)
    decision_rows, candidate_rows = [], []
    for step, ep in enumerate(episodes):
        candidates = []
        def add_candidate(source: str, action: str):
            """Add one allowed candidate action and preserve its source label."""
            if action in ep.allowed_actions:
                candidates.append({"source": source, "action": action})

        add_candidate("RuleLookup", ep.allowed_actions[0])
        add_candidate("M8 RewardRanker", m8_policy.select(ep))
        eg_idx = eg_bandit.select(ep.primary_rc, ep.allowed_actions)
        add_candidate("EpsilonGreedy", ep.allowed_actions[eg_idx])
        x = _encode_state({**ep.state, "__rc__": ep.primary_rc}, train_stats)
        m6_idx = m6_bandit.select(ep.primary_rc, x, ep.allowed_actions)
        add_candidate("M6 LinUCB", ep.allowed_actions[m6_idx])
        _, ucb_scores, shortlist, _, key = _shortlist_for_episode(m7_bandit, ep, train_stats)
        m7_action, m7_meta = _choose_m7_action(shortlist, ucb_scores, arbiter_cache.get(key, {}))
        add_candidate("M7 LinUCB+LLM" if m7_meta.get("source") == "llm_blend" else "M7 UCB fallback", m7_action)

        posterior = posterior_rc_candidates(ep, k=3)
        unique = {}
        for cand in candidates:
            unique.setdefault(cand["action"], set()).add(cand["source"])
        scored = []
        for action, sources in unique.items():
            exp_reward, exp_regret, worst_regret = score_action_under_posterior(ep, action, posterior)
            intent_reward, _ = oracle_reward(ep.primary_rc, action, ep.state)
            cost = ACTION_BY_CODE[action]["cost_weight"]
            scored.append({
                "episode_id": ep.episode_id,
                "primary_rc": ep.primary_rc,
                "action": action,
                "candidate_sources": " + ".join(sorted(sources)),
                "candidate_source_count": len(sources),
                "posterior_expected_reward": exp_reward,
                "posterior_expected_regret": exp_regret,
                "posterior_worst_regret": worst_regret,
                "intent_reward": intent_reward,
                "action_cost": cost,
                "action_risky": ACTION_BY_CODE[action]["risky"],
            })
        scored_df = pd.DataFrame(scored)
        selected = (
            scored_df.sort_values(
                ["posterior_expected_reward", "posterior_worst_regret", "action_cost", "intent_reward"],
                ascending=[False, True, True, False],
            )
            .iloc[0]
            .to_dict()
        )
        reward, _ = oracle_reward(ep.primary_rc, selected["action"], ep.state)
        best_action, best_reward = _best_action_for_episode(ep)
        decision_rows.append({
            "step": step,
            "episode_id": ep.episode_id,
            "primary_rc": ep.primary_rc,
            "chosen_action": selected["action"],
            "best_action": best_action,
            "reward": float(reward),
            "best_reward": float(best_reward),
            "regret": float(best_reward - reward),
            "candidate_sources": selected["candidate_sources"],
            "candidate_count": int(len(scored_df)),
            "posterior_expected_reward": selected["posterior_expected_reward"],
            "posterior_worst_regret": selected["posterior_worst_regret"],
        })
        candidate_rows.extend(scored)
    return pd.DataFrame(decision_rows), pd.DataFrame(candidate_rows)

rule_trace = trace_from_decisions(
    "RuleLookup",
    baseline_decisions[(baseline_decisions["policy"] == "rule_lookup") & (baseline_decisions["run_id"] == 0)],
)
eg_trace = trace_from_decisions("EpsilonGreedy", eg_holdout_decisions)
m6_trace = trace_from_decisions("M6 LinUCB", m6_holdout_decisions)
m7_trace = trace_from_decisions("M7 LinUCB+LLM", m7_decisions)
m8_trace = trace_from_decisions("M8 RewardRanker", m8_decisions)
hybrid_decisions, hybrid_candidate_frame = select_hybrid_orchestrator_actions(test_eps)
hybrid_trace = trace_from_decisions("Hybrid Orchestrator", hybrid_decisions)

final_decision_traces = pd.concat([rule_trace, eg_trace, m6_trace, m7_trace, m8_trace, hybrid_trace], ignore_index=True)
episode_lookup = {ep.episode_id: ep for ep in test_eps}
mode_rows = []
for _, row in final_decision_traces.iterrows():
    ep = episode_lookup.get(row["episode_id"])
    if ep is None:
        continue
    scores = score_action_reward_modes(ep, row["chosen_action"])
    mode_rows.append({
        "policy": row["policy"],
        "episode_id": row["episode_id"],
        "primary_rc": row["primary_rc"],
        "chosen_action": row["chosen_action"],
        "regret": row["regret"],
        **scores,
    })
reward_mode_case_df = pd.DataFrame(mode_rows)
reward_mode_long = reward_mode_case_df.melt(
    id_vars=["policy", "episode_id", "primary_rc", "chosen_action", "regret"],
    value_vars=["intent_reward", "multi_objective_reward", "simulated_outcome_reward"],
    var_name="reward_mode", value_name="reward_value",
)
mode_summary = (
    reward_mode_long.groupby(["policy", "reward_mode"], as_index=False)
    .agg(mean_reward=("reward_value", "mean"),
         std_reward=("reward_value", "std"),
         episodes=("episode_id", "nunique"))
)
mode_pivot = mode_summary.pivot(index="policy", columns="reward_mode", values="mean_reward").reset_index()
mode_pivot["multi_minus_intent"] = mode_pivot["multi_objective_reward"] - mode_pivot["intent_reward"]
mode_pivot["outcome_minus_intent"] = mode_pivot["simulated_outcome_reward"] - mode_pivot["intent_reward"]
mode_rc_summary = (
    reward_mode_long.groupby(["primary_rc", "policy", "reward_mode"], as_index=False)
    .agg(mean_reward=("reward_value", "mean"), episodes=("episode_id", "nunique"))
)

hybrid_summary = pd.DataFrame([{
    "policy": "Hybrid Orchestrator",
    "mean_reward": hybrid_decisions["reward"].mean(),
    "mean_regret": hybrid_decisions["regret"].mean(),
    "oracle_hits": int((hybrid_decisions["regret"] <= 1e-12).sum()),
    "episodes": len(hybrid_decisions),
    "mean_candidate_count": hybrid_decisions["candidate_count"].mean(),
}])
hybrid_source_component_summary = (
    hybrid_decisions.assign(source_component=hybrid_decisions["candidate_sources"].str.split(" + ", regex=False))
    .explode("source_component")
    .groupby("source_component", as_index=False)
    .agg(selected_episodes=("episode_id", "nunique"),
         mean_reward=("reward", "mean"),
         mean_regret=("regret", "mean"))
    .sort_values("selected_episodes", ascending=False)
)
hybrid_source_combination_summary = (
    hybrid_decisions
    .groupby("candidate_sources", as_index=False)
    .agg(selected_episodes=("episode_id", "nunique"),
         mean_reward=("reward", "mean"),
         mean_regret=("regret", "mean"))
    .sort_values("selected_episodes", ascending=False)
)
display(hybrid_source_component_summary)
display(hybrid_source_combination_summary.head(10))

reward_mode_audit = pd.DataFrame([
    {"check": "all_final_policies_scored", "passed": set(reward_mode_case_df["policy"]) == {"RuleLookup", "EpsilonGreedy", "M6 LinUCB", "M7 LinUCB+LLM", "M8 RewardRanker", "Hybrid Orchestrator"},
     "detail": f"policies={reward_mode_case_df['policy'].nunique()}"},
    {"check": "one_decision_per_policy_episode", "passed": reward_mode_case_df.groupby("policy")["episode_id"].nunique().eq(len(test_eps)).all(),
     "detail": f"episodes_per_policy={len(test_eps)}"},
    {"check": "outcome_effects_cover_selected_actions", "passed": set(reward_mode_case_df["chosen_action"]).issubset(set(ACTION_OUTCOME_EFFECTS)),
     "detail": f"selected_actions={reward_mode_case_df['chosen_action'].nunique()}"},
    {"check": "hybrid_uses_multiple_candidate_sources", "passed": hybrid_candidate_frame["candidate_sources"].nunique() >= 3 if "candidate_sources" in hybrid_candidate_frame else hybrid_decisions["candidate_sources"].nunique() >= 3,
     "detail": f"candidate_rows={len(hybrid_candidate_frame)}"},
])
print("Reward-mode and hybrid-orchestrator audit:")
print(reward_mode_audit.to_string(index=False))
assert reward_mode_audit["passed"].all()

print("\nReward mode summary by policy:")
print(mode_summary.round(4).to_string(index=False))
print("\nReward mode deltas against intent reward:")
print(mode_pivot.round(4).to_string(index=False))
print("\nHybrid orchestrator summary:")
print(hybrid_summary.round(4).to_string(index=False))
print("\nHybrid selected candidate source component summary:")
print(hybrid_source_component_summary.round(4).to_string(index=False))
print("\nHybrid selected candidate source combination summary:")
print(hybrid_source_combination_summary.head(10).round(4).to_string(index=False))

fig_modes = px.bar(
    mode_summary,
    x="policy", y="mean_reward", color="reward_mode", barmode="group",
    error_y="std_reward",
    title="Reward Mode Comparison by Final Policy",
    labels={"policy": "Policy", "mean_reward": "Mean reward", "reward_mode": "Reward mode"},
)
fig_modes.update_layout(width=1100, height=520, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_modes, "fig21_reward_modes_by_policy"); fig_modes.show()

delta_long = mode_pivot.melt(
    id_vars="policy",
    value_vars=["multi_minus_intent", "outcome_minus_intent"],
    var_name="delta_type", value_name="reward_delta",
)
fig_deltas = px.bar(
    delta_long,
    x="policy", y="reward_delta", color="delta_type", barmode="group",
    title="Reward Mode Shift Relative to Intent Reward",
    labels={"policy": "Policy", "reward_delta": "Reward delta", "delta_type": "Delta"},
)
fig_deltas.add_hline(y=0, line_color="#222", line_width=1)
fig_deltas.update_layout(width=1050, height=470, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_deltas, "fig22_reward_mode_deltas"); fig_deltas.show()

intent_by_rc = mode_rc_summary[mode_rc_summary["reward_mode"] == "intent_reward"].pivot(
    index="primary_rc", columns="policy", values="mean_reward"
).fillna(0.0)
fig_mode_rc = px.imshow(
    intent_by_rc.values,
    x=list(intent_by_rc.columns), y=list(intent_by_rc.index),
    color_continuous_scale="Viridis", text_auto=".3f", aspect="auto",
    title="Intent Reward by Root Cause and Final Policy",
    labels={"x": "Policy", "y": "Root cause", "color": "Mean intent reward"},
)
fig_mode_rc.update_layout(width=1050, height=520, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_mode_rc, "fig23_intent_reward_by_rc_policy"); fig_mode_rc.show()

fig_hybrid_sources = px.bar(
    hybrid_source_component_summary,
    x="source_component", y="selected_episodes", color="mean_regret", text="selected_episodes",
    color_continuous_scale="Reds",
    title="Hybrid Orchestrator Selected Candidate Sources",
    labels={"source_component": "Candidate source", "selected_episodes": "Selected episodes", "mean_regret": "Mean regret"},
)
fig_hybrid_sources.update_layout(width=900, height=480, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_hybrid_sources, "fig24_hybrid_candidate_sources"); fig_hybrid_sources.show()


,source_component,selected_episodes,mean_reward,mean_regret
0,EpsilonGreedy,102,0.6545,0.0210
5,RuleLookup,94,0.6777,0.0226
4,M8 RewardRanker,94,0.6777,0.0226
1,M6 LinUCB,65,0.7423,0.0349
3,M7 UCB fallback,42,0.6905,0.0529
2,M7 LinUCB+LLM,18,0.8750,0.0000


,candidate_sources,selected_episodes,mean_reward,mean_regret
8,EpsilonGreedy + M8 RewardRanker + RuleLookup,36,0.5194,0.0000
5,EpsilonGreedy + M6 LinUCB + M7 UCB fallback + ...,31,0.7435,0.0619
3,EpsilonGreedy + M6 LinUCB + M7 LinUCB+LLM + M8...,17,0.9000,0.0000
0,EpsilonGreedy,6,0.4683,0.0367
6,EpsilonGreedy + M6 LinUCB + M8 RewardRanker + ...,5,0.7100,0.0000
11,M6 LinUCB + M7 UCB fallback + M8 RewardRanker ...,4,0.6500,0.0500
4,EpsilonGreedy + M6 LinUCB + M7 UCB fallback,3,0.5000,0.0000
7,EpsilonGreedy + M7 UCB fallback,2,0.4750,0.0000
9,M6 LinUCB,2,0.4500,0.0500
1,EpsilonGreedy + M6 LinUCB,1,0.4500,0.0000


Reward-mode and hybrid-orchestrator audit:
                                 check  passed                  detail
             all_final_policies_scored    True              policies=6
       one_decision_per_policy_episode    True episodes_per_policy=111
outcome_effects_cover_selected_actions    True     selected_actions=14
hybrid_uses_multiple_candidate_sources    True      candidate_rows=182

Reward mode summary by policy:
             policy              reward_mode  mean_reward  std_reward  episodes
      EpsilonGreedy            intent_reward       0.6279      0.2238       111
      EpsilonGreedy   multi_objective_reward       0.7544      0.1474       111
      EpsilonGreedy simulated_outcome_reward       0.0462      0.0395       111
Hybrid Orchestrator            intent_reward       0.6456      0.2120       111
Hybrid Orchestrator   multi_objective_reward       0.7648      0.1410       111
Hybrid Orchestrator simulated_outcome_reward       0.0441      0.0384       111
          

**Interpretation:** We use the reward-mode comparison to check whether the final ranking is an artifact of one reward definition. The audit passes: all 6 final policies are scored, every policy has one decision for each of the 111 test episodes, the selected actions are all covered by the action-aware outcome simulator, and the hybrid uses multiple candidate sources. Under the primary intent reward, the implemented `Hybrid Orchestrator` is strongest at 0.6456 with standard deviation 0.2120. RuleLookup and M8 both score 0.6351, EpsilonGreedy scores 0.6279, M6 scores 0.6150, and M7 scores 0.6110 in the representative traces used for this reward-mode table.

The multi-objective view raises every policy because it adds credit for low cost and safety. The hybrid remains highest at 0.7648, followed by RuleLookup and M8 at 0.7595, EpsilonGreedy at 0.7544, M6 at 0.7361, and M7 at 0.7310. The reward-mode delta figure shows that this uplift is consistent rather than selective: the multi-objective reward adds about 0.1192 to 0.1265 across policies. The simulated outcome view is much lower for every policy, between 0.0425 and 0.0504, because it is a conservative action-effect simulation rather than observed post-remediation telemetry. We therefore treat the outcome score as a sensitivity check, not as causal evidence.

The hybrid source figure is the most important final addition. The combined policy does not invent a new standalone model; it chooses among candidates proposed by the existing models. It reaches 0.6456 reward, 0.0229 regret, and 96 oracle-hit decisions out of 111, with an average of 1.6396 unique candidate actions per episode. The most common selected source combination is `EpsilonGreedy + M8 RewardRanker + RuleLookup` with 36 selected episodes and 0.0000 mean regret. This supports the final design choice: the models are complementary. RuleLookup and M8 provide the stable structured prior, EpsilonGreedy supplies online adaptation, M6 supplies contextual bandit diagnostics, M7 supplies auditable reasoning when valid, and the posterior arbiter chooses among their candidates under diagnosis uncertainty.

<a id="sec-20"></a>

## 20 - Final Verdict

The final verdict follows from the benchmark, case studies, robustness tests, and reward-mode comparison together. We do not reduce the notebook to a single winning model; the executed results show that each model contributes a different capability, and the implemented `Hybrid Orchestrator` combines those capabilities into the strongest final policy.

In [27]:

final_score_rows = []
for policy_name, decisions in [
    ("RuleLookup", rule_trace),
    ("EpsilonGreedy", eg_trace),
    ("M6 LinUCB", m6_trace),
    ("M7 LinUCB+LLM", m7_trace),
    ("M8 RewardRanker", m8_trace),
    ("Hybrid Orchestrator", hybrid_trace),
]:
    final_score_rows.append({
        "policy": policy_name,
        "mean_intent_reward": decisions["reward"].mean(),
        "mean_regret": decisions["regret"].mean(),
        "oracle_hits": int((decisions["regret"] <= 1e-12).sum()),
        "episodes": len(decisions),
        "deployment_role": {
            "RuleLookup": "transparent safety anchor",
            "EpsilonGreedy": "best online learner",
            "M6 LinUCB": "contextual bandit baseline",
            "M7 LinUCB+LLM": "auditable reasoning layer",
            "M8 RewardRanker": "offline reward-ranker lens",
            "Hybrid Orchestrator": "combined deployment policy",
        }[policy_name],
    })
final_score_table = pd.DataFrame(final_score_rows).sort_values(
    ["mean_intent_reward", "mean_regret"], ascending=[False, True]
)
final_score_table["norm_intent_reward"] = final_score_table["mean_intent_reward"].map(norm_reward)

model_role_table = pd.DataFrame([
    {"component": "RuleLookup", "kept_because": "It ties M8 on clean reward and is fully transparent.", "used_in_combination_as": "safety anchor and fallback"},
    {"component": "EpsilonGreedy", "kept_because": "It is the strongest online learner in the benchmark comparison.", "used_in_combination_as": "online adaptation candidate"},
    {"component": "M6 LinUCB", "kept_because": "It supplies a contextual linear bandit baseline with stable arm-level diagnostics.", "used_in_combination_as": "contextual candidate generator"},
    {"component": "M7 LinUCB+LLM", "kept_because": "It adds auditable reasoning when the prompt cache has valid coverage.", "used_in_combination_as": "reasoning candidate under cache validity"},
    {"component": "M8 RewardRanker", "kept_because": "It learns the offline reward surface and exposes reward-generalization failures.", "used_in_combination_as": "offline reward-ranker candidate"},
    {"component": "Posterior arbiter", "kept_because": "It chooses among candidates under diagnosis uncertainty.", "used_in_combination_as": "final selector for the hybrid orchestrator"},
])

print("Final policy score table:")
print(final_score_table.round(4).to_string(index=False))
print("\nWhy I combine the models:")
print(model_role_table.to_string(index=False))
print(f"\nM7 current-prompt LLM coverage: {m7_sweep['n_llm_used'].mean():.1f}/{len(test_eps)}")
print(f"Counterfactual robustness rows: {len(robust_df)}")
print(f"Diagnosis-posterior rows: {len(uncertainty_df)}")
print(f"Reward-mode case rows: {len(reward_mode_case_df)}")
print(f"All artefacts in {FIG_DIR}")
print(f"Artefact registry size: {len(ARTEFACT_REGISTRY)}")

fig_final_scores = px.bar(
    final_score_table,
    x="policy", y="mean_intent_reward", color="deployment_role", text="mean_intent_reward",
    title="Final Scorecard Including the Combined Hybrid Orchestrator",
    labels={"policy": "Policy", "mean_intent_reward": "Mean intent reward", "deployment_role": "Deployment role"},
)
fig_final_scores.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig_final_scores.update_layout(width=1100, height=520, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_final_scores, "fig25_final_scorecard_with_hybrid"); fig_final_scores.show()

fig_final_regret = px.bar(
    final_score_table.sort_values("mean_regret"),
    x="policy", y="mean_regret", color="deployment_role", text="mean_regret",
    title="Final Mean Regret Including the Combined Hybrid Orchestrator",
    labels={"policy": "Policy", "mean_regret": "Mean regret", "deployment_role": "Deployment role"},
)
fig_final_regret.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig_final_regret.update_layout(width=1100, height=520, template="plotly_white", **PLOTLY_LAYOUT)
save_fig(fig_final_regret, "fig26_final_regret_with_hybrid"); fig_final_regret.show()


Final policy score table:
             policy  mean_intent_reward  mean_regret  oracle_hits  episodes            deployment_role  norm_intent_reward
Hybrid Orchestrator              0.6456       0.0229           96       111 combined deployment policy              0.8036
         RuleLookup              0.6351       0.0333           92       111  transparent safety anchor              0.7139
    M8 RewardRanker              0.6351       0.0333           92       111 offline reward-ranker lens              0.7139
      EpsilonGreedy              0.6279       0.0405          102       111        best online learner              0.6520
          M6 LinUCB              0.6150       0.0534           80       111 contextual bandit baseline              0.5414
      M7 LinUCB+LLM              0.6110       0.0575           80       111  auditable reasoning layer              0.5066

Why I combine the models:
        component                                                                     

**Interpretation:** The final scorecard confirms the portfolio argument. The `Hybrid Orchestrator` reaches the best final intent reward at 0.6456, the lowest mean regret at 0.0229, and 96 oracle-hit decisions out of 111. RuleLookup and M8 tie at 0.6351 reward and 0.0333 regret with 92 oracle hits, so they remain strong safety anchors but not the whole deployment answer. EpsilonGreedy reaches 0.6279 with 102 oracle hits, which shows why it remains valuable as the online adaptation candidate even though its mean reward is below the hybrid. M6 reaches 0.6150 and M7 reaches 0.6110 in this final representative scorecard; their role is not to win the aggregate table, but to contribute contextual bandit structure and auditable LLM reasoning.

The model-role table makes the final architecture explicit. We keep RuleLookup because it is transparent. We keep EpsilonGreedy because it is the strongest online learner in the benchmark comparison. We keep M6 because it provides contextual arm diagnostics. We keep M7 because it provides reasoning when the prompt cache has valid coverage. We keep M8 because it learns and audits the offline reward surface. We combine them with the posterior arbiter because diagnosis uncertainty is the main scientific weakness left in the benchmark. This is the clean final conclusion: the notebook should not end with a single-model winner; it should end with an evaluated combination that uses each model for the thing it does best.

<a id="conclusion"></a>

## Conclusion

This notebook builds an auditable benchmark for QoS remediation policies under a root-cause-aware action mask. We start from incident-aligned QoS data, clean and engineer decision-time features, construct chronological train-validation-test episodes, define a root-cause vocabulary, validate the action catalogue, implement the reward rule engine, and then compare structured baselines, online bandits, LLM-guided arbitration, and offline reward ranking on the same real holdout split.

The single-model benchmark result is conservative. RuleLookup and M8 RewardRanker tie at 0.6351 reward and 0.0333 regret on the real holdout, so we do not claim that supervised reward ranking materially beats the structured prior. EpsilonGreedy is the best online learner in the benchmark comparison at 0.6240, while M6 and M7 provide contextual and reasoning-based alternatives with lower aggregate reward. The robustness and posterior sections show why the problem remains scientifically interesting: no-problem and transport-delay decisions are stable, but rare radio diagnoses such as coverage holes, weak signal, and SINR degradation still create high worst-case regret under diagnosis uncertainty.

The final contribution is the implemented combination. We evaluate a `Hybrid Orchestrator` that takes candidate actions from RuleLookup, EpsilonGreedy, M6 LinUCB, M7 LinUCB+LLM, and M8 RewardRanker, then selects among them with the diagnosis-posterior arbiter. This combined policy reaches 0.6456 mean intent reward, 0.0229 mean regret, and 96 oracle-hit decisions out of 111. That result is the strongest final endpoint because it uses each model for a different job: RuleLookup gives transparency and a safety fallback, EpsilonGreedy gives online adaptation, M6 gives contextual bandit diagnostics, M7 gives inspectable reasoning when cache coverage is valid, M8 gives an offline reward-surface audit, and the posterior arbiter protects the final choice under diagnostic uncertainty.

We therefore conclude that the scientifically defensible deployment design is not a single winning model. It is a combined, auditable policy stack. The benchmark shows that structured telecom priors carry much of the performance, learned policies add adaptation and diagnostics, LLM reasoning adds interpretability where coverage exists, and posterior arbitration is the mechanism that turns those pieces into a stronger decision policy. The next research step should add real post-action remediation logs so the simulated outcome reward can be replaced by observed causal recovery.